# 36 — Supervisor, Publication, and Reproducibility Package

**Origin:** Consolidates Existing Previous Versions of Notebooks 30 and 35, Pre-refactor\
**Refactor status:** Finished\
**Validation status:** Finished\
**Completion gate passed:** Yes\
**Output root:** `outputs/36_supervisor_publication_reproducibility_package/`\
**Depends on:** Notebooks 01–35

## Purpose

Assemble the final supervisor-facing, publication-facing, and reproducibility delivery package from the completed evaluation pipeline.

This notebook packages, indexes, and validates already generated evidence. It does not rerun restoration models, recompute scientific metrics, alter upstream results, or create new scientific evidence.

## Approved evidence population

The package represents:

- 35 completed upstream notebooks;
- 50 controlled paintings;
- 525 registered experimental cases;
- 410 restoration cases;
- 1,785 approved comparison candidates;
- 11 separate quality anchors;
- 165 repeated-seed uncertainty groups;
- 23,964 indexed visual records;
- 104 indexed reports;
- 10 bounded SDXL feasibility cases;
- 18 thesis figures;
- 6 publication figures;
- 4 self-contained model reports;
- 1 self-contained final evaluation report;
- 4 model cards;
- 30 case-report records;
- 50 painting-report records.

## Research questions

1. **How can a multi-metric evaluation framework be designed to assess AI-generated painting restorations beyond traditional image similarity metrics?**
2. **How do selected pretrained inpainting models differ in restoration quality across artistic styles and artificial damage types?**
3. **To what extent can uncertainty estimation from multiple restoration candidates identify speculative or unreliable restored regions?**

The package answers these questions only within the validated controlled benchmark and its explicitly bounded extensions.

## Package boundary

The portable package includes:

- the final self-contained HTML report;
- four self-contained model reports;
- 18 thesis figures and 6 publication figures;
- four model cards;
- eight compact tables and report indexes;
- all 35 upstream run manifests;
- all declared evaluation-configuration snapshots;
- both requirements files;
- the Notebook 35 deployment-readiness report;
- the Streamlit entry point and read-only application helper;
- Notebook 36 reports, meeting documents, indexes, manifests, and provenance.

The following collections remain indexed at their canonical repository paths but are not duplicated:

- 30 self-contained case reports;
- 50 self-contained painting reports;
- 30 selected-case grids;
- the complete 23,964-record dashboard visual collection;
- restoration candidates, masks, maps, and raw image collections;
- model weights and local caches.

## Interpretation boundaries

- Visual plausibility is not historical correctness, restoration trustworthiness, or conservation approval.
- The benchmark uses controlled synthetic damage and does not establish real-world conservation generality.
- SDXL remains a bounded ten-case feasibility study rather than a fourth fully evaluated benchmark.
- Repeated-seed variability is an empirical uncertainty proxy, not calibrated confidence.
- Computational flags, retrieval results, and recommendations are decision support rather than expert ground truth.
- No universal combined quality, uncertainty, or trustworthiness score is constructed.
- Notebook 36 republishes validated evidence and does not strengthen unsupported claims.

## Canonical outputs

```text
reports/supervisor_summary.md
reports/reproducibility_appendix.md
reports/limitations_and_deviations.md
data/artifact_index.csv
data/key_findings.json
data/open_questions.md
data/feedback_agenda.md
package/README.md
package/reports/
package/figures/
package/tables/
package/model_cards/
package/manifests/notebook_runs/
package/configuration/evaluation/
package/environment/
package/application/
package/provenance/reproducibility_snapshot.json
manifests/package_manifest.json
manifests/run_manifest.json
manifests/artifacts.csv
validation/checks.csv

## Batch 1 — Contract, paths, and environment preflight

This batch:

- discovers the repository root without assuming the working directory;
- loads the versioned Notebook 36 package contract;
- creates only the approved notebook-owned output directories;
- blocks execution when stale files already exist in the output root;
- loads the current project inventory and project-path registry;
- verifies the roadmap, implementation guidelines, and evidence-coverage contract;
- confirms that Notebook 34 and Notebook 35 artifacts are registered;
- validates every fixed source in the package copy plan;
- confirms that all 35 upstream run manifests are declared;
- verifies accepted Python versions and required packages;
- checks the Notebook 36 helper for valid Python syntax;
- validates the expected package groups, file count, estimated size, and destination uniqueness;
- applies a blocking gate before upstream evidence is loaded.

This batch creates directories only. It does not copy package files, write canonical outputs, load scientific tables, or recompute evidence.

In [1]:
from __future__ import annotations

import ast
import json
import platform
import sys
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import display


CURRENT_LOCATION = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        candidate
        for candidate in (CURRENT_LOCATION, *CURRENT_LOCATION.parents)
        if (
            candidate
            / "config"
            / "evaluation"
            / "supervisor_package.yaml"
        ).is_file()
        and (candidate / "src" / "restoration_eval").is_dir()
        and (candidate / "notebooks").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the painting-restoration-eval repository root."
    )

SOURCE_ROOT = PROJECT_ROOT / "src"

if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))


from restoration_eval.paths import (
    PATHS_MODULE_VERSION,
    PROJECT_PATHS_SCHEMA_VERSION,
    validate_project_paths_registry,
)
from restoration_eval.supervisor_package import (
    SUPERVISOR_PACKAGE_CONFIG_SCHEMA_VERSION,
    SUPERVISOR_PACKAGE_VERSION,
    build_copy_plan,
    copy_plan_frame,
    create_output_directories,
    load_supervisor_package_config,
    output_paths,
    required_input_paths,
    safe_repo_path,
)
from restoration_eval.validation import (
    VALIDATION_MODULE_VERSION,
    ValidationCollector,
)


NOTEBOOK_ID = "36"
NOTEBOOK_STEM = "36_supervisor_publication_reproducibility_package"

RUN_STARTED_AT_UTC = (
    datetime.now(timezone.utc)
    .isoformat(timespec="seconds")
    .replace("+00:00", "Z")
)

VALIDATION = ValidationCollector()

print("Notebook 36 environment initialized.")
print("Project root:", PROJECT_ROOT)
print("Run started:", RUN_STARTED_AT_UTC)
print("Python:", platform.python_version())
print("Platform:", platform.platform())
print(
    "Helper versions:",
    {
        "supervisor_package": SUPERVISOR_PACKAGE_VERSION,
        "validation": VALIDATION_MODULE_VERSION,
        "paths": PATHS_MODULE_VERSION,
    },
)

Notebook 36 environment initialized.
Project root: D:\Masters\FH\Thesis\painting-restoration-eval
Run started: 2026-09-03T21:13:19Z
Python: 3.12.6
Platform: Windows-11-10.0.26200-SP0
Helper versions: {'supervisor_package': '1.0.0', 'validation': '1.0.0', 'paths': '1.0.0'}


In [2]:
CONFIG_PATH = (
    PROJECT_ROOT
    / "config"
    / "evaluation"
    / "supervisor_package.yaml"
)

CONFIG = load_supervisor_package_config(PROJECT_ROOT)

NOTEBOOK_CONFIG = CONFIG["notebook"]
RUNTIME_CONFIG = CONFIG["runtime"]
EXPECTED_POPULATION = CONFIG["expected_population"]
PACKAGE_POLICY = CONFIG["package_policy"]
SCIENTIFIC_BOUNDARIES = CONFIG["scientific_boundaries"]
RESEARCH_QUESTIONS = CONFIG["research_questions"]

OUTPUT_ROOT = safe_repo_path(
    NOTEBOOK_CONFIG["output_root"],
    PROJECT_ROOT,
)

OUTPUT_ROOT_PREEXISTED = OUTPUT_ROOT.exists()

PREEXISTING_OUTPUT_FILES = (
    sorted(
        path.relative_to(OUTPUT_ROOT).as_posix()
        for path in OUTPUT_ROOT.rglob("*")
        if path.is_file()
    )
    if OUTPUT_ROOT_PREEXISTED
    else []
)

OUTPUT_PATHS = output_paths(CONFIG, PROJECT_ROOT)
CREATED_OUTPUT_DIRS = create_output_directories(CONFIG, PROJECT_ROOT)

REQUIRED_INPUT_PATHS = required_input_paths(
    CONFIG,
    PROJECT_ROOT,
)

COPY_PLAN = build_copy_plan(
    CONFIG,
    PROJECT_ROOT,
)

COPY_PLAN_TABLE = copy_plan_frame(
    COPY_PLAN,
    PROJECT_ROOT,
)

EXPECTED_OUTPUT_KEYS = {
    "supervisor_summary",
    "reproducibility_appendix",
    "limitations_and_deviations",
    "artifact_index",
    "key_findings",
    "open_questions",
    "feedback_agenda",
    "package_root",
    "package_readme",
    "reproducibility_snapshot",
    "package_manifest",
    "run_manifest",
    "artifacts",
    "validation",
}

EXPECTED_COPY_GROUP_COUNTS = {
    "application": 2,
    "configuration": 25,
    "environment": 2,
    "manifests": 35,
    "model_cards": 4,
    "publication_figures": 6,
    "reports": 6,
    "tables": 8,
    "thesis_figures": 18,
}

OBSERVED_COPY_GROUP_COUNTS = {
    str(group): int(count)
    for group, count in (
        COPY_PLAN_TABLE
        .groupby("group", dropna=False)
        .size()
        .sort_index()
        .items()
    )
}

COPY_PLAN_SIZE_BYTES = int(
    COPY_PLAN_TABLE["size_bytes"].sum()
)

COPY_PLAN_SIZE_MIB = (
    COPY_PLAN_SIZE_BYTES / (1024 ** 2)
)

MAXIMUM_PACKAGE_SIZE_BYTES = int(
    PACKAGE_POLICY["maximum_package_size_mib"]
    * (1024 ** 2)
)

OUTPUT_PATHS_ARE_OWNED = all(
    path.resolve().is_relative_to(OUTPUT_ROOT.resolve())
    for path in OUTPUT_PATHS.values()
)

BOUNDARY_FLAGS = {
    key: bool(SCIENTIFIC_BOUNDARIES[key])
    for key in (
        "no_scientific_recomputation",
        "no_restoration_inference",
        "no_universal_combined_score",
        "controlled_synthetic_scope_only",
        "uncertainty_is_not_calibrated_confidence",
        "sdxl_is_bounded_feasibility_only",
        "computational_flags_are_not_expert_ground_truth",
    )
}

CONTRACT_CHECKS = (
    (
        "configuration_schema",
        "The approved Notebook 36 configuration schema is loaded",
        SUPERVISOR_PACKAGE_CONFIG_SCHEMA_VERSION,
        CONFIG["schema_version"],
        (
            CONFIG["schema_version"]
            == SUPERVISOR_PACKAGE_CONFIG_SCHEMA_VERSION
        ),
    ),
    (
        "notebook_identity",
        "The notebook identity is fixed",
        {"id": NOTEBOOK_ID, "stem": NOTEBOOK_STEM},
        {
            "id": NOTEBOOK_CONFIG["id"],
            "stem": NOTEBOOK_CONFIG["stem"],
        },
        (
            NOTEBOOK_CONFIG["id"] == NOTEBOOK_ID
            and NOTEBOOK_CONFIG["stem"] == NOTEBOOK_STEM
        ),
    ),
    (
        "clean_output_root",
        "No stale files existed in the Notebook 36 output root",
        [],
        PREEXISTING_OUTPUT_FILES,
        not PREEXISTING_OUTPUT_FILES,
    ),
    (
        "canonical_output_keys",
        "All approved output roles are declared exactly once",
        sorted(EXPECTED_OUTPUT_KEYS),
        sorted(OUTPUT_PATHS),
        set(OUTPUT_PATHS) == EXPECTED_OUTPUT_KEYS,
    ),
    (
        "notebook_output_ownership",
        "Every declared output belongs to Notebook 36",
        True,
        OUTPUT_PATHS_ARE_OWNED,
        OUTPUT_PATHS_ARE_OWNED,
    ),
    (
        "required_input_count",
        "All governing, manifest, and fixed-copy inputs resolve",
        66,
        len(REQUIRED_INPUT_PATHS),
        len(REQUIRED_INPUT_PATHS) == 66,
    ),
    (
        "copy_plan_file_count",
        "The approved copy plan contains the expected files",
        106,
        len(COPY_PLAN),
        len(COPY_PLAN) == 106,
    ),
    (
        "copy_plan_groups",
        "Copy-plan groups match the approved package structure",
        EXPECTED_COPY_GROUP_COUNTS,
        OBSERVED_COPY_GROUP_COUNTS,
        OBSERVED_COPY_GROUP_COUNTS == EXPECTED_COPY_GROUP_COUNTS,
    ),
    (
        "copy_plan_destinations_unique",
        "Every copied artifact has one unique destination",
        True,
        bool(COPY_PLAN_TABLE["destination_path"].is_unique),
        bool(COPY_PLAN_TABLE["destination_path"].is_unique),
    ),
    (
        "package_size_limit",
        "The planned package remains below the 50 MiB ceiling",
        f"<= {PACKAGE_POLICY['maximum_package_size_mib']} MiB",
        round(COPY_PLAN_SIZE_MIB, 3),
        COPY_PLAN_SIZE_BYTES <= MAXIMUM_PACKAGE_SIZE_BYTES,
    ),
    (
        "scientific_boundaries",
        "All mandatory scope boundaries are active",
        {key: True for key in BOUNDARY_FLAGS},
        BOUNDARY_FLAGS,
        all(BOUNDARY_FLAGS.values()),
    ),
)

for check_id, description, expected, observed, passed in CONTRACT_CHECKS:
    VALIDATION.add(
        validation_stage="batch_1_contract",
        check_id=check_id,
        check_description=description,
        severity="blocking",
        expected=expected,
        observed=observed,
        passed=bool(passed),
        details=(
            ""
            if passed
            else "Notebook 36 differs from the approved package contract."
        ),
    )

VALIDATION.raise_for_blocking()

OUTPUT_CONTRACT_VIEW = pd.DataFrame(
    [
        {
            "output_key": key,
            "repository_relative_path": (
                path.relative_to(PROJECT_ROOT).as_posix()
            ),
            "kind": (
                "directory"
                if key == "package_root"
                else "file"
            ),
            "exists_before_persistence": path.exists(),
        }
        for key, path in OUTPUT_PATHS.items()
    ]
).sort_values("output_key").reset_index(drop=True)

COPY_PLAN_SUMMARY = (
    COPY_PLAN_TABLE
    .groupby("group", dropna=False)
    .agg(
        files=("destination_path", "size"),
        size_bytes=("size_bytes", "sum"),
    )
    .reset_index()
)

COPY_PLAN_SUMMARY["size_mib"] = (
    COPY_PLAN_SUMMARY["size_bytes"] / (1024 ** 2)
)

display(OUTPUT_CONTRACT_VIEW)
display(COPY_PLAN_SUMMARY)

print("Notebook 36 contract passed.")
print("Pre-existing output files:", len(PREEXISTING_OUTPUT_FILES))
print("Created output directories:", len(CREATED_OUTPUT_DIRS))
print("Fixed package-copy files:", len(COPY_PLAN))
print("Estimated copied size MiB:", round(COPY_PLAN_SIZE_MIB, 3))
print("Canonical files persisted by this cell: 0")

,output_key,repository_relative_path,kind,exists_before_persistence
0,artifact_index,outputs/36_supervisor_publication_reproducibil...,file,False
1,artifacts,outputs/36_supervisor_publication_reproducibil...,file,False
2,feedback_agenda,outputs/36_supervisor_publication_reproducibil...,file,False
3,key_findings,outputs/36_supervisor_publication_reproducibil...,file,False
4,limitations_and_deviations,outputs/36_supervisor_publication_reproducibil...,file,False
5,open_questions,outputs/36_supervisor_publication_reproducibil...,file,False
6,package_manifest,outputs/36_supervisor_publication_reproducibil...,file,False
7,package_readme,outputs/36_supervisor_publication_reproducibil...,file,False
8,package_root,outputs/36_supervisor_publication_reproducibil...,directory,True
9,reproducibility_appendix,outputs/36_supervisor_publication_reproducibil...,file,False


,group,files,size_bytes,size_mib
0,application,2,111186,0.106035
1,configuration,25,456708,0.435551
2,environment,2,955,0.000911
3,manifests,35,734530,0.700502
4,model_cards,4,40142,0.038282
5,publication_figures,6,658247,0.627753
6,reports,6,24380109,23.250684
7,tables,8,785263,0.748885
8,thesis_figures,18,1769332,1.687366


Notebook 36 contract passed.
Pre-existing output files: 0
Created output directories: 15
Fixed package-copy files: 106
Estimated copied size MiB: 27.596
Canonical files persisted by this cell: 0


In [3]:
INVENTORY_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "inventory"
    / "project_file_inventory.csv"
)

INVENTORY_RUN_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "inventory"
    / "inventory_run.json"
)

PROJECT_PATHS_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "inventory"
    / "project_paths.json"
)

with INVENTORY_RUN_PATH.open(
    "r",
    encoding="utf-8-sig",
) as handle:
    INVENTORY_RUN = json.load(handle)

INVENTORY = pd.read_csv(
    INVENTORY_PATH,
    low_memory=False,
)

with PROJECT_PATHS_PATH.open(
    "r",
    encoding="utf-8-sig",
) as handle:
    PROJECT_PATHS_REGISTRY = json.load(handle)

PROJECT_PATH_ERRORS = validate_project_paths_registry(
    PROJECT_PATHS_REGISTRY
)

IMPLEMENTATION_PATH = (
    PROJECT_ROOT
    / "docs"
    / "refactoring_implementation_guidelines.md"
)

ROADMAP_PATH = (
    PROJECT_ROOT
    / "docs"
    / "final_notebook_roadmap.md"
)

EVIDENCE_AUDIT_PATH = (
    PROJECT_ROOT
    / "docs"
    / "evidence_dependency_audit.md"
)

EVIDENCE_COVERAGE_PATH = (
    PROJECT_ROOT
    / "config"
    / "evaluation"
    / "evidence_coverage.yaml"
)

IMPLEMENTATION_TEXT = IMPLEMENTATION_PATH.read_text(
    encoding="utf-8-sig"
)

ROADMAP_TEXT = ROADMAP_PATH.read_text(
    encoding="utf-8-sig"
)

EVIDENCE_AUDIT_TEXT = EVIDENCE_AUDIT_PATH.read_text(
    encoding="utf-8-sig"
)

with EVIDENCE_COVERAGE_PATH.open(
    "r",
    encoding="utf-8-sig",
) as handle:
    EVIDENCE_COVERAGE = yaml.safe_load(handle)

PLANNED_NOTEBOOKS = EVIDENCE_COVERAGE[
    "evidence_coverage"
]["planned_notebooks"]

N36_COVERAGE = next(
    (
        record
        for record in PLANNED_NOTEBOOKS
        if str(record.get("notebook_id")) == NOTEBOOK_ID
    ),
    None,
)

INVENTORY_PATHS = set(
    INVENTORY["relative_path"].astype(str)
)

PREPARATION_FILES = {
    "README.md",
    "config/evaluation/evidence_coverage.yaml",
    "config/evaluation/supervisor_package.yaml",
    "docs/evidence_dependency_audit.md",
    "docs/final_notebook_roadmap.md",
    "docs/literature_reference_log.md",
    "docs/methodology_notes.md",
    "docs/refactoring_implementation_guidelines.md",
    "notebooks/36_supervisor_publication_reproducibility_package.ipynb",
    "outputs/inventory/project_paths.json",
    "outputs/inventory/project_paths.md",
    "src/restoration_eval/supervisor_package.py",
}

REGISTRY_ARTIFACTS = pd.DataFrame(
    PROJECT_PATHS_REGISTRY["artifacts"]
)

N34_REGISTRY_RECORDS = REGISTRY_ARTIFACTS.loc[
    REGISTRY_ARTIFACTS["producer_notebook"]
    == "34_final_streamlit_dashboard_assets"
].copy()

N35_REGISTRY_RECORDS = REGISTRY_ARTIFACTS.loc[
    REGISTRY_ARTIFACTS["producer_notebook"]
    == "35_dashboard_and_deployment_validation"
].copy()

CURRENT_PYTHON_MINOR = (
    f"{sys.version_info.major}.{sys.version_info.minor}"
)

IMPLEMENTATION_TEXT_CASEFOLD = IMPLEMENTATION_TEXT.casefold()
ROADMAP_TEXT_CASEFOLD = ROADMAP_TEXT.casefold()
AUDIT_TEXT_CASEFOLD = EVIDENCE_AUDIT_TEXT.casefold()

AUDIT_TEXT_NORMALIZED = " ".join(
    EVIDENCE_AUDIT_TEXT.casefold().split()
)

MANUAL_EDITING_POLICY_PRESENT = (
    "must not insert batch 1 or any later cells"
    in IMPLEMENTATION_TEXT_CASEFOLD
)

PACKAGE_POLICY_PRESENT = (
    "final supervisor and reproducibility package"
    in IMPLEMENTATION_TEXT_CASEFOLD
)

ROADMAP_ENTRY_PRESENT = all(
    fragment in ROADMAP_TEXT_CASEFOLD
    for fragment in (
        "36_supervisor_publication_reproducibility_package.ipynb",
        "approved evidence population",
        "package boundary",
        "35 completed upstream notebook manifests",
    )
)

EVIDENCE_AUDIT_ENTRY_PRESENT = all(
    fragment in AUDIT_TEXT_NORMALIZED
    for fragment in (
        "notebook 36 preparation contract",
        "1,785 approved candidates",
        "165 uncertainty groups",
        "creates no new scientific evidence",
    )
)

GOVERNANCE_CHECKS = (
    (
        "inventory_status",
        "The current project inventory completed successfully",
        "completed",
        INVENTORY_RUN.get("status"),
        INVENTORY_RUN.get("status") == "completed",
    ),
    (
        "inventory_read_errors",
        "The project inventory contains no read errors",
        0,
        int(INVENTORY_RUN["summary"]["read_error_count"]),
        int(INVENTORY_RUN["summary"]["read_error_count"]) == 0,
    ),
    (
        "inventory_repository_root",
        "The inventory belongs to this repository",
        str(PROJECT_ROOT),
        INVENTORY_RUN.get("repository_root"),
        (
            Path(INVENTORY_RUN["repository_root"]).resolve()
            == PROJECT_ROOT
        ),
    ),
    (
        "supported_python",
        "The kernel uses Python 3.11 or 3.12",
        sorted(RUNTIME_CONFIG["accepted_python_minor_versions"]),
        CURRENT_PYTHON_MINOR,
        (
            CURRENT_PYTHON_MINOR
            in RUNTIME_CONFIG["accepted_python_minor_versions"]
        ),
    ),
    (
        "preparation_files_indexed",
        "All Notebook 36 preparation files appear in the inventory",
        sorted(PREPARATION_FILES),
        sorted(PREPARATION_FILES & INVENTORY_PATHS),
        PREPARATION_FILES.issubset(INVENTORY_PATHS),
    ),
    (
        "project_paths_schema",
        "The project-path registry uses the canonical schema",
        PROJECT_PATHS_SCHEMA_VERSION,
        PROJECT_PATHS_REGISTRY.get("registry_schema_version"),
        (
            PROJECT_PATHS_REGISTRY.get("registry_schema_version")
            == PROJECT_PATHS_SCHEMA_VERSION
            and not PROJECT_PATH_ERRORS
        ),
    ),
    (
        "project_paths_count",
        "The artifact registry includes the completed N01–N35 state",
        218,
        len(REGISTRY_ARTIFACTS),
        len(REGISTRY_ARTIFACTS) == 218,
    ),
    (
        "notebook_34_registry_records",
        "All Notebook 34 dashboard artifacts are registered",
        17,
        len(N34_REGISTRY_RECORDS),
        len(N34_REGISTRY_RECORDS) == 17,
    ),
    (
        "notebook_35_registry_records",
        "Both Notebook 35 completion artifacts are registered",
        2,
        len(N35_REGISTRY_RECORDS),
        len(N35_REGISTRY_RECORDS) == 2,
    ),
    (
        "notebook_35_warning_preservation",
        "Notebook 35 non-blocking warnings remain visible",
        2,
        int(
            N35_REGISTRY_RECORDS[
                "validation_status"
            ].eq("warning").sum()
        ),
        (
            len(N35_REGISTRY_RECORDS) == 2
            and N35_REGISTRY_RECORDS[
                "validation_status"
            ].eq("warning").all()
        ),
    ),
    (
        "manual_notebook_editing_policy",
        "Notebook cells remain user-pasted and user-executed",
        True,
        MANUAL_EDITING_POLICY_PRESENT,
        MANUAL_EDITING_POLICY_PRESENT,
    ),
    (
        "final_package_policy",
        "The implementation guideline contains the final-package policy",
        True,
        PACKAGE_POLICY_PRESENT,
        PACKAGE_POLICY_PRESENT,
    ),
    (
        "roadmap_entry",
        "The detailed Notebook 36 roadmap entry is present",
        True,
        ROADMAP_ENTRY_PRESENT,
        ROADMAP_ENTRY_PRESENT,
    ),
    (
        "evidence_audit_entry",
        "The Notebook 36 evidence gate is documented",
        True,
        EVIDENCE_AUDIT_ENTRY_PRESENT,
        EVIDENCE_AUDIT_ENTRY_PRESENT,
    ),
    (
        "machine_coverage_entry",
        "The machine-readable Notebook 36 evidence gate is present",
        "planned_supported",
        (
            None
            if N36_COVERAGE is None
            else N36_COVERAGE.get("status")
        ),
        (
            N36_COVERAGE is not None
            and N36_COVERAGE.get("status")
            == "planned_supported"
            and not bool(
                N36_COVERAGE.get(
                    "creates_new_scientific_evidence",
                    True,
                )
            )
        ),
    ),
)

for check_id, description, expected, observed, passed in GOVERNANCE_CHECKS:
    VALIDATION.add(
        validation_stage="batch_1_inventory_governance",
        check_id=check_id,
        check_description=description,
        severity="blocking",
        expected=expected,
        observed=observed,
        passed=bool(passed),
        details=(
            ""
            if passed
            else "Inventory, registry, or governance preflight failed."
        ),
    )

VALIDATION.raise_for_blocking()

PREPARATION_INVENTORY = (
    INVENTORY.loc[
        INVENTORY["relative_path"].isin(PREPARATION_FILES),
        [
            "relative_path",
            "file_kind",
            "format",
            "size_bytes",
            "read_error_count",
        ],
    ]
    .sort_values("relative_path")
    .reset_index(drop=True)
)

REGISTRY_COMPLETION_VIEW = pd.concat(
    [
        N34_REGISTRY_RECORDS.assign(notebook_id="34"),
        N35_REGISTRY_RECORDS.assign(notebook_id="35"),
    ],
    ignore_index=True,
)[
    [
        "notebook_id",
        "artifact_key",
        "validation_status",
        "row_count",
        "relative_path",
    ]
]

display(PREPARATION_INVENTORY)
display(REGISTRY_COMPLETION_VIEW)

print("Inventory and governance preflight passed.")
print("Inventory run ID:", INVENTORY_RUN["inventory_run_id"])
print("Indexed files:", INVENTORY_RUN["summary"]["file_count"])
print("Registered artifacts:", len(REGISTRY_ARTIFACTS))
print("Notebook 34 registry records:", len(N34_REGISTRY_RECORDS))
print("Notebook 35 registry records:", len(N35_REGISTRY_RECORDS))

,relative_path,file_kind,format,size_bytes,read_error_count
0,README.md,markdown,markdown,32026,0
1,config/evaluation/evidence_coverage.yaml,yaml,yaml,81494,0
2,config/evaluation/supervisor_package.yaml,yaml,yaml,12342,0
3,docs/evidence_dependency_audit.md,markdown,markdown,45036,0
4,docs/final_notebook_roadmap.md,markdown,markdown,89457,0
5,docs/literature_reference_log.md,markdown,markdown,160648,0
6,docs/methodology_notes.md,markdown,markdown,20702,0
7,docs/refactoring_implementation_guidelines.md,markdown,markdown,70470,0
8,notebooks/36_supervisor_publication_reproducib...,notebook,notebook,640,0
9,outputs/inventory/project_paths.json,json,json,141618,0


,notebook_id,artifact_key,validation_status,row_count,relative_path
0,34,dashboard.asset_manifest,passed,15,outputs/34_final_streamlit_dashboard_assets/ma...
1,34,dashboard.index.case_index,passed,1785,outputs/34_final_streamlit_dashboard_assets/da...
2,34,dashboard.index.filter_options,passed,16,outputs/34_final_streamlit_dashboard_assets/da...
3,34,dashboard.index.painting_index,passed,50,outputs/34_final_streamlit_dashboard_assets/da...
4,34,dashboard.index.report_index,passed,104,outputs/34_final_streamlit_dashboard_assets/da...
5,34,dashboard.index.visual_asset_index,passed,23964,outputs/34_final_streamlit_dashboard_assets/da...
6,34,dashboard.summary,passed,1,outputs/34_final_streamlit_dashboard_assets/da...
7,34,dashboard.table.compute_summary,passed,35,outputs/34_final_streamlit_dashboard_assets/da...
8,34,dashboard.table.headline_findings,passed,8,outputs/34_final_streamlit_dashboard_assets/da...
9,34,dashboard.table.metric_framework,passed,178,outputs/34_final_streamlit_dashboard_assets/da...


Inventory and governance preflight passed.
Inventory run ID: inventory_20260903T140759Z_77bc8d8c
Indexed files: 18970
Registered artifacts: 218
Notebook 34 registry records: 17
Notebook 35 registry records: 2


In [4]:
HELPER_PATH = (
    PROJECT_ROOT
    / "src"
    / "restoration_eval"
    / "supervisor_package.py"
)

REQUIREMENTS_PATHS = [
    PROJECT_ROOT / "requirements.txt",
    PROJECT_ROOT / "requirements_experiments.txt",
]

HELPER_SOURCE = HELPER_PATH.read_text(
    encoding="utf-8"
)

try:
    ast.parse(
        HELPER_SOURCE,
        filename=str(HELPER_PATH),
    )
    HELPER_SYNTAX_VALID = True
    HELPER_SYNTAX_ERROR = ""
except SyntaxError as exc:
    HELPER_SYNTAX_VALID = False
    HELPER_SYNTAX_ERROR = (
        f"{exc.msg} at line {exc.lineno}, "
        f"column {exc.offset}"
    )

PACKAGE_DISTRIBUTIONS = {
    "pandas": "pandas",
    "numpy": "numpy",
    "pyyaml": "PyYAML",
    "pillow": "Pillow",
}

INSTALLED_VERSIONS = {}

for package_name in RUNTIME_CONFIG["packages"]:
    distribution_name = PACKAGE_DISTRIBUTIONS[
        str(package_name).lower()
    ]

    try:
        INSTALLED_VERSIONS[package_name] = (
            metadata.version(distribution_name)
        )
    except metadata.PackageNotFoundError:
        INSTALLED_VERSIONS[package_name] = "not_installed"

MISSING_PACKAGES = sorted(
    package
    for package, version in INSTALLED_VERSIONS.items()
    if version == "not_installed"
)

MISSING_COPY_SOURCES = (
    COPY_PLAN_TABLE.loc[
        ~COPY_PLAN_TABLE["source_path"].map(
            lambda value: (
                PROJECT_ROOT / value
            ).is_file()
        ),
        "source_path",
    ]
    .astype(str)
    .tolist()
)

MAXIMUM_RELATIVE_PATH_LENGTH = int(
    COPY_PLAN_TABLE["destination_path"]
    .astype(str)
    .str.len()
    .max()
)

SOURCE_CHECKS = (
    (
        "helper_syntax",
        "The Notebook 36 helper has valid Python syntax",
        "valid",
        HELPER_SYNTAX_ERROR or "valid",
        HELPER_SYNTAX_VALID,
    ),
    (
        "required_packages",
        "All required packaging dependencies are installed",
        [],
        MISSING_PACKAGES,
        not MISSING_PACKAGES,
    ),
    (
        "requirements_files",
        "Both environment declaration files exist",
        2,
        sum(path.is_file() for path in REQUIREMENTS_PATHS),
        all(path.is_file() for path in REQUIREMENTS_PATHS),
    ),
    (
        "copy_sources",
        "Every fixed package-copy source exists",
        [],
        MISSING_COPY_SOURCES,
        not MISSING_COPY_SOURCES,
    ),
    (
        "upstream_manifest_count",
        "Exactly 35 upstream run manifests are declared",
        35,
        len(CONFIG["upstream_run_manifests"]),
        len(CONFIG["upstream_run_manifests"]) == 35,
    ),
    (
        "thesis_figure_count",
        "The package includes all 18 thesis figures",
        18,
        OBSERVED_COPY_GROUP_COUNTS.get("thesis_figures", 0),
        OBSERVED_COPY_GROUP_COUNTS.get("thesis_figures", 0) == 18,
    ),
    (
        "publication_figure_count",
        "The package includes all six publication figures",
        6,
        OBSERVED_COPY_GROUP_COUNTS.get(
            "publication_figures",
            0,
        ),
        (
            OBSERVED_COPY_GROUP_COUNTS.get(
                "publication_figures",
                0,
            )
            == 6
        ),
    ),
    (
        "configuration_snapshot_count",
        "All 25 evaluation configurations are included",
        25,
        OBSERVED_COPY_GROUP_COUNTS.get("configuration", 0),
        OBSERVED_COPY_GROUP_COUNTS.get("configuration", 0) == 25,
    ),
    (
        "destination_path_length",
        "All package destinations satisfy the path-length contract",
        (
            f"<= "
            f"{PACKAGE_POLICY['maximum_relative_path_characters']}"
        ),
        MAXIMUM_RELATIVE_PATH_LENGTH,
        (
            MAXIMUM_RELATIVE_PATH_LENGTH
            <= int(
                PACKAGE_POLICY[
                    "maximum_relative_path_characters"
                ]
            )
        ),
    ),
)

for check_id, description, expected, observed, passed in SOURCE_CHECKS:
    VALIDATION.add(
        validation_stage="batch_1_dependencies_sources",
        check_id=check_id,
        check_description=description,
        severity="blocking",
        expected=expected,
        observed=observed,
        passed=bool(passed),
        details=(
            ""
            if passed
            else "A required dependency or fixed package source is unavailable."
        ),
    )

VALIDATION.raise_for_blocking()

DEPENDENCY_VIEW = pd.DataFrame(
    [
        {
            "package": package,
            "installed_version": version,
            "available": version != "not_installed",
        }
        for package, version in sorted(
            INSTALLED_VERSIONS.items()
        )
    ]
)

SOURCE_GROUP_VIEW = (
    COPY_PLAN_TABLE
    .groupby(["group", "role"], dropna=False)
    .agg(
        files=("source_path", "size"),
        size_bytes=("size_bytes", "sum"),
    )
    .reset_index()
)

SOURCE_GROUP_VIEW["size_mib"] = (
    SOURCE_GROUP_VIEW["size_bytes"] / (1024 ** 2)
)

display(DEPENDENCY_VIEW)
display(SOURCE_GROUP_VIEW)

print("Dependency and source-plan preflight passed.")
print("Missing packages:", MISSING_PACKAGES)
print("Missing copy sources:", len(MISSING_COPY_SOURCES))
print("Maximum destination path length:", MAXIMUM_RELATIVE_PATH_LENGTH)
print("Planned package files:", len(COPY_PLAN_TABLE))

,package,installed_version,available
0,numpy,1.26.4,True
1,pandas,2.3.3,True
2,pillow,9.5.0,True
3,pyyaml,6.0.3,True


,group,role,files,size_bytes,size_mib
0,application,dashboard_entrypoint,1,89402,0.085260
1,application,dashboard_helper,1,21784,0.020775
2,configuration,configuration_snapshot,25,456708,0.435551
3,environment,dashboard_environment,1,107,0.000102
4,environment,experiment_environment,1,848,0.000809
5,manifests,upstream_run_manifest,35,734530,0.700502
6,model_cards,model_card,4,40142,0.038282
7,publication_figures,publication_figure,6,658247,0.627753
8,reports,deployment_readiness,1,6525,0.006223
9,reports,final_self_contained_report,1,14109594,13.455957


Dependency and source-plan preflight passed.
Missing packages: []
Missing copy sources: 0
Maximum destination path length: 141
Planned package files: 106


In [5]:
CANONICAL_FILES_AFTER_BATCH = sorted(
    path.relative_to(OUTPUT_ROOT).as_posix()
    for path in OUTPUT_ROOT.rglob("*")
    if path.is_file()
)

VALIDATION.add(
    validation_stage="batch_1_final_preflight",
    check_id="no_canonical_files_written",
    check_description=(
        "Batch 1 created directories but persisted no "
        "canonical Notebook 36 files"
    ),
    severity="blocking",
    expected=[],
    observed=CANONICAL_FILES_AFTER_BATCH,
    passed=not CANONICAL_FILES_AFTER_BATCH,
    details=(
        ""
        if not CANONICAL_FILES_AFTER_BATCH
        else (
            "Remove stale Notebook 36 files and rerun "
            "Batch 1 from a clean output root."
        )
    ),
)

VALIDATION.raise_for_blocking()

BATCH_1_VALIDATION = VALIDATION.to_dataframe()

BATCH_1_STAGE_SUMMARY = (
    BATCH_1_VALIDATION
    .groupby(
        ["validation_stage", "severity"],
        dropna=False,
    )
    .agg(
        checks=("check_id", "size"),
        passed=("passed", "sum"),
    )
    .reset_index()
)

BATCH_1_STAGE_SUMMARY["failed"] = (
    BATCH_1_STAGE_SUMMARY["checks"]
    - BATCH_1_STAGE_SUMMARY["passed"]
)

display(BATCH_1_STAGE_SUMMARY)

print("Batch 1 preflight passed.")
print("Validation checks:", len(BATCH_1_VALIDATION))
print("Blocking failures:", len(VALIDATION.blocking_failures))
print("Upstream manifests declared:", len(CONFIG["upstream_run_manifests"]))
print("Fixed package-copy files:", len(COPY_PLAN_TABLE))
print("Estimated copied size MiB:", round(COPY_PLAN_SIZE_MIB, 3))
print("Registered project artifacts:", len(REGISTRY_ARTIFACTS))
print("Output root:", OUTPUT_ROOT)
print("Canonical files persisted by Batch 1: 0")

,validation_stage,severity,checks,passed,failed
0,batch_1_contract,blocking,11,11,0
1,batch_1_dependencies_sources,blocking,9,9,0
2,batch_1_final_preflight,blocking,1,1,0
3,batch_1_inventory_governance,blocking,15,15,0


Batch 1 preflight passed.
Validation checks: 36
Blocking failures: 0
Upstream manifests declared: 35
Fixed package-copy files: 106
Estimated copied size MiB: 27.596
Registered project artifacts: 218
Output root: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\36_supervisor_publication_reproducibility_package
Canonical files persisted by Batch 1: 0


## Batch 2 — Upstream completion and artifact-registry audit

This batch validates the completed Notebook 01–35 evidence chain before any final package content is assembled.

It verifies:

- exactly 35 upstream run manifests;
- unique notebook IDs and run IDs;
- `run_manifest.v1` schema coverage;
- completed run status and passed completion gates;
- zero upstream blocking failures;
- the documented Notebook 35 non-blocking warning;
- all 417 manifest-declared output records;
- repository-relative and existing declared output paths;
- all 218 project-path registry records;
- complete Notebook 01–35 producer coverage;
- unique artifact keys and normalized paths;
- valid SHA-256 declarations;
- the expected 216 passed and two warning registry records;
- the exact registered artifacts needed for the final package;
- the continued visibility of indexed-but-not-bundled report and visual collections.

This batch reads metadata and validates paths. It does not recompute checksums for large upstream directory collections, copy package files, or persist canonical Notebook 36 outputs.

In [6]:
from restoration_eval.supervisor_package import (
    load_upstream_manifests,
)


UPSTREAM_MANIFESTS = load_upstream_manifests(
    CONFIG,
    PROJECT_ROOT,
)

RAW_UPSTREAM_MANIFESTS = []
MANIFEST_OUTPUT_RECORDS = []

for manifest_position, manifest_relative_path in enumerate(
    CONFIG["upstream_run_manifests"],
    start=1,
):
    manifest_path = safe_repo_path(
        manifest_relative_path,
        PROJECT_ROOT,
        must_exist=True,
    )

    with manifest_path.open(
        "r",
        encoding="utf-8-sig",
    ) as handle:
        manifest_payload = json.load(handle)

    RAW_UPSTREAM_MANIFESTS.append(
        {
            "manifest_position": manifest_position,
            "manifest_path": (
                manifest_path
                .relative_to(PROJECT_ROOT)
                .as_posix()
            ),
            "manifest_schema_version": (
                manifest_payload.get(
                    "manifest_schema_version",
                    "",
                )
            ),
            "notebook_id": str(
                manifest_payload.get("notebook_id", "")
            ),
            "notebook_name": str(
                manifest_payload.get("notebook_name", "")
            ),
            "run_id": str(
                manifest_payload.get("run_id", "")
            ),
            "run_status": str(
                manifest_payload.get("run_status", "")
            ),
            "validation_status": str(
                manifest_payload.get(
                    "validation_status",
                    "",
                )
            ),
            "completion_gate_passed": bool(
                manifest_payload.get(
                    "completion_gate_passed",
                    False,
                )
            ),
            "output_record_count": len(
                manifest_payload.get("outputs", [])
            ),
        }
    )

    for output_position, output_record in enumerate(
        manifest_payload.get("outputs", []),
        start=1,
    ):
        relative_path = str(
            output_record.get("relative_path", "")
        ).strip()

        output_path = (
            (PROJECT_ROOT / relative_path).resolve()
            if relative_path
            else PROJECT_ROOT
        )

        MANIFEST_OUTPUT_RECORDS.append(
            {
                "notebook_id": str(
                    manifest_payload.get(
                        "notebook_id",
                        "",
                    )
                ),
                "output_position": output_position,
                "artifact_key": str(
                    output_record.get(
                        "artifact_key",
                        output_record.get(
                            "output_key",
                            "",
                        ),
                    )
                ),
                "relative_path": relative_path,
                "path_is_normalized": (
                    bool(relative_path)
                    and "\\" not in relative_path
                    and not Path(
                        relative_path
                    ).is_absolute()
                ),
                "path_is_inside_repository": (
                    output_path.is_relative_to(
                        PROJECT_ROOT
                    )
                ),
                "path_exists": (
                    bool(relative_path)
                    and output_path.exists()
                ),
            }
        )

MANIFEST_DETAILS = (
    pd.DataFrame(RAW_UPSTREAM_MANIFESTS)
    .sort_values("notebook_id", kind="stable")
    .reset_index(drop=True)
)

MANIFEST_OUTPUTS = pd.DataFrame(
    MANIFEST_OUTPUT_RECORDS
)

EXPECTED_NOTEBOOK_IDS = [
    f"{number:02d}"
    for number in range(1, 36)
]

EXPECTED_MANIFEST_PATHS = sorted(
    Path(path).as_posix()
    for path in CONFIG["upstream_run_manifests"]
)

OBSERVED_MANIFEST_PATHS = sorted(
    MANIFEST_DETAILS["manifest_path"]
    .astype(str)
    .tolist()
)

VALIDATION_STATUS_COUNTS = {
    str(status): int(count)
    for status, count in (
        MANIFEST_DETAILS[
            "validation_status"
        ]
        .value_counts(dropna=False)
        .sort_index()
        .items()
    )
}

NONPASSED_MANIFESTS = (
    MANIFEST_DETAILS.loc[
        MANIFEST_DETAILS[
            "validation_status"
        ].ne("passed"),
        [
            "notebook_id",
            "notebook_name",
            "validation_status",
        ],
    ]
    .reset_index(drop=True)
)

EXPECTED_NONPASSED_MANIFESTS = [
    {
        "notebook_id": "35",
        "validation_status": "warning",
    }
]

OBSERVED_NONPASSED_MANIFESTS = (
    NONPASSED_MANIFESTS[
        [
            "notebook_id",
            "validation_status",
        ]
    ]
    .to_dict(orient="records")
)

MISSING_MANIFEST_OUTPUTS = (
    MANIFEST_OUTPUTS.loc[
        ~MANIFEST_OUTPUTS["path_exists"],
        [
            "notebook_id",
            "artifact_key",
            "relative_path",
        ],
    ]
    .to_dict(orient="records")
)

INVALID_MANIFEST_OUTPUT_PATHS = (
    MANIFEST_OUTPUTS.loc[
        ~(
            MANIFEST_OUTPUTS[
                "path_is_normalized"
            ]
            & MANIFEST_OUTPUTS[
                "path_is_inside_repository"
            ]
        ),
        [
            "notebook_id",
            "artifact_key",
            "relative_path",
        ],
    ]
    .to_dict(orient="records")
)

MANIFEST_CHECKS = (
    (
        "manifest_count",
        "Exactly 35 upstream manifests are loaded",
        35,
        len(MANIFEST_DETAILS),
        len(MANIFEST_DETAILS) == 35,
    ),
    (
        "notebook_id_sequence",
        "The manifests cover Notebook 01 through Notebook 35 exactly once",
        EXPECTED_NOTEBOOK_IDS,
        MANIFEST_DETAILS[
            "notebook_id"
        ].tolist(),
        (
            MANIFEST_DETAILS[
                "notebook_id"
            ].tolist()
            == EXPECTED_NOTEBOOK_IDS
        ),
    ),
    (
        "manifest_paths",
        "Loaded manifest paths match the fixed configuration",
        EXPECTED_MANIFEST_PATHS,
        OBSERVED_MANIFEST_PATHS,
        (
            OBSERVED_MANIFEST_PATHS
            == EXPECTED_MANIFEST_PATHS
        ),
    ),
    (
        "manifest_schema",
        "Every upstream manifest uses run_manifest.v1",
        ["run_manifest.v1"],
        sorted(
            MANIFEST_DETAILS[
                "manifest_schema_version"
            ].unique().tolist()
        ),
        MANIFEST_DETAILS[
            "manifest_schema_version"
        ].eq("run_manifest.v1").all(),
    ),
    (
        "unique_notebook_ids",
        "Upstream notebook IDs are unique",
        True,
        bool(
            MANIFEST_DETAILS[
                "notebook_id"
            ].is_unique
        ),
        bool(
            MANIFEST_DETAILS[
                "notebook_id"
            ].is_unique
        ),
    ),
    (
        "unique_run_ids",
        "Every upstream notebook has a unique run ID",
        True,
        bool(
            MANIFEST_DETAILS["run_id"].is_unique
            and MANIFEST_DETAILS[
                "run_id"
            ].ne("").all()
        ),
        bool(
            MANIFEST_DETAILS["run_id"].is_unique
            and MANIFEST_DETAILS[
                "run_id"
            ].ne("").all()
        ),
    ),
    (
        "completed_runs",
        "Every upstream run is completed",
        35,
        int(
            MANIFEST_DETAILS[
                "run_status"
            ].eq("completed").sum()
        ),
        MANIFEST_DETAILS[
            "run_status"
        ].eq("completed").all(),
    ),
    (
        "completion_gates",
        "Every upstream completion gate passed",
        35,
        int(
            MANIFEST_DETAILS[
                "completion_gate_passed"
            ].sum()
        ),
        MANIFEST_DETAILS[
            "completion_gate_passed"
        ].all(),
    ),
    (
        "blocking_failures",
        "No upstream manifest reports a blocking failure",
        0,
        int(
            UPSTREAM_MANIFESTS[
                "blocking_failures"
            ].sum()
        ),
        (
            int(
                UPSTREAM_MANIFESTS[
                    "blocking_failures"
                ].sum()
            )
            == 0
        ),
    ),
    (
        "validation_statuses",
        "Only Notebook 35 retains a documented non-blocking warning",
        {
            "passed": 34,
            "warning": 1,
        },
        VALIDATION_STATUS_COUNTS,
        (
            VALIDATION_STATUS_COUNTS
            == {
                "passed": 34,
                "warning": 1,
            }
        ),
    ),
    (
        "nonpassed_manifest_identity",
        "The non-passed manifest is the approved Notebook 35 warning",
        EXPECTED_NONPASSED_MANIFESTS,
        OBSERVED_NONPASSED_MANIFESTS,
        (
            OBSERVED_NONPASSED_MANIFESTS
            == EXPECTED_NONPASSED_MANIFESTS
        ),
    ),
    (
        "manifest_output_record_count",
        "The 35 manifests declare 417 output records",
        417,
        len(MANIFEST_OUTPUTS),
        len(MANIFEST_OUTPUTS) == 417,
    ),
    (
        "manifest_output_paths",
        "Every declared output path is normalized and repository-relative",
        [],
        INVALID_MANIFEST_OUTPUT_PATHS,
        not INVALID_MANIFEST_OUTPUT_PATHS,
    ),
    (
        "manifest_output_existence",
        "Every manifest-declared output currently exists",
        [],
        MISSING_MANIFEST_OUTPUTS,
        not MISSING_MANIFEST_OUTPUTS,
    ),
)

for check_id, description, expected, observed, passed in MANIFEST_CHECKS:
    VALIDATION.add(
        validation_stage="batch_2_upstream_manifests",
        check_id=check_id,
        check_description=description,
        severity="blocking",
        expected=expected,
        observed=observed,
        passed=bool(passed),
        details=(
            ""
            if passed
            else (
                "The completed Notebook 01–35 "
                "manifest chain is inconsistent."
            )
        ),
    )

VALIDATION.raise_for_blocking()

MANIFEST_DISPLAY = MANIFEST_DETAILS[
    [
        "notebook_id",
        "notebook_name",
        "run_status",
        "validation_status",
        "completion_gate_passed",
        "output_record_count",
    ]
]

display(MANIFEST_DISPLAY)
display(NONPASSED_MANIFESTS)

print("Upstream manifest audit passed.")
print("Completed manifests:", len(MANIFEST_DETAILS))
print("Completion gates passed:", int(
    MANIFEST_DETAILS["completion_gate_passed"].sum()
))
print("Declared output records:", len(MANIFEST_OUTPUTS))
print("Missing declared outputs:", len(MISSING_MANIFEST_OUTPUTS))
print("Blocking failures:", int(
    UPSTREAM_MANIFESTS["blocking_failures"].sum()
))

,notebook_id,notebook_name,run_status,validation_status,completion_gate_passed,output_record_count
0,01,Dataset Verification,completed,passed,True,7
1,02,Image Preprocessing,completed,passed,True,7
2,03,Canonical Mask Generation,completed,passed,True,9
3,04,Canonical Damaged-Image Generation,completed,passed,True,7
4,05,Damage-Size Sensitivity Dataset Generation,completed,passed,True,8
5,06,Mask Robustness Dataset Generation,completed,passed,True,8
6,07,Synthetic Degradation Dataset Generation,completed,passed,True,9
7,08,Experiment Contracts and Region Policy,completed,passed,True,9
8,09,OpenCV Telea Restoration,completed,passed,True,7
9,10,LaMa Restoration,completed,passed,True,7


,notebook_id,notebook_name,validation_status
0,35,35_dashboard_and_deployment_validation,warning


Upstream manifest audit passed.
Completed manifests: 35
Completion gates passed: 35
Declared output records: 417
Missing declared outputs: 0
Blocking failures: 0


In [7]:
REGISTRY_REQUIRED_FIELDS = {
    "artifact_key",
    "producer_notebook",
    "relative_path",
    "artifact_type",
    "artifact_role",
    "schema_version",
    "dataset_scope",
    "experiment_scope",
    "validation_status",
    "row_count",
    "file_count",
    "checksum",
}

REGISTRY_PATH_RECORDS = []

for artifact in PROJECT_PATHS_REGISTRY["artifacts"]:
    relative_path = str(
        artifact.get("relative_path", "")
    ).strip()

    resolved_path = (
        (PROJECT_ROOT / relative_path).resolve()
        if relative_path
        else PROJECT_ROOT
    )

    REGISTRY_PATH_RECORDS.append(
        {
            "artifact_key": str(
                artifact.get("artifact_key", "")
            ),
            "producer_notebook": str(
                artifact.get(
                    "producer_notebook",
                    "",
                )
            ),
            "relative_path": relative_path,
            "validation_status": str(
                artifact.get(
                    "validation_status",
                    "",
                )
            ),
            "path_is_normalized": (
                bool(relative_path)
                and "\\" not in relative_path
                and not Path(
                    relative_path
                ).is_absolute()
            ),
            "path_is_inside_repository": (
                resolved_path.is_relative_to(
                    PROJECT_ROOT
                )
            ),
            "path_exists": (
                bool(relative_path)
                and resolved_path.exists()
            ),
        }
    )

REGISTRY_PATH_AUDIT = pd.DataFrame(
    REGISTRY_PATH_RECORDS
)

REGISTRY_PRODUCER_SUMMARY = (
    REGISTRY_ARTIFACTS
    .groupby(
        [
            "producer_notebook",
            "validation_status",
        ],
        dropna=False,
    )
    .agg(
        artifact_count=("artifact_key", "size"),
        declared_files=("file_count", "sum"),
    )
    .reset_index()
    .sort_values(
        [
            "producer_notebook",
            "validation_status",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

EXPECTED_PRODUCER_IDS = [
    f"{number:02d}"
    for number in range(1, 36)
]

OBSERVED_PRODUCER_IDS = sorted(
    {
        str(value)[:2]
        for value in REGISTRY_ARTIFACTS[
            "producer_notebook"
        ]
    }
)

REGISTRY_STATUS_COUNTS = {
    str(status): int(count)
    for status, count in (
        REGISTRY_ARTIFACTS[
            "validation_status"
        ]
        .value_counts(dropna=False)
        .sort_index()
        .items()
    )
}

MISSING_REGISTRY_PATHS = (
    REGISTRY_PATH_AUDIT.loc[
        ~REGISTRY_PATH_AUDIT["path_exists"],
        [
            "artifact_key",
            "producer_notebook",
            "relative_path",
        ],
    ]
    .to_dict(orient="records")
)

INVALID_REGISTRY_PATHS = (
    REGISTRY_PATH_AUDIT.loc[
        ~(
            REGISTRY_PATH_AUDIT[
                "path_is_normalized"
            ]
            & REGISTRY_PATH_AUDIT[
                "path_is_inside_repository"
            ]
        ),
        [
            "artifact_key",
            "producer_notebook",
            "relative_path",
        ],
    ]
    .to_dict(orient="records")
)

INVALID_DECLARED_CHECKSUMS = (
    REGISTRY_ARTIFACTS.loc[
        ~(
            REGISTRY_ARTIFACTS[
                "checksum"
            ]
            .fillna("")
            .astype(str)
            .str.fullmatch(r"[0-9a-f]{64}")
        ),
        [
            "artifact_key",
            "producer_notebook",
            "checksum",
        ],
    ]
    .to_dict(orient="records")
)

PACKAGE_EVIDENCE_KEYS = {
    "model_cards_compute.model_cards",
    "model_cards_compute.model_card_reports",
    "model_reports.lama",
    "model_reports.opencv_telea",
    "model_reports.stable_diffusion_inpainting",
    "model_reports.sdxl_inpainting",
    "model_reports.report_index",
    "case_painting_reports.case_report_index",
    "case_painting_reports.painting_report_index",
    "case_painting_reports.selected_cases",
    "case_painting_reports.case_reports",
    "case_painting_reports.painting_reports",
    "case_painting_reports.selected_case_grids",
    "final_report.html",
    "final_report.thesis_figures",
    "final_report.publication_figures",
    "final_report.thesis_tables",
    "final_report.latex_tables",
    "final_report.evidence_catalog",
    "dashboard.index.case_index",
    "dashboard.index.painting_index",
    "dashboard.index.visual_asset_index",
    "dashboard.index.report_index",
    "dashboard.deployment_readiness",
    "dashboard.deployment_validation",
}

REGISTERED_ARTIFACT_KEYS = set(
    REGISTRY_ARTIFACTS[
        "artifact_key"
    ].astype(str)
)

MISSING_PACKAGE_EVIDENCE_KEYS = sorted(
    PACKAGE_EVIDENCE_KEYS
    - REGISTERED_ARTIFACT_KEYS
)

N35_WARNING_ARTIFACTS = (
    REGISTRY_ARTIFACTS.loc[
        (
            REGISTRY_ARTIFACTS[
                "producer_notebook"
            ]
            == (
                "35_dashboard_and_"
                "deployment_validation"
            )
        ),
        [
            "artifact_key",
            "validation_status",
            "relative_path",
        ],
    ]
    .sort_values("artifact_key")
    .reset_index(drop=True)
)

REGISTRY_CHECKS = (
    (
        "registry_required_fields",
        "The registry contains every canonical field",
        sorted(REGISTRY_REQUIRED_FIELDS),
        sorted(
            REGISTRY_REQUIRED_FIELDS
            & set(REGISTRY_ARTIFACTS.columns)
        ),
        REGISTRY_REQUIRED_FIELDS.issubset(
            REGISTRY_ARTIFACTS.columns
        ),
    ),
    (
        "registry_record_count",
        "The project registry contains 218 artifacts",
        218,
        len(REGISTRY_ARTIFACTS),
        len(REGISTRY_ARTIFACTS) == 218,
    ),
    (
        "registry_unique_keys",
        "Every registered artifact key is unique",
        True,
        bool(
            REGISTRY_ARTIFACTS[
                "artifact_key"
            ].is_unique
        ),
        bool(
            REGISTRY_ARTIFACTS[
                "artifact_key"
            ].is_unique
        ),
    ),
    (
        "registry_producer_coverage",
        "The registry covers Notebook 01 through Notebook 35",
        EXPECTED_PRODUCER_IDS,
        OBSERVED_PRODUCER_IDS,
        (
            OBSERVED_PRODUCER_IDS
            == EXPECTED_PRODUCER_IDS
        ),
    ),
    (
        "registry_status_counts",
        "The registry preserves 216 passed and two warning records",
        {
            "passed": 216,
            "warning": 2,
        },
        REGISTRY_STATUS_COUNTS,
        (
            REGISTRY_STATUS_COUNTS
            == {
                "passed": 216,
                "warning": 2,
            }
        ),
    ),
    (
        "registry_paths",
        "All registry paths are normalized and repository-relative",
        [],
        INVALID_REGISTRY_PATHS,
        not INVALID_REGISTRY_PATHS,
    ),
    (
        "registry_path_existence",
        "Every registered artifact currently exists",
        [],
        MISSING_REGISTRY_PATHS,
        not MISSING_REGISTRY_PATHS,
    ),
    (
        "registry_checksum_declarations",
        "Every registry record contains a SHA-256 declaration",
        [],
        INVALID_DECLARED_CHECKSUMS,
        not INVALID_DECLARED_CHECKSUMS,
    ),
    (
        "package_evidence_keys",
        "All package and indexed-omission evidence is registered",
        [],
        MISSING_PACKAGE_EVIDENCE_KEYS,
        not MISSING_PACKAGE_EVIDENCE_KEYS,
    ),
    (
        "notebook_35_warning_artifacts",
        "Both Notebook 35 artifacts retain non-blocking warning status",
        2,
        int(
            N35_WARNING_ARTIFACTS[
                "validation_status"
            ].eq("warning").sum()
        ),
        (
            len(N35_WARNING_ARTIFACTS) == 2
            and N35_WARNING_ARTIFACTS[
                "validation_status"
            ].eq("warning").all()
        ),
    ),
)

for check_id, description, expected, observed, passed in REGISTRY_CHECKS:
    VALIDATION.add(
        validation_stage="batch_2_artifact_registry",
        check_id=check_id,
        check_description=description,
        severity="blocking",
        expected=expected,
        observed=observed,
        passed=bool(passed),
        details=(
            ""
            if passed
            else (
                "The artifact registry cannot support "
                "the final package contract."
            )
        ),
    )

VALIDATION.raise_for_blocking()

display(REGISTRY_PRODUCER_SUMMARY)
display(N35_WARNING_ARTIFACTS)

print("Artifact-registry audit passed.")
print("Registered artifacts:", len(REGISTRY_ARTIFACTS))
print("Registered producers:", len(OBSERVED_PRODUCER_IDS))
print("Passed artifacts:", REGISTRY_STATUS_COUNTS.get("passed", 0))
print("Warning artifacts:", REGISTRY_STATUS_COUNTS.get("warning", 0))
print("Missing registered paths:", len(MISSING_REGISTRY_PATHS))
print(
    "Missing package evidence keys:",
    len(MISSING_PACKAGE_EVIDENCE_KEYS),
)

,producer_notebook,validation_status,artifact_count,declared_files
0,01_dataset_verification,passed,5,5
1,02_image_preprocessing,passed,5,54
2,03_canonical_mask_generation,passed,7,256
3,04_canonical_damaged_image_generation,passed,5,254
4,05_damage_size_sensitivity_dataset_generation,passed,6,74
5,06_mask_robustness_dataset_generation,passed,6,154
6,07_synthetic_degradation_dataset_generation,passed,7,335
7,08_experiment_contracts_and_region_policy,passed,7,7
8,09_opencv_telea_restoration,passed,5,414
9,10_lama_restoration,passed,5,414


,artifact_key,validation_status,relative_path
0,dashboard.deployment_readiness,warning,outputs/35_dashboard_and_deployment_validation...
1,dashboard.deployment_validation,warning,outputs/35_dashboard_and_deployment_validation...


Artifact-registry audit passed.
Registered artifacts: 218
Registered producers: 35
Passed artifacts: 216
Warning artifacts: 2
Missing registered paths: 0
Missing package evidence keys: 0


In [8]:
CANONICAL_FILES_AFTER_BATCH_2 = sorted(
    path.relative_to(OUTPUT_ROOT).as_posix()
    for path in OUTPUT_ROOT.rglob("*")
    if path.is_file()
)

VALIDATION.add(
    validation_stage="batch_2_final_audit",
    check_id="no_canonical_files_written",
    check_description=(
        "Batch 2 remained a read-only metadata and path audit"
    ),
    severity="blocking",
    expected=[],
    observed=CANONICAL_FILES_AFTER_BATCH_2,
    passed=not CANONICAL_FILES_AFTER_BATCH_2,
    details=(
        ""
        if not CANONICAL_FILES_AFTER_BATCH_2
        else (
            "Batch 2 must not persist package files "
            "or canonical Notebook 36 outputs."
        )
    ),
)

VALIDATION.raise_for_blocking()

BATCH_2_VALIDATION = VALIDATION.to_dataframe()

BATCH_2_STAGE_SUMMARY = (
    BATCH_2_VALIDATION
    .groupby(
        ["validation_stage", "severity"],
        dropna=False,
    )
    .agg(
        checks=("check_id", "size"),
        passed=("passed", "sum"),
    )
    .reset_index()
)

BATCH_2_STAGE_SUMMARY["failed"] = (
    BATCH_2_STAGE_SUMMARY["checks"]
    - BATCH_2_STAGE_SUMMARY["passed"]
)

display(BATCH_2_STAGE_SUMMARY)

print("Batch 2 upstream evidence audit passed.")
print("Cumulative validation checks:", len(BATCH_2_VALIDATION))
print("Blocking failures:", len(VALIDATION.blocking_failures))
print("Completed upstream notebooks:", len(MANIFEST_DETAILS))
print("Manifest-declared outputs:", len(MANIFEST_OUTPUTS))
print("Registered artifacts:", len(REGISTRY_ARTIFACTS))
print("Package evidence keys:", len(PACKAGE_EVIDENCE_KEYS))
print("Canonical files persisted through Batch 2: 0")

,validation_stage,severity,checks,passed,failed
0,batch_1_contract,blocking,11,11,0
1,batch_1_dependencies_sources,blocking,9,9,0
2,batch_1_final_preflight,blocking,1,1,0
3,batch_1_inventory_governance,blocking,15,15,0
4,batch_2_artifact_registry,blocking,10,10,0
5,batch_2_final_audit,blocking,1,1,0
6,batch_2_upstream_manifests,blocking,14,14,0


Batch 2 upstream evidence audit passed.
Cumulative validation checks: 61
Blocking failures: 0
Completed upstream notebooks: 35
Manifest-declared outputs: 417
Registered artifacts: 218
Package evidence keys: 25
Canonical files persisted through Batch 2: 0


## Batch 3 — Evidence synthesis and research-question mapping

This batch loads the normalized final evidence used to construct the supervisor package.

It:

- verifies the final evidence population against the Notebook 36 contract;
- loads the eight approved headline findings;
- loads the exact research-question coverage records;
- separates approved comparison candidates from additional executed candidates;
- combines model coverage, quality-anchor wins, and observed runtime evidence;
- constructs concise model conclusions and limitations;
- reproduces the three proposal research questions exactly;
- maps every research-question answer to validated source notebooks and paths;
- adds direct plain-language implications without strengthening the upstream claims;
- constructs the eight-record key-finding table used by later reports.

This is evidence synthesis only. It does not calculate new scientific metrics or persist Notebook 36 outputs.

In [9]:
DASHBOARD_SUMMARY_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "34_final_streamlit_dashboard_assets"
    / "data"
    / "dashboard_summary.json"
)

HEADLINE_FINDINGS_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "34_final_streamlit_dashboard_assets"
    / "data"
    / "dashboard_tables"
    / "headline_findings.csv"
)

RQ_COVERAGE_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "34_final_streamlit_dashboard_assets"
    / "data"
    / "dashboard_tables"
    / "research_question_coverage.csv"
)

CASE_INDEX_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "34_final_streamlit_dashboard_assets"
    / "data"
    / "dashboard_indexes"
    / "case_index.csv"
)

MODEL_CARDS_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "30_model_cards_compute_and_scalability"
    / "data"
    / "model_cards.csv"
)

COMPUTE_SCALABILITY_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "30_model_cards_compute_and_scalability"
    / "metrics"
    / "compute_scalability.csv"
)

MODEL_REPORT_INDEX_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "31_model_report_generation"
    / "data"
    / "report_index.csv"
)

CASE_REPORT_INDEX_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "32_case_and_painting_report_generation"
    / "data"
    / "case_report_index.csv"
)

PAINTING_REPORT_INDEX_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "32_case_and_painting_report_generation"
    / "data"
    / "painting_report_index.csv"
)

THESIS_TABLES_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "33_final_evaluation_report"
    / "data"
    / "thesis_tables.csv"
)

with DASHBOARD_SUMMARY_PATH.open(
    "r",
    encoding="utf-8-sig",
) as handle:
    DASHBOARD_SUMMARY = json.load(handle)

HEADLINE_FINDINGS = pd.read_csv(
    HEADLINE_FINDINGS_PATH,
    low_memory=False,
)

RQ_COVERAGE = pd.read_csv(
    RQ_COVERAGE_PATH,
    low_memory=False,
)

CASE_INDEX = pd.read_csv(
    CASE_INDEX_PATH,
    low_memory=False,
)

MODEL_CARDS = pd.read_csv(
    MODEL_CARDS_PATH,
    low_memory=False,
)

COMPUTE_SCALABILITY = pd.read_csv(
    COMPUTE_SCALABILITY_PATH,
    low_memory=False,
)

MODEL_REPORT_INDEX = pd.read_csv(
    MODEL_REPORT_INDEX_PATH,
    low_memory=False,
)

CASE_REPORT_INDEX = pd.read_csv(
    CASE_REPORT_INDEX_PATH,
    low_memory=False,
)

PAINTING_REPORT_INDEX = pd.read_csv(
    PAINTING_REPORT_INDEX_PATH,
    low_memory=False,
)

THESIS_TABLES = pd.read_csv(
    THESIS_TABLES_PATH,
    low_memory=False,
)

DASHBOARD_POPULATION = DASHBOARD_SUMMARY["population"]

OBSERVED_EVIDENCE_POPULATION = {
    "upstream_notebooks": len(MANIFEST_DETAILS),
    "paintings": int(
        DASHBOARD_POPULATION["paintings"]
    ),
    "registered_cases": int(
        DASHBOARD_POPULATION["registered_cases"]
    ),
    "restoration_cases": int(
        DASHBOARD_POPULATION["restoration_cases"]
    ),
    "approved_candidates": int(
        DASHBOARD_POPULATION["approved_candidates"]
    ),
    "quality_anchors": 11,
    "uncertainty_groups": int(
        DASHBOARD_POPULATION[
            "canonical_uncertainty_groups"
        ]
        + DASHBOARD_POPULATION[
            "damage_size_uncertainty_groups"
        ]
    ),
    "visual_records": int(
        DASHBOARD_POPULATION["visual_records"]
    ),
    "indexed_reports": int(
        DASHBOARD_POPULATION["reports"]
    ),
    "sdxl_cases": int(
        DASHBOARD_POPULATION["sdxl_candidates"]
    ),
    "thesis_figures": int(
        OBSERVED_COPY_GROUP_COUNTS[
            "thesis_figures"
        ]
    ),
    "publication_figures": int(
        OBSERVED_COPY_GROUP_COUNTS[
            "publication_figures"
        ]
    ),
    "model_reports": len(MODEL_REPORT_INDEX),
    "model_cards": len(MODEL_CARDS),
    "case_report_records": len(CASE_REPORT_INDEX),
    "painting_report_records": len(
        PAINTING_REPORT_INDEX
    ),
}

EXPECTED_EVIDENCE_POPULATION = {
    key: int(value)
    for key, value in EXPECTED_POPULATION.items()
}

EVIDENCE_INPUT_CHECKS = (
    (
        "dashboard_summary_schema",
        "The dashboard summary uses dashboard_package.v1",
        "dashboard_package.v1",
        DASHBOARD_SUMMARY.get("schema_version"),
        (
            DASHBOARD_SUMMARY.get("schema_version")
            == "dashboard_package.v1"
        ),
    ),
    (
        "headline_finding_schema",
        "Headline findings use the approved package schema",
        ["dashboard_package.v1"],
        sorted(
            HEADLINE_FINDINGS[
                "schema_version"
            ].unique().tolist()
        ),
        HEADLINE_FINDINGS[
            "schema_version"
        ].eq("dashboard_package.v1").all(),
    ),
    (
        "research_question_schema",
        "Research-question coverage uses the approved package schema",
        ["dashboard_package.v1"],
        sorted(
            RQ_COVERAGE[
                "schema_version"
            ].unique().tolist()
        ),
        RQ_COVERAGE[
            "schema_version"
        ].eq("dashboard_package.v1").all(),
    ),
    (
        "model_card_schema",
        "Model cards use model_cards.v1",
        ["model_cards.v1"],
        sorted(
            MODEL_CARDS[
                "schema_version"
            ].unique().tolist()
        ),
        MODEL_CARDS[
            "schema_version"
        ].eq("model_cards.v1").all(),
    ),
    (
        "thesis_table_schema",
        "Final thesis tables use thesis_tables.v1",
        ["thesis_tables.v1"],
        sorted(
            THESIS_TABLES[
                "schema_version"
            ].unique().tolist()
        ),
        THESIS_TABLES[
            "schema_version"
        ].eq("thesis_tables.v1").all(),
    ),
    (
        "source_table_status",
        "Every loaded normalized evidence row has status ok",
        ["ok"],
        sorted(
            set(
                HEADLINE_FINDINGS["status"]
                .dropna()
                .astype(str)
            )
            | set(
                RQ_COVERAGE["status"]
                .dropna()
                .astype(str)
            )
            | set(
                MODEL_CARDS["status"]
                .dropna()
                .astype(str)
            )
            | set(
                THESIS_TABLES["status"]
                .dropna()
                .astype(str)
            )
        ),
        all(
            frame["status"].eq("ok").all()
            for frame in (
                HEADLINE_FINDINGS,
                RQ_COVERAGE,
                MODEL_CARDS,
                THESIS_TABLES,
            )
        ),
    ),
    (
        "headline_finding_count",
        "All eight approved headline findings are available",
        8,
        len(HEADLINE_FINDINGS),
        len(HEADLINE_FINDINGS) == 8,
    ),
    (
        "research_question_record_count",
        "Three research questions and one practical-output record are available",
        4,
        len(RQ_COVERAGE),
        len(RQ_COVERAGE) == 4,
    ),
    (
        "evidence_population",
        "The loaded evidence exactly matches the Notebook 36 contract",
        EXPECTED_EVIDENCE_POPULATION,
        OBSERVED_EVIDENCE_POPULATION,
        (
            OBSERVED_EVIDENCE_POPULATION
            == EXPECTED_EVIDENCE_POPULATION
        ),
    ),
)

for check_id, description, expected, observed, passed in EVIDENCE_INPUT_CHECKS:
    VALIDATION.add(
        validation_stage="batch_3_evidence_inputs",
        check_id=check_id,
        check_description=description,
        severity="blocking",
        expected=expected,
        observed=observed,
        passed=bool(passed),
        details=(
            ""
            if passed
            else (
                "Normalized final evidence does not "
                "match the approved Notebook 36 scope."
            )
        ),
    )

VALIDATION.raise_for_blocking()

EVIDENCE_POPULATION_VIEW = pd.DataFrame(
    [
        {
            "population_item": key,
            "expected": EXPECTED_EVIDENCE_POPULATION[key],
            "observed": OBSERVED_EVIDENCE_POPULATION[key],
            "matches": (
                EXPECTED_EVIDENCE_POPULATION[key]
                == OBSERVED_EVIDENCE_POPULATION[key]
            ),
        }
        for key in EXPECTED_EVIDENCE_POPULATION
    ]
)

display(EVIDENCE_POPULATION_VIEW)

print("Normalized evidence inputs loaded.")
print("Headline findings:", len(HEADLINE_FINDINGS))
print("Research-question records:", len(RQ_COVERAGE))
print("Model cards:", len(MODEL_CARDS))
print("Evidence population mismatches:", int(
    (~EVIDENCE_POPULATION_VIEW["matches"]).sum()
))

,population_item,expected,observed,matches
0,upstream_notebooks,35,35,True
1,paintings,50,50,True
2,registered_cases,525,525,True
3,restoration_cases,410,410,True
4,approved_candidates,1785,1785,True
5,quality_anchors,11,11,True
6,uncertainty_groups,165,165,True
7,visual_records,23964,23964,True
8,indexed_reports,104,104,True
9,sdxl_cases,10,10,True


Normalized evidence inputs loaded.
Headline findings: 8
Research-question records: 4
Model cards: 4
Evidence population mismatches: 0


In [10]:
APPROVED_MODEL_COUNTS = (
    CASE_INDEX
    .groupby(
        "model_id",
        dropna=False,
    )
    .agg(
        approved_candidate_count=(
            "candidate_id",
            "size",
        ),
        approved_case_count=(
            "case_id",
            "nunique",
        ),
        approved_painting_count=(
            "painting_id",
            "nunique",
        ),
    )
    .reset_index()
)

OVERALL_COMPUTE = (
    COMPUTE_SCALABILITY.loc[
        (
            COMPUTE_SCALABILITY[
                "record_type"
            ].eq("observed")
        )
        & (
            COMPUTE_SCALABILITY[
                "scenario_id"
            ].eq("observed_overall_all")
        ),
        [
            "model_id",
            "candidate_count",
            "inference_count",
            "total_runtime_seconds",
            "mean_runtime_seconds",
            "median_runtime_seconds",
            "p95_runtime_seconds",
            "failure_rate",
            "applicability_status",
        ],
    ]
    .copy()
)

NUMERIC_COMPUTE_COLUMNS = [
    "candidate_count",
    "inference_count",
    "total_runtime_seconds",
    "mean_runtime_seconds",
    "median_runtime_seconds",
    "p95_runtime_seconds",
    "failure_rate",
]

for column in NUMERIC_COMPUTE_COLUMNS:
    OVERALL_COMPUTE[column] = pd.to_numeric(
        OVERALL_COMPUTE[column],
        errors="coerce",
    )

OVERALL_COMPUTE = OVERALL_COMPUTE.rename(
    columns={
        "candidate_count": (
            "executed_candidate_count"
        ),
        "inference_count": (
            "executed_inference_count"
        ),
    }
)

QUALITY_ANCHOR_WINS = {
    "opencv_telea": 1,
    "lama": 10,
    "stable_diffusion_inpainting": 0,
    "sdxl_inpainting": pd.NA,
}

MODEL_CONCLUSION_MAP = {
    "opencv_telea": (
        "OpenCV Telea is the fastest evaluated baseline "
        "and leads crop SSIM. It is useful when speed and "
        "deterministic local filling matter, but it is not "
        "the strongest general model across the complete "
        "metric framework."
    ),
    "lama": (
        "LaMa leads 10 of 11 quality anchors and is the "
        "strongest general restoration baseline in this "
        "controlled benchmark."
    ),
    "stable_diffusion_inpainting": (
        "Stable Diffusion provides prompt-conditioned "
        "candidate diversity but leads none of the 11 "
        "aggregate quality anchors. Its outputs require "
        "case-level and repeated-seed inspection."
    ),
    "sdxl_inpainting": (
        "SDXL produced ten completed feasibility cases. "
        "The evidence is sufficient for bounded qualitative "
        "inspection but not for a full benchmark ranking."
    ),
}

MODEL_LIMITATION_MAP = {
    "opencv_telea": (
        "Its single crop-SSIM win must not be interpreted "
        "as universal restoration superiority."
    ),
    "lama": (
        "General benchmark leadership does not establish "
        "historical correctness or conservation approval."
    ),
    "stable_diffusion_inpainting": (
        "Seed variation is not calibrated confidence, and "
        "visual plausibility may conceal structural drift."
    ),
    "sdxl_inpainting": (
        "Ten cases do not represent the full 410-case "
        "restoration population."
    ),
}

MODEL_EVIDENCE_SUMMARY = (
    MODEL_CARDS[
        [
            "model_id",
            "display_name",
            "evaluation_status",
        ]
    ]
    .merge(
        APPROVED_MODEL_COUNTS,
        on="model_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        OVERALL_COMPUTE,
        on="model_id",
        how="left",
        validate="one_to_one",
    )
)

MODEL_EVIDENCE_SUMMARY[
    "quality_anchor_wins"
] = MODEL_EVIDENCE_SUMMARY[
    "model_id"
].map(QUALITY_ANCHOR_WINS)

MODEL_EVIDENCE_SUMMARY[
    "conclusion"
] = MODEL_EVIDENCE_SUMMARY[
    "model_id"
].map(MODEL_CONCLUSION_MAP)

MODEL_EVIDENCE_SUMMARY[
    "limitation"
] = MODEL_EVIDENCE_SUMMARY[
    "model_id"
].map(MODEL_LIMITATION_MAP)

MODEL_EVIDENCE_SUMMARY = (
    MODEL_EVIDENCE_SUMMARY
    .sort_values(
        "model_id",
        kind="stable",
    )
    .reset_index(drop=True)
)

OBSERVED_APPROVED_CANDIDATES = {
    str(row.model_id): int(
        row.approved_candidate_count
    )
    for row in MODEL_EVIDENCE_SUMMARY.itertuples()
}

EXPECTED_APPROVED_CANDIDATES = {
    "lama": 410,
    "opencv_telea": 410,
    "sdxl_inpainting": 10,
    "stable_diffusion_inpainting": 955,
}

FASTEST_MODEL_ID = str(
    MODEL_EVIDENCE_SUMMARY.loc[
        MODEL_EVIDENCE_SUMMARY[
            "median_runtime_seconds"
        ].idxmin(),
        "model_id",
    ]
)

MODEL_EVIDENCE_CHECKS = (
    (
        "model_count",
        "The evidence contains four declared model records",
        4,
        len(MODEL_EVIDENCE_SUMMARY),
        len(MODEL_EVIDENCE_SUMMARY) == 4,
    ),
    (
        "approved_candidate_counts",
        "Approved candidate counts remain distinct from executed supporting candidates",
        EXPECTED_APPROVED_CANDIDATES,
        OBSERVED_APPROVED_CANDIDATES,
        (
            OBSERVED_APPROVED_CANDIDATES
            == EXPECTED_APPROVED_CANDIDATES
        ),
    ),
    (
        "quality_anchor_total",
        "The three fully evaluated models account for all 11 anchor wins",
        11,
        int(
            MODEL_EVIDENCE_SUMMARY[
                "quality_anchor_wins"
            ].dropna().sum()
        ),
        (
            int(
                MODEL_EVIDENCE_SUMMARY[
                    "quality_anchor_wins"
                ].dropna().sum()
            )
            == 11
        ),
    ),
    (
        "lama_anchor_lead",
        "LaMa retains the validated 10-of-11 anchor lead",
        10,
        int(QUALITY_ANCHOR_WINS["lama"]),
        QUALITY_ANCHOR_WINS["lama"] == 10,
    ),
    (
        "telea_fastest",
        "OpenCV Telea is the fastest observed overall model",
        "opencv_telea",
        FASTEST_MODEL_ID,
        FASTEST_MODEL_ID == "opencv_telea",
    ),
    (
        "sdxl_partial_status",
        "SDXL remains partial evaluation only",
        "partial_evaluation",
        (
            MODEL_EVIDENCE_SUMMARY.loc[
                MODEL_EVIDENCE_SUMMARY[
                    "model_id"
                ].eq("sdxl_inpainting"),
                "evaluation_status",
            ].iloc[0]
        ),
        (
            MODEL_EVIDENCE_SUMMARY.loc[
                MODEL_EVIDENCE_SUMMARY[
                    "model_id"
                ].eq("sdxl_inpainting"),
                "evaluation_status",
            ].iloc[0]
            == "partial_evaluation"
        ),
    ),
    (
        "observed_failure_rates",
        "All four observed execution summaries report zero failed candidates",
        0.0,
        float(
            MODEL_EVIDENCE_SUMMARY[
                "failure_rate"
            ].max()
        ),
        (
            float(
                MODEL_EVIDENCE_SUMMARY[
                    "failure_rate"
                ].max()
            )
            == 0.0
        ),
    ),
    (
        "model_narrative_coverage",
        "Every model has one direct conclusion and limitation",
        4,
        int(
            (
                MODEL_EVIDENCE_SUMMARY[
                    "conclusion"
                ].notna()
                & MODEL_EVIDENCE_SUMMARY[
                    "limitation"
                ].notna()
            ).sum()
        ),
        (
            MODEL_EVIDENCE_SUMMARY[
                "conclusion"
            ].notna().all()
            and MODEL_EVIDENCE_SUMMARY[
                "limitation"
            ].notna().all()
        ),
    ),
)

for check_id, description, expected, observed, passed in MODEL_EVIDENCE_CHECKS:
    VALIDATION.add(
        validation_stage="batch_3_model_synthesis",
        check_id=check_id,
        check_description=description,
        severity="blocking",
        expected=expected,
        observed=observed,
        passed=bool(passed),
        details=(
            ""
            if passed
            else (
                "Model synthesis conflicts with "
                "the normalized upstream evidence."
            )
        ),
    )

VALIDATION.raise_for_blocking()

MODEL_DISPLAY_COLUMNS = [
    "display_name",
    "evaluation_status",
    "approved_case_count",
    "approved_candidate_count",
    "executed_candidate_count",
    "quality_anchor_wins",
    "median_runtime_seconds",
    "conclusion",
    "limitation",
]

display(
    MODEL_EVIDENCE_SUMMARY[
        MODEL_DISPLAY_COLUMNS
    ]
)

print("Model evidence synthesis passed.")
print("Approved candidates:", int(
    MODEL_EVIDENCE_SUMMARY[
        "approved_candidate_count"
    ].sum()
))
print("Fastest observed model:", FASTEST_MODEL_ID)
print("LaMa quality-anchor wins:", QUALITY_ANCHOR_WINS["lama"])
print(
    "Stable Diffusion executed candidates:",
    int(
        MODEL_EVIDENCE_SUMMARY.loc[
            MODEL_EVIDENCE_SUMMARY[
                "model_id"
            ].eq(
                "stable_diffusion_inpainting"
            ),
            "executed_candidate_count",
        ].iloc[0]
    ),
)

,display_name,evaluation_status,approved_case_count,approved_candidate_count,executed_candidate_count,quality_anchor_wins,median_runtime_seconds,conclusion,limitation
0,LaMa,fully_evaluated,410,410,410.0,10,1.567178,LaMa leads 10 of 11 quality anchors and is the...,General benchmark leadership does not establis...
1,OpenCV Telea,fully_evaluated,410,410,410.0,1,0.423579,OpenCV Telea is the fastest evaluated baseline...,Its single crop-SSIM win must not be interpret...
2,SDXL Inpainting,partial_evaluation,10,10,10.0,<NA>,294.916000,SDXL produced ten completed feasibility cases....,Ten cases do not represent the full 410-case r...
3,Stable Diffusion Inpainting,fully_evaluated,410,955,1330.0,0,9.535352,Stable Diffusion provides prompt-conditioned c...,"Seed variation is not calibrated confidence, a..."


Model evidence synthesis passed.
Approved candidates: 1785
Fastest observed model: opencv_telea
LaMa quality-anchor wins: 10
Stable Diffusion executed candidates: 1330


In [11]:
RQ_RECORDS = (
    RQ_COVERAGE.loc[
        RQ_COVERAGE[
            "research_question_id"
        ].isin(["rq1", "rq2", "rq3"])
    ]
    .sort_values(
        "display_order",
        kind="stable",
    )
    .reset_index(drop=True)
)

RQ_PLAIN_CONCLUSIONS = {
    "rq1": (
        "No single metric can establish restoration "
        "quality. Trustworthy comparison requires several "
        "metric families evaluated in the regions where "
        "they are meaningful, with disagreements retained."
    ),
    "rq2": (
        "LaMa is the strongest general baseline in this "
        "controlled benchmark, leading 10 of 11 quality "
        "anchors. Telea is fastest and leads crop SSIM, "
        "while diffusion results require closer case-level "
        "and variability review."
    ),
    "rq3": (
        "Repeated-seed variability can identify Stable "
        "Diffusion cases and regions that deserve closer "
        "visual review. It cannot determine whether a "
        "restoration is correct or historically plausible."
    ),
}

RQ_NUMERIC_EVIDENCE = {
    "rq1": (
        "11 separate quality anchors spanning "
        "complementary metric families and regions"
    ),
    "rq2": (
        "LaMa leads 10 of 11 anchors; Telea leads "
        "crop SSIM and has the lowest observed runtime"
    ),
    "rq3": (
        "165 four-seed uncertainty groups: "
        "130 canonical and 35 damage-size groups"
    ),
}

RQ_SYNTHESIS_RECORDS = []
RQ_SOURCE_PATH_RECORDS = []

for row in RQ_RECORDS.itertuples(index=False):
    question_id = str(row.research_question_id)

    source_notebook_ids = json.loads(
        row.source_notebook_ids_json
    )

    source_paths = json.loads(
        row.source_paths_json
    )

    for source_path in source_paths:
        resolved_source = (
            PROJECT_ROOT / source_path
        ).resolve()

        RQ_SOURCE_PATH_RECORDS.append(
            {
                "research_question_id": question_id,
                "relative_path": source_path,
                "path_exists": (
                    resolved_source.exists()
                ),
                "path_is_inside_repository": (
                    resolved_source.is_relative_to(
                        PROJECT_ROOT
                    )
                ),
            }
        )

    RQ_SYNTHESIS_RECORDS.append(
        {
            "research_question_id": question_id,
            "research_question": (
                RESEARCH_QUESTIONS[question_id]
            ),
            "coverage_status": row.coverage_status,
            "evidence_answer": (
                row.supported_interpretation
            ),
            "plain_conclusion": (
                RQ_PLAIN_CONCLUSIONS[question_id]
            ),
            "numeric_evidence": (
                RQ_NUMERIC_EVIDENCE[question_id]
            ),
            "prohibited_interpretation": (
                row.prohibited_interpretation
            ),
            "source_notebook_ids": (
                source_notebook_ids
            ),
            "source_paths": source_paths,
        }
    )

RQ_SYNTHESIS = pd.DataFrame(
    RQ_SYNTHESIS_RECORDS
)

RQ_SOURCE_PATH_AUDIT = pd.DataFrame(
    RQ_SOURCE_PATH_RECORDS
)

KEY_FINDING_IMPLICATIONS = {
    "Controlled painting collection": (
        "The balanced collection supports controlled "
        "within-benchmark comparisons, but conclusions "
        "must remain limited to this collection."
    ),
    "Approved restoration evidence": (
        "The complete candidate catalog remains available "
        "for inspection; the report does not rely only on "
        "its selected examples."
    ),
    "Strongest general benchmark result": (
        "LaMa is the best default baseline among the three "
        "fully evaluated methods when several forms of "
        "evidence are considered together."
    ),
    "One metric changes the winner": (
        "A conclusion based only on SSIM would favour "
        "Telea and conceal LaMa's broader advantage. "
        "Metric choice materially changes the verdict."
    ),
    "Repeated-run uncertainty coverage": (
        "Stable Diffusion outputs that change more across "
        "seeds should receive closer visual inspection."
    ),
    "SDXL remains bounded evidence": (
        "SDXL can be discussed as a feasibility result, "
        "but it cannot be ranked as a fully evaluated "
        "fourth model."
    ),
    "Human review remains central": (
        "The conservative flag rate makes automated "
        "restoration unsuitable for unsupervised acceptance."
    ),
    "No universal restoration score": (
        "Keeping evidence separate prevents a convenient "
        "single number from hiding metric and regional "
        "disagreement."
    ),
}

KEY_FINDING_KEYS = {
    "Controlled painting collection": "controlled_collection",
    "Approved restoration evidence": "approved_candidates",
    "Strongest general benchmark result": "lama_anchor_lead",
    "One metric changes the winner": "metric_disagreement",
    "Repeated-run uncertainty coverage": "uncertainty_coverage",
    "SDXL remains bounded evidence": "sdxl_boundary",
    "Human review remains central": "human_review",
    "No universal restoration score": "no_combined_score",
}

KEY_FINDINGS = HEADLINE_FINDINGS[
    [
        "finding_id",
        "display_order",
        "title",
        "value",
        "value_unit",
        "conclusion",
        "evidence_strength",
        "scope",
        "denominator",
        "source_notebook_ids_json",
        "source_paths_json",
        "limitation",
    ]
].copy()

KEY_FINDINGS["finding_key"] = (
    KEY_FINDINGS["title"].map(
        KEY_FINDING_KEYS
    )
)

KEY_FINDINGS["plain_implication"] = (
    KEY_FINDINGS["title"].map(
        KEY_FINDING_IMPLICATIONS
    )
)

KEY_FINDINGS = (
    KEY_FINDINGS[
        [
            "finding_key",
            "finding_id",
            "display_order",
            "title",
            "value",
            "value_unit",
            "conclusion",
            "plain_implication",
            "evidence_strength",
            "scope",
            "denominator",
            "source_notebook_ids_json",
            "source_paths_json",
            "limitation",
        ]
    ]
    .sort_values(
        "display_order",
        kind="stable",
    )
    .reset_index(drop=True)
)

RQ_CHECKS = (
    (
        "proposal_question_count",
        "Exactly three proposal research questions are synthesized",
        3,
        len(RQ_SYNTHESIS),
        len(RQ_SYNTHESIS) == 3,
    ),
    (
        "proposal_question_ids",
        "Research-question IDs retain the approved order",
        ["rq1", "rq2", "rq3"],
        RQ_SYNTHESIS[
            "research_question_id"
        ].tolist(),
        (
            RQ_SYNTHESIS[
                "research_question_id"
            ].tolist()
            == ["rq1", "rq2", "rq3"]
        ),
    ),
    (
        "proposal_question_text",
        "The proposal questions are reproduced exactly from the contract",
        [
            RESEARCH_QUESTIONS["rq1"],
            RESEARCH_QUESTIONS["rq2"],
            RESEARCH_QUESTIONS["rq3"],
        ],
        RQ_SYNTHESIS[
            "research_question"
        ].tolist(),
        (
            RQ_SYNTHESIS[
                "research_question"
            ].tolist()
            == [
                RESEARCH_QUESTIONS["rq1"],
                RESEARCH_QUESTIONS["rq2"],
                RESEARCH_QUESTIONS["rq3"],
            ]
        ),
    ),
    (
        "research_question_coverage",
        "All three questions are addressed by completed evidence",
        ["addressed_by_completed_evidence"],
        sorted(
            RQ_SYNTHESIS[
                "coverage_status"
            ].unique().tolist()
        ),
        RQ_SYNTHESIS[
            "coverage_status"
        ].eq(
            "addressed_by_completed_evidence"
        ).all(),
    ),
    (
        "research_question_sources",
        "Every research-question source path exists inside the repository",
        0,
        int(
            (
                ~RQ_SOURCE_PATH_AUDIT[
                    "path_exists"
                ]
                | ~RQ_SOURCE_PATH_AUDIT[
                    "path_is_inside_repository"
                ]
            ).sum()
        ),
        (
            RQ_SOURCE_PATH_AUDIT[
                "path_exists"
            ].all()
            and RQ_SOURCE_PATH_AUDIT[
                "path_is_inside_repository"
            ].all()
        ),
    ),
    (
        "key_finding_count",
        "All eight approved findings are retained",
        8,
        len(KEY_FINDINGS),
        len(KEY_FINDINGS) == 8,
    ),
    (
        "key_finding_keys",
        "Every finding has one stable package key",
        8,
        int(
            KEY_FINDINGS[
                "finding_key"
            ].notna().sum()
        ),
        (
            KEY_FINDINGS[
                "finding_key"
            ].notna().all()
            and KEY_FINDINGS[
                "finding_key"
            ].is_unique
        ),
    ),
    (
        "key_finding_implications",
        "Every fact is followed by a direct restoration implication",
        8,
        int(
            KEY_FINDINGS[
                "plain_implication"
            ].notna().sum()
        ),
        KEY_FINDINGS[
            "plain_implication"
        ].notna().all(),
    ),
)

for check_id, description, expected, observed, passed in RQ_CHECKS:
    VALIDATION.add(
        validation_stage="batch_3_research_questions",
        check_id=check_id,
        check_description=description,
        severity="blocking",
        expected=expected,
        observed=observed,
        passed=bool(passed),
        details=(
            ""
            if passed
            else (
                "Research-question synthesis or "
                "key-finding traceability failed."
            )
        ),
    )

VALIDATION.raise_for_blocking()

display(
    RQ_SYNTHESIS[
        [
            "research_question_id",
            "research_question",
            "evidence_answer",
            "plain_conclusion",
            "numeric_evidence",
            "prohibited_interpretation",
        ]
    ]
)

display(
    KEY_FINDINGS[
        [
            "finding_key",
            "title",
            "value",
            "value_unit",
            "plain_implication",
            "limitation",
        ]
    ]
)

print("Research-question synthesis passed.")
print("Proposal research questions:", len(RQ_SYNTHESIS))
print("Validated key findings:", len(KEY_FINDINGS))
print(
    "Research-question source paths:",
    len(RQ_SOURCE_PATH_AUDIT),
)
print(
    "Missing research-question sources:",
    int(
        (~RQ_SOURCE_PATH_AUDIT["path_exists"]).sum()
    ),
)

,research_question_id,research_question,evidence_answer,plain_conclusion,numeric_evidence,prohibited_interpretation
0,rq1,How can a multi-metric evaluation framework be...,Restoration quality requires complementary pix...,No single metric can establish restoration qua...,11 separate quality anchors spanning complemen...,No individual metric or combined universal sco...
1,rq2,How do selected pretrained inpainting models d...,LaMa is the strongest general baseline in this...,LaMa is the strongest general baseline in this...,LaMa leads 10 of 11 anchors; Telea leads crop ...,The result does not establish a universally be...
2,rq3,To what extent can uncertainty estimation from...,Repeated-seed variability identifies diffusion...,Repeated-seed variability can identify Stable ...,165 four-seed uncertainty groups: 130 canonica...,Empirical variability is not calibrated confid...


,finding_key,title,value,value_unit,plain_implication,limitation
0,controlled_collection,Controlled painting collection,50.0,paintings,The balanced collection supports controlled wi...,The controlled collection does not establish r...
1,approved_candidates,Approved restoration evidence,1785.0,candidates,The complete candidate catalog remains availab...,Candidate count is not an independent sample-s...
2,lama_anchor_lead,Strongest general benchmark result,10.0,of 11 quality anchors,LaMa is the best default baseline among the th...,Anchor wins are metric-specific benchmark resu...
3,metric_disagreement,One metric changes the winner,1.0,of 11 quality anchors,A conclusion based only on SSIM would favour T...,The result demonstrates metric disagreement; i...
4,uncertainty_coverage,Repeated-run uncertainty coverage,165.0,uncertainty groups,Stable Diffusion outputs that change more acro...,Seed variability is an empirical uncertainty p...
5,sdxl_boundary,SDXL remains bounded evidence,10.0,candidates,"SDXL can be discussed as a feasibility result,...",SDXL results must not be generalized to all 41...
6,human_review,Human review remains central,95.4,% of candidates,The conservative flag rate makes automated res...,Computational review flags are diagnostic rule...
7,no_combined_score,No universal restoration score,0.0,approved combined scores,Keeping evidence separate prevents a convenien...,The dashboard supports evidence review and doe...


Research-question synthesis passed.
Proposal research questions: 3
Validated key findings: 8
Research-question source paths: 18
Missing research-question sources: 0


In [12]:
CANONICAL_FILES_AFTER_BATCH_3 = sorted(
    path.relative_to(OUTPUT_ROOT).as_posix()
    for path in OUTPUT_ROOT.rglob("*")
    if path.is_file()
)

VALIDATION.add(
    validation_stage="batch_3_final_synthesis",
    check_id="no_canonical_files_written",
    check_description=(
        "Batch 3 synthesized evidence in memory only"
    ),
    severity="blocking",
    expected=[],
    observed=CANONICAL_FILES_AFTER_BATCH_3,
    passed=not CANONICAL_FILES_AFTER_BATCH_3,
    details=(
        ""
        if not CANONICAL_FILES_AFTER_BATCH_3
        else (
            "Evidence synthesis must not persist "
            "Notebook 36 outputs before the report stage."
        )
    ),
)

VALIDATION.raise_for_blocking()

BATCH_3_VALIDATION = VALIDATION.to_dataframe()

BATCH_3_STAGE_SUMMARY = (
    BATCH_3_VALIDATION
    .groupby(
        ["validation_stage", "severity"],
        dropna=False,
    )
    .agg(
        checks=("check_id", "size"),
        passed=("passed", "sum"),
    )
    .reset_index()
)

BATCH_3_STAGE_SUMMARY["failed"] = (
    BATCH_3_STAGE_SUMMARY["checks"]
    - BATCH_3_STAGE_SUMMARY["passed"]
)

display(BATCH_3_STAGE_SUMMARY)

print("Batch 3 evidence synthesis passed.")
print("Cumulative validation checks:", len(BATCH_3_VALIDATION))
print("Blocking failures:", len(VALIDATION.blocking_failures))
print("Validated evidence-population fields:", len(
    OBSERVED_EVIDENCE_POPULATION
))
print("Model conclusions:", len(MODEL_EVIDENCE_SUMMARY))
print("Research-question answers:", len(RQ_SYNTHESIS))
print("Key findings:", len(KEY_FINDINGS))
print("Canonical files persisted through Batch 3: 0")

,validation_stage,severity,checks,passed,failed
0,batch_1_contract,blocking,11,11,0
1,batch_1_dependencies_sources,blocking,9,9,0
2,batch_1_final_preflight,blocking,1,1,0
3,batch_1_inventory_governance,blocking,15,15,0
4,batch_2_artifact_registry,blocking,10,10,0
5,batch_2_final_audit,blocking,1,1,0
6,batch_2_upstream_manifests,blocking,14,14,0
7,batch_3_evidence_inputs,blocking,9,9,0
8,batch_3_final_synthesis,blocking,1,1,0
9,batch_3_model_synthesis,blocking,8,8,0


Batch 3 evidence synthesis passed.
Cumulative validation checks: 87
Blocking failures: 0
Validated evidence-population fields: 16
Model conclusions: 4
Research-question answers: 3
Key findings: 8
Canonical files persisted through Batch 3: 0


## Batch 4 — Supervisor-facing reports and meeting documents

This batch creates the concise human-facing entry points for the final package:

- `reports/supervisor_summary.md`;
- `data/key_findings.json`;
- `data/open_questions.md`;
- `data/feedback_agenda.md`.

The supervisor summary follows the approved structure:

- decision snapshot;
- evaluated scope;
- exact research questions and evidence-bounded answers;
- direct model conclusions;
- robustness and uncertainty;
- trustworthiness and explainability;
- reproducibility and dashboard status;
- limitations;
- decisions requested from the supervisor.

The report uses actual validated numbers, emphasizes important findings, and follows factual statements with short conclusions about restoration quality or model suitability. Its six figure links point to the package paths that will be populated in Batch 6.

In [13]:
from restoration_eval.supervisor_package import (
    atomic_write_json,
    atomic_write_text,
)


REPORT_GENERATED_AT_UTC = (
    datetime.now(timezone.utc)
    .isoformat(timespec="seconds")
    .replace("+00:00", "Z")
)

MODEL_BY_ID = (
    MODEL_EVIDENCE_SUMMARY
    .set_index("model_id")
)

KEY_FINDING_RECORDS = []

for row in KEY_FINDINGS.itertuples(index=False):
    KEY_FINDING_RECORDS.append(
        {
            "finding_key": str(row.finding_key),
            "finding_id": str(row.finding_id),
            "display_order": int(row.display_order),
            "title": str(row.title),
            "value": float(row.value),
            "value_unit": str(row.value_unit),
            "conclusion": str(row.conclusion),
            "plain_implication": str(
                row.plain_implication
            ),
            "evidence_strength": str(
                row.evidence_strength
            ),
            "scope": str(row.scope),
            "denominator": str(row.denominator),
            "source_notebook_ids": json.loads(
                row.source_notebook_ids_json
            ),
            "source_paths": json.loads(
                row.source_paths_json
            ),
            "limitation": str(row.limitation),
        }
    )

KEY_FINDINGS_PAYLOAD = {
    "schema_version": "supervisor_key_findings.v1",
    "notebook_id": NOTEBOOK_ID,
    "notebook_stem": NOTEBOOK_STEM,
    "generated_at_utc": REPORT_GENERATED_AT_UTC,
    "creates_new_scientific_evidence": False,
    "dataset_scope": "controlled_50",
    "finding_count": len(KEY_FINDING_RECORDS),
    "findings": KEY_FINDING_RECORDS,
}

RQ_SECTIONS = []

for row in RQ_SYNTHESIS.itertuples(index=False):
    rq_number = str(
        row.research_question_id
    ).replace("rq", "RQ")

    RQ_SECTIONS.append(
        "\n".join(
            [
                f"### {rq_number}",
                "",
                f"> {row.research_question}",
                "",
                (
                    f"**Evidence:** "
                    f"{row.evidence_answer}"
                ),
                "",
                (
                    f"**Key evidence:** "
                    f"{row.numeric_evidence}."
                ),
                "",
                (
                    f"**Conclusion:** "
                    f"{row.plain_conclusion}"
                ),
                "",
                (
                    f"**Boundary:** "
                    f"{row.prohibited_interpretation}"
                ),
            ]
        )
    )

RQ_REPORT_TEXT = "\n\n".join(RQ_SECTIONS)

MODEL_TABLE_ROWS = []

for model_id in (
    "lama",
    "opencv_telea",
    "stable_diffusion_inpainting",
    "sdxl_inpainting",
):
    row = MODEL_BY_ID.loc[model_id]

    anchor_wins = (
        "Not applicable"
        if pd.isna(row["quality_anchor_wins"])
        else (
            f"{int(row['quality_anchor_wins'])} of 11"
        )
    )

    MODEL_TABLE_ROWS.append(
        "| {model} | {status} | {approved:,} | {executed:,} | "
        "{wins} | {runtime:.3f} s | {conclusion} |".format(
            model=str(row["display_name"]),
            status=str(row["evaluation_status"]).replace(
                "_",
                " ",
            ),
            approved=int(
                row["approved_candidate_count"]
            ),
            executed=int(
                row["executed_candidate_count"]
            ),
            wins=anchor_wins,
            runtime=float(
                row["median_runtime_seconds"]
            ),
            conclusion=str(
                row["conclusion"]
            ).replace("|", "\\|"),
        )
    )

MODEL_TABLE_TEXT = "\n".join(
    [
        (
            "| Model | Evaluation status | Approved candidates | "
            "Executed candidates | Quality-anchor wins | "
            "Median runtime | Direct conclusion |"
        ),
        "|---|---:|---:|---:|---:|---:|---|",
        *MODEL_TABLE_ROWS,
    ]
)

SUPERVISOR_SUMMARY_TEXT = f"""# Trustworthy Evaluation of AI-Assisted Painting Restoration

## Supervisor Review Summary

**Notebook:** `36_supervisor_publication_reproducibility_package.ipynb`  
**Evidence scope:** Controlled 50-painting benchmark and declared extensions  
**Generated:** `{REPORT_GENERATED_AT_UTC}`  
**Package status:** Assembly in progress; scientific evidence complete through Notebook 35  
**Scientific role:** Evidence synthesis and delivery only—no new metrics or restoration inference

## 1. Decision snapshot

- **Strongest general baseline:** **LaMa**, leading **10 of 11 quality anchors**.
- **Fastest evaluated baseline:** **OpenCV Telea**, with a median observed runtime of **{float(MODEL_BY_ID.loc["opencv_telea", "median_runtime_seconds"]):.3f} seconds** per recorded candidate.
- **Metric disagreement matters:** Telea leads **crop SSIM**, while LaMa leads the other **10 anchors**. SSIM alone would therefore give an incomplete result.
- **Diffusion requires closer review:** Stable Diffusion leads **0 of 11 aggregate anchors** and has **165 repeated-seed uncertainty groups** available for variability analysis.
- **SDXL remains limited:** only **10 feasibility candidates** were evaluated, so it is not ranked as a fourth full benchmark.
- **Human inspection remains necessary:** **1,703 of 1,785 candidates (95.4%)** trigger conservative computational review guidance.
- **No universal score is used:** metric families, regions, uncertainty, and failure evidence remain separate.

### Central conclusion

**Visual plausibility is not the same as restoration trustworthiness.**

LaMa is the best general benchmark baseline, but no model result should be accepted from appearance or one metric alone. Restoration evidence must remain traceable across regions, metric families, uncertainty, diagnostic maps, and case-level inspection.

![Benchmark summary](../package/figures/publication/01_benchmark_summary.png)

## 2. What was evaluated

| Evidence component | Validated scope |
|---|---:|
| Paintings | **50** |
| Visual categories | **5**, with 10 paintings each |
| Registered experimental cases | **525** |
| Restoration cases | **410** |
| Approved comparison candidates | **1,785** |
| Fully evaluated models | **3** |
| Bounded SDXL feasibility model | **1** |
| Quality anchors | **11** |
| Repeated-seed uncertainty groups | **165** |
| Indexed visual records | **23,964** |
| Indexed reports | **104** |
| Thesis figures | **18** |
| Publication figures | **6** |

The evidence is broad enough for controlled comparison across the declared paintings, models, metrics, regions, and synthetic damage conditions. It does **not** establish performance on real conservation treatments or unseen museum collections.

## 3. Research questions

{RQ_REPORT_TEXT}

![Stress-test summary](../package/figures/publication/02_stress_test_summary.png)

## 4. Model conclusions

{MODEL_TABLE_TEXT}

### What this means

- **LaMa:** the strongest default baseline when overall restoration evidence matters more than one isolated metric.
- **OpenCV Telea:** the practical speed baseline and competitive for thin or local filling, but weaker as a general multi-metric solution.
- **Stable Diffusion:** useful for studying prompt-conditioned and repeated-seed behaviour, but its candidates require more intensive visual and uncertainty review.
- **SDXL:** useful only as bounded feasibility evidence in the current project.

The Stable Diffusion executed total includes additional prompt and repeated-seed candidates. The **955 approved Stable Diffusion candidates** are the population retained in the 1,785-candidate comparison catalog.

## 5. Robustness and uncertainty

The project separates three different questions:

- **Damage-size sensitivity:** how results change as the damaged area increases.
- **Mask robustness:** how results change when mask geometry or placement changes.
- **Generative uncertainty:** how Stable Diffusion candidates change across repeated seeds for a fixed case and prompt configuration.

These quantities are not interchangeable. Mask variation is input robustness, and prompt variation is prompt sensitivity; neither is relabelled as generative uncertainty.

The **165 uncertainty groups** comprise:

- **130 canonical Stable Diffusion groups**;
- **35 damage-size Stable Diffusion groups**;
- four seeds per eligible group.

**Conclusion:** greater repeated-seed variation identifies cases or regions that deserve closer visual review. It does not prove that a low-variation restoration is correct.

![Uncertainty and spatial summary](../package/figures/publication/03_uncertainty_spatial_summary.png)

## 6. Trustworthiness and explainability

The framework keeps review evidence separate:

- reference and perceptual evidence;
- feature similarity;
- texture and brushstroke proxies;
- colour consistency;
- seam and boundary consistency;
- semantic and structural affinity;
- difference and uncertainty maps;
- counterfactual, example-based, and rule-based explanations;
- independent computational failure flags.

**1,703 of 1,785 candidates** receive conservative review guidance. This high rate does not mean that 95.4% are objectively failed restorations. It means the rules are intentionally cautious and that automatic acceptance would be inappropriate.

**Conclusion:** computational explanations can show why a candidate was flagged and where disagreement occurs, but an expert must decide whether that evidence is meaningful for conservation.

![Trustworthiness and ablation summary](../package/figures/publication/04_trustworthiness_ablation_summary.png)

![Explainability summary](../package/figures/publication/05_explainability_summary.png)

## 7. Reproducibility and delivery status

- **35 of 35** upstream notebook completion gates passed.
- **417** manifest-declared outputs currently exist.
- **218** canonical artifacts are registered.
- The final package copy plan contains **106 files** and approximately **{COPY_PLAN_SIZE_MIB:.2f} MiB** of source material.
- All five bundled HTML reports are designed to remain self-contained.
- All eight Streamlit pages passed the Notebook 35 runtime smoke test.
- The dashboard is **ready for local supervisor demonstration**.
- Public deployment is **not complete**.
- Notebook 35 retains non-blocking dependency-version warnings that must remain visible.

![Quality and compute summary](../package/figures/publication/06_quality_compute_summary.png)

## 8. Limitations that must remain visible

- The dataset contains controlled synthetic damage rather than verified physical conservation interventions.
- The 50-painting collection does not establish universal artistic or museum-domain generality.
- Repeated observations, models, and seeds sharing the same painting are not independent paintings.
- Feature similarity is not historical authenticity.
- Seed variability is not calibrated confidence.
- Computational flags are not expert annotations.
- Retrieval neighbours provide context rather than proof.
- SDXL has only ten completed cases.
- No model output constitutes a treatment recommendation or conservation approval.

## 9. Decisions requested from the supervisor

1. Is the controlled 50-painting scope sufficient for the thesis claims as currently bounded?
2. Is the multi-metric, region-aware framework an appropriate central methodological contribution?
3. Is LaMa's 10-of-11 anchor lead sufficient to describe it as the strongest general benchmark baseline?
4. Is it acceptable to retain SDXL strictly as bounded feasibility evidence?
5. Should a human or conservator review study be framed as future work rather than added to the current empirical scope?
6. Is local dashboard demonstration sufficient, or is public deployment required before submission?
7. Which figures and findings should be prioritized in the thesis defence and any publication draft?

## 10. Recommended review route

1. Read this summary.
2. Open `reports/final_evaluation.html` for the complete thesis-level narrative.
3. Open the four reports under `reports/models/` for model-specific evidence.
4. Use `tables/` for exact compact evidence and report indexes.
5. Use the Streamlit application for filtered case, image, map, and report inspection.
6. Use `provenance/reproducibility_snapshot.json` and `manifests/notebook_runs/` for audit and reproduction.

---

**Interpretation boundary:** This package supports evidence review and thesis communication. It does not authenticate paintings, approve conservation treatment, or replace expert judgement.
"""

OPEN_QUESTIONS_TEXT = f"""# Open Questions for Supervisor Review

**Generated:** `{REPORT_GENERATED_AT_UTC}`  
**Scope:** Decisions about interpretation, delivery, and future work—not unresolved computational failures.

### Decision 1 — Thesis scope

Is the balanced **50-painting controlled collection** sufficient for the bounded claims, or should broader real-world generalization remain an explicit limitation and future-work requirement?

### Decision 2 — Central contribution

Should the thesis foreground the **multi-metric, region-aware evaluation framework** as its main contribution, with the model ranking treated as an empirical demonstration?

### Decision 3 — Model conclusion

Is the wording **“LaMa is the strongest general baseline in this controlled benchmark”** acceptable given its **10-of-11 quality-anchor lead** and the retained metric disagreement?

### Decision 4 — SDXL boundary

Should SDXL remain a **ten-case feasibility study**, or is a future rerun on stronger hardware required outside the current completed benchmark?

### Decision 5 — Human review

Should expert or conservator assessment remain a clearly stated future validation stage rather than being added after completion of the computational pipeline?

### Decision 6 — Delivery

Is the validated local Streamlit demonstration sufficient for supervision and defence, or is a public deployment required?

### Decision 7 — Publication focus

Which combination should receive priority in a paper or defence:

- the evaluation architecture;
- conditional model comparison;
- robustness and uncertainty;
- trustworthiness flags and XAI;
- the reproducible dashboard and report package?

## Current recommendation

Freeze the completed scientific scope, retain the limitations explicitly, and use supervisor feedback to prioritize writing and presentation rather than silently expanding the experiment.
"""

FEEDBACK_AGENDA_TEXT = f"""# Supervisor Feedback Agenda

**Generated:** `{REPORT_GENERATED_AT_UTC}`  
**Suggested duration:** 45–60 minutes

## 1. Five-minute orientation

- Central thesis: visual plausibility is not restoration trustworthiness.
- Validated scope: **50 paintings**, **410 restoration cases**, and **1,785 approved candidates**.
- Main result: **LaMa leads 10 of 11 quality anchors**.
- Delivery status: dashboard ready for local demonstration; final package assembly in progress.

## 2. Evaluation framework — 10 minutes

Discuss:

- complementary metric families;
- canonical region policy;
- why metric disagreement is retained;
- why no universal combined score is reported.

**Decision requested:** approve the framework as the thesis's central methodological contribution.

## 3. Model evidence — 10 minutes

Discuss:

- LaMa as the strongest general baseline;
- Telea as the fastest deterministic baseline;
- Stable Diffusion variability and case-level review;
- SDXL as bounded feasibility evidence.

**Decision requested:** approve the comparative wording and SDXL boundary.

## 4. Robustness, uncertainty, and trustworthiness — 10 minutes

Discuss:

- damage-size sensitivity;
- mask robustness;
- synthetic-degradation sensitivity;
- 165 repeated-seed uncertainty groups;
- conservative review flags;
- explainability and diagnostic maps.

**Decision requested:** confirm that uncertainty remains an inspection signal rather than calibrated confidence.

## 5. Thesis and publication assets — 10 minutes

Review:

- final self-contained report;
- four model reports;
- 18 thesis figures;
- six publication figures;
- model cards and compact tables;
- Streamlit case exploration.

**Decision requested:** identify the most important figures and findings for the defence and publication draft.

## 6. Final actions — 5 minutes

- Confirm whether public dashboard deployment is required.
- Confirm whether human/expert review remains future work.
- Confirm that the empirical scope should now remain frozen.
- Record wording changes requested for the thesis conclusions.
"""

print("Supervisor-facing content constructed in memory.")
print("Supervisor-summary characters:", len(SUPERVISOR_SUMMARY_TEXT))
print("Supervisor-summary figure links:", SUPERVISOR_SUMMARY_TEXT.count("!["))
print("Key-finding records:", len(KEY_FINDING_RECORDS))
print("Open-question decisions:", OPEN_QUESTIONS_TEXT.count("### Decision"))
print("Agenda sections:", FEEDBACK_AGENDA_TEXT.count("## "))

SUPERVISOR_SUMMARY_TEXT = SUPERVISOR_SUMMARY_TEXT.replace(
    "**Package status:** Assembly in progress; "
    "scientific evidence complete through Notebook 35",
    "**Package validation:** See the final Notebook 36 "
    "run manifest and validation ledger; "
    "scientific evidence is complete through Notebook 35",
)

SUPERVISOR_SUMMARY_TEXT = SUPERVISOR_SUMMARY_TEXT.replace(
    "- **218** canonical artifacts are registered.",
    "- **218** upstream canonical artifacts were registered "
    "at Notebook 36 preflight; Notebook 36 delivery records "
    "are registered after final validation.",
)

FEEDBACK_AGENDA_TEXT = FEEDBACK_AGENDA_TEXT.replace(
    "- Delivery status: dashboard ready for local demonstration; "
    "final package assembly in progress.",
    "- Delivery status: dashboard ready for local demonstration; "
    "package completion is recorded in the final Notebook 36 "
    "run manifest and validation ledger.",
)

print("Delivery wording corrected; registry count labelled as a snapshot.")

Supervisor-facing content constructed in memory.
Supervisor-summary characters: 11770
Supervisor-summary figure links: 6
Key-finding records: 8
Open-question decisions: 7
Agenda sections: 6
Delivery wording corrected; registry count labelled as a snapshot.


In [14]:
SUPERVISOR_SUMMARY_PATH = OUTPUT_PATHS[
    "supervisor_summary"
]

KEY_FINDINGS_PATH = OUTPUT_PATHS[
    "key_findings"
]

OPEN_QUESTIONS_PATH = OUTPUT_PATHS[
    "open_questions"
]

FEEDBACK_AGENDA_PATH = OUTPUT_PATHS[
    "feedback_agenda"
]

atomic_write_text(
    SUPERVISOR_SUMMARY_PATH,
    SUPERVISOR_SUMMARY_TEXT,
)

atomic_write_json(
    KEY_FINDINGS_PATH,
    KEY_FINDINGS_PAYLOAD,
)

atomic_write_text(
    OPEN_QUESTIONS_PATH,
    OPEN_QUESTIONS_TEXT,
)

atomic_write_text(
    FEEDBACK_AGENDA_PATH,
    FEEDBACK_AGENDA_TEXT,
)

BATCH_4_WRITTEN_PATHS = [
    SUPERVISOR_SUMMARY_PATH,
    KEY_FINDINGS_PATH,
    OPEN_QUESTIONS_PATH,
    FEEDBACK_AGENDA_PATH,
]

BATCH_4_WRITE_VIEW = pd.DataFrame(
    [
        {
            "artifact": path.stem,
            "relative_path": (
                path.relative_to(
                    PROJECT_ROOT
                ).as_posix()
            ),
            "format": (
                path.suffix.lower().lstrip(".")
            ),
            "size_bytes": path.stat().st_size,
        }
        for path in BATCH_4_WRITTEN_PATHS
    ]
)

display(BATCH_4_WRITE_VIEW)

print("Supervisor-facing outputs persisted.")
print("Files written:", len(BATCH_4_WRITTEN_PATHS))
print(
    "Total bytes:",
    int(BATCH_4_WRITE_VIEW["size_bytes"].sum()),
)

,artifact,relative_path,format,size_bytes
0,supervisor_summary,outputs/36_supervisor_publication_reproducibil...,md,11922
1,key_findings,outputs/36_supervisor_publication_reproducibil...,json,8729
2,open_questions,outputs/36_supervisor_publication_reproducibil...,md,1877
3,feedback_agenda,outputs/36_supervisor_publication_reproducibil...,md,2130


Supervisor-facing outputs persisted.
Files written: 4
Total bytes: 24658


In [15]:
RELOADED_SUPERVISOR_SUMMARY = (
    SUPERVISOR_SUMMARY_PATH.read_text(
        encoding="utf-8",
    )
)

with KEY_FINDINGS_PATH.open(
    "r",
    encoding="utf-8-sig",
) as handle:
    RELOADED_KEY_FINDINGS = json.load(handle)

RELOADED_OPEN_QUESTIONS = (
    OPEN_QUESTIONS_PATH.read_text(
        encoding="utf-8",
    )
)

RELOADED_FEEDBACK_AGENDA = (
    FEEDBACK_AGENDA_PATH.read_text(
        encoding="utf-8",
    )
)

EXPECTED_BATCH_4_FILES = sorted(
    path.relative_to(OUTPUT_ROOT).as_posix()
    for path in BATCH_4_WRITTEN_PATHS
)

OBSERVED_BATCH_4_FILES = sorted(
    path.relative_to(OUTPUT_ROOT).as_posix()
    for path in OUTPUT_ROOT.rglob("*")
    if path.is_file()
)

REQUIRED_SUMMARY_HEADINGS = [
    "## 1. Decision snapshot",
    "## 2. What was evaluated",
    "## 3. Research questions",
    "## 4. Model conclusions",
    "## 5. Robustness and uncertainty",
    "## 6. Trustworthiness and explainability",
    "## 7. Reproducibility and delivery status",
    "## 8. Limitations that must remain visible",
    "## 9. Decisions requested from the supervisor",
    "## 10. Recommended review route",
]

MISSING_SUMMARY_HEADINGS = [
    heading
    for heading in REQUIRED_SUMMARY_HEADINGS
    if heading not in RELOADED_SUPERVISOR_SUMMARY
]

MISSING_RESEARCH_QUESTIONS = [
    question
    for question in RESEARCH_QUESTIONS.values()
    if question not in RELOADED_SUPERVISOR_SUMMARY
]

REQUIRED_NUMERIC_STATEMENTS = [
    "10 of 11 quality anchors",
    "1,703 of 1,785 candidates",
    "165 uncertainty groups",
    "35 of 35",
    "23,964",
    "104",
]

MISSING_NUMERIC_STATEMENTS = [
    statement
    for statement in REQUIRED_NUMERIC_STATEMENTS
    if statement not in RELOADED_SUPERVISOR_SUMMARY
]

OBSOLETE_REPORT_PATHS = [
    "outputs/supervisor_package",
    "outputs/dashboard/",
    "outputs/reports/",
    "outputs/figures/",
]

FOUND_OBSOLETE_REPORT_PATHS = [
    value
    for value in OBSOLETE_REPORT_PATHS
    if value in RELOADED_SUPERVISOR_SUMMARY
]

BATCH_4_CHECKS = (
    (
        "canonical_files",
        "Exactly the four approved Batch 4 files exist",
        EXPECTED_BATCH_4_FILES,
        OBSERVED_BATCH_4_FILES,
        (
            OBSERVED_BATCH_4_FILES
            == EXPECTED_BATCH_4_FILES
        ),
    ),
    (
        "supervisor_summary_reload",
        "The supervisor summary reloads without content loss",
        SUPERVISOR_SUMMARY_TEXT,
        RELOADED_SUPERVISOR_SUMMARY,
        (
            RELOADED_SUPERVISOR_SUMMARY
            == SUPERVISOR_SUMMARY_TEXT
        ),
    ),
    (
        "supervisor_summary_structure",
        "The supervisor summary retains every approved section",
        [],
        MISSING_SUMMARY_HEADINGS,
        not MISSING_SUMMARY_HEADINGS,
    ),
    (
        "research_question_wording",
        "All three proposal questions appear exactly",
        [],
        MISSING_RESEARCH_QUESTIONS,
        not MISSING_RESEARCH_QUESTIONS,
    ),
    (
        "numeric_evidence",
        "Required headline numbers remain visible",
        [],
        MISSING_NUMERIC_STATEMENTS,
        not MISSING_NUMERIC_STATEMENTS,
    ),
    (
        "publication_figures",
        "The summary references all six publication figures",
        6,
        RELOADED_SUPERVISOR_SUMMARY.count("!["),
        (
            RELOADED_SUPERVISOR_SUMMARY.count("![")
            == 6
        ),
    ),
    (
        "key_findings_schema",
        "The key-findings JSON uses the approved schema",
        "supervisor_key_findings.v1",
        RELOADED_KEY_FINDINGS.get(
            "schema_version"
        ),
        (
            RELOADED_KEY_FINDINGS.get(
                "schema_version"
            )
            == "supervisor_key_findings.v1"
        ),
    ),
    (
        "key_findings_count",
        "All eight validated findings reload",
        8,
        RELOADED_KEY_FINDINGS.get(
            "finding_count"
        ),
        (
            RELOADED_KEY_FINDINGS.get(
                "finding_count"
            )
            == 8
            and len(
                RELOADED_KEY_FINDINGS.get(
                    "findings",
                    [],
                )
            )
            == 8
        ),
    ),
    (
        "key_findings_scientific_role",
        "The key-findings output declares no new science",
        False,
        RELOADED_KEY_FINDINGS.get(
            "creates_new_scientific_evidence"
        ),
        (
            RELOADED_KEY_FINDINGS.get(
                "creates_new_scientific_evidence"
            )
            is False
        ),
    ),
    (
        "open_question_count",
        "Seven supervisor decisions are listed",
        7,
        RELOADED_OPEN_QUESTIONS.count(
            "### Decision"
        ),
        (
            RELOADED_OPEN_QUESTIONS.count(
                "### Decision"
            )
            == 7
        ),
    ),
    (
        "feedback_agenda_sections",
        "The meeting agenda contains six timed sections",
        6,
        RELOADED_FEEDBACK_AGENDA.count("## "),
        (
            RELOADED_FEEDBACK_AGENDA.count("## ")
            == 6
        ),
    ),
    (
        "obsolete_report_paths",
        "The new summary contains no retired global output paths",
        [],
        FOUND_OBSOLETE_REPORT_PATHS,
        not FOUND_OBSOLETE_REPORT_PATHS,
    ),
)

for check_id, description, expected, observed, passed in BATCH_4_CHECKS:
    VALIDATION.add(
        validation_stage="batch_4_supervisor_documents",
        check_id=check_id,
        check_description=description,
        severity="blocking",
        expected=expected,
        observed=observed,
        passed=bool(passed),
        details=(
            ""
            if passed
            else (
                "A supervisor-facing document does not "
                "match the approved report contract."
            )
        ),
    )

VALIDATION.raise_for_blocking()

BATCH_4_VALIDATION = VALIDATION.to_dataframe()

BATCH_4_STAGE_SUMMARY = (
    BATCH_4_VALIDATION
    .groupby(
        ["validation_stage", "severity"],
        dropna=False,
    )
    .agg(
        checks=("check_id", "size"),
        passed=("passed", "sum"),
    )
    .reset_index()
)

BATCH_4_STAGE_SUMMARY["failed"] = (
    BATCH_4_STAGE_SUMMARY["checks"]
    - BATCH_4_STAGE_SUMMARY["passed"]
)

display(BATCH_4_STAGE_SUMMARY)

print("Batch 4 supervisor documents passed.")
print("Cumulative validation checks:", len(BATCH_4_VALIDATION))
print("Blocking failures:", len(VALIDATION.blocking_failures))
print("Supervisor-summary sections:", len(REQUIRED_SUMMARY_HEADINGS))
print("Embedded figure links:", RELOADED_SUPERVISOR_SUMMARY.count("!["))
print("Key findings:", RELOADED_KEY_FINDINGS["finding_count"])
print("Open decisions:", RELOADED_OPEN_QUESTIONS.count("### Decision"))
print("Canonical files persisted through Batch 4:", len(
    OBSERVED_BATCH_4_FILES
))

,validation_stage,severity,checks,passed,failed
0,batch_1_contract,blocking,11,11,0
1,batch_1_dependencies_sources,blocking,9,9,0
2,batch_1_final_preflight,blocking,1,1,0
3,batch_1_inventory_governance,blocking,15,15,0
4,batch_2_artifact_registry,blocking,10,10,0
5,batch_2_final_audit,blocking,1,1,0
6,batch_2_upstream_manifests,blocking,14,14,0
7,batch_3_evidence_inputs,blocking,9,9,0
8,batch_3_final_synthesis,blocking,1,1,0
9,batch_3_model_synthesis,blocking,8,8,0


Batch 4 supervisor documents passed.
Cumulative validation checks: 99
Blocking failures: 0
Supervisor-summary sections: 10
Embedded figure links: 6
Key findings: 8
Open decisions: 7
Canonical files persisted through Batch 4: 4


## Batch 5 — Reproducibility appendix and limitations

This batch:

- records the current Python, platform, package, Git, hardware, dataset, model-revision, configuration, and seed state;
- distinguishes the portable review package from a complete executable repository clone;
- records observed compute evidence without rerunning any model;
- documents deviations, dependency warnings, evidence limits, and intentional package omissions;
- writes and reloads the reproducibility appendix and limitations report.

No restoration inference or scientific metric computation is performed.

In [16]:
from restoration_eval.manifests import (
    git_state,
    sha256_file,
)


SNAPSHOT_GENERATED_AT_UTC = (
    datetime.now(timezone.utc)
    .isoformat(timespec="seconds")
    .replace("+00:00", "Z")
)

CURRENT_GIT_STATE = git_state(PROJECT_ROOT)

REPRO_PACKAGE_DISTRIBUTIONS = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "Pillow": "Pillow",
    "opencv-python": "opencv-python",
    "scikit-image": "scikit-image",
    "scipy": "scipy",
    "PyYAML": "PyYAML",
    "torch": "torch",
    "torchvision": "torchvision",
    "lpips": "lpips",
    "diffusers": "diffusers",
    "transformers": "transformers",
    "safetensors": "safetensors",
    "accelerate": "accelerate",
    "iopaint": "iopaint",
    "streamlit": "streamlit",
    "plotly": "plotly",
    "pyarrow": "pyarrow",
    "jmespath": "jmespath",
}

PACKAGE_VERSION_RECORDS = []

for display_name, distribution_name in (
    REPRO_PACKAGE_DISTRIBUTIONS.items()
):
    try:
        installed_version = metadata.version(
            distribution_name
        )
        availability = "installed"
    except metadata.PackageNotFoundError:
        installed_version = "not_installed"
        availability = "not_installed"

    PACKAGE_VERSION_RECORDS.append(
        {
            "package": display_name,
            "distribution": distribution_name,
            "installed_version": installed_version,
            "availability": availability,
        }
    )

PACKAGE_VERSION_SNAPSHOT = pd.DataFrame(
    PACKAGE_VERSION_RECORDS
).sort_values(
    "package",
    kind="stable",
).reset_index(drop=True)

REQUIREMENT_RECORDS = []

for requirement_path in REQUIREMENTS_PATHS:
    REQUIREMENT_RECORDS.append(
        {
            "relative_path": (
                requirement_path
                .relative_to(PROJECT_ROOT)
                .as_posix()
            ),
            "size_bytes": requirement_path.stat().st_size,
            "sha256": sha256_file(requirement_path),
        }
    )

REQUIREMENT_SNAPSHOT = pd.DataFrame(
    REQUIREMENT_RECORDS
)

MODEL_REPRO_COLUMNS = [
    "model_id",
    "display_name",
    "evaluation_status",
    "implementation",
    "implementation_version",
    "model_identifier",
    "model_revision",
    "configuration_id",
    "seed_policy",
    "execution_device",
    "execution_backend",
    "precision",
    "inference_width",
    "inference_height",
    "output_width",
    "output_height",
    "observed_hardware_json",
]

MODEL_REPRODUCIBILITY = (
    MODEL_CARDS[MODEL_REPRO_COLUMNS]
    .copy()
    .sort_values("model_id", kind="stable")
    .reset_index(drop=True)
)

CONFIGURATION_SNAPSHOT = (
    COPY_PLAN_TABLE.loc[
        COPY_PLAN_TABLE["group"].eq(
            "configuration"
        ),
        [
            "source_path",
            "destination_path",
            "size_bytes",
            "source_sha256",
        ],
    ]
    .copy()
    .sort_values("source_path", kind="stable")
    .reset_index(drop=True)
)

SOURCE_CONFIGURATION_RELATIVE_PATHS = [
    "config/datasets/controlled_50.yaml",
    "config/preprocessing/canonical_768.yaml",
    "config/masks/canonical_binary.yaml",
    "config/experiments/canonical_damage.yaml",
    "config/experiments/damage_size_sensitivity.yaml",
    "config/experiments/evaluation_contract.yaml",
    "config/experiments/lama.yaml",
    "config/experiments/mask_robustness.yaml",
    "config/experiments/opencv_telea.yaml",
    "config/experiments/sdxl.yaml",
    "config/experiments/stable_diffusion.yaml",
    "config/experiments/stable_diffusion_scratch_prompt_ablation.yaml",
    "config/experiments/synthetic_degradation.yaml",
]

SOURCE_CONFIGURATION_RECORDS = []

for relative_path in SOURCE_CONFIGURATION_RELATIVE_PATHS:
    source_path = safe_repo_path(
        relative_path,
        PROJECT_ROOT,
        must_exist=True,
    )

    SOURCE_CONFIGURATION_RECORDS.append(
        {
            "relative_path": relative_path,
            "size_bytes": source_path.stat().st_size,
            "sha256": sha256_file(source_path),
            "package_policy": "indexed_not_bundled",
        }
    )

SOURCE_CONFIGURATION_SNAPSHOT = pd.DataFrame(
    SOURCE_CONFIGURATION_RECORDS
)

UPSTREAM_MANIFEST_SNAPSHOT = (
    UPSTREAM_MANIFESTS[
        [
            "notebook_id",
            "notebook_name",
            "run_id",
            "run_status",
            "completion_gate_passed",
            "blocking_failures",
            "warning_failures",
            "manifest_path",
            "manifest_sha256",
        ]
    ]
    .copy()
    .sort_values("notebook_id", kind="stable")
    .reset_index(drop=True)
)

DATASET_CONFIG_PATH = (
    PROJECT_ROOT
    / "config"
    / "datasets"
    / "controlled_50.yaml"
)

with DATASET_CONFIG_PATH.open(
    "r",
    encoding="utf-8-sig",
) as handle:
    DATASET_CONFIG = yaml.safe_load(handle)

DATASET_IDENTITY = {
    "config_schema_version": (
        DATASET_CONFIG["config_schema_version"]
    ),
    "dataset_id": DATASET_CONFIG["dataset"]["dataset_id"],
    "dataset_version": (
        DATASET_CONFIG["dataset"]["dataset_version"]
    ),
    "dataset_scope": (
        DATASET_CONFIG["dataset"]["dataset_scope"]
    ),
    "execution_profile": (
        DATASET_CONFIG["dataset"]["execution_profile"]
    ),
    "expected_paintings": int(
        DATASET_CONFIG["expected"]["total_paintings"]
    ),
    "configuration_path": (
        DATASET_CONFIG_PATH
        .relative_to(PROJECT_ROOT)
        .as_posix()
    ),
    "configuration_sha256": sha256_file(
        DATASET_CONFIG_PATH
    ),
}

with (
    PROJECT_ROOT
    / "config"
    / "experiments"
    / "stable_diffusion.yaml"
).open(
    "r",
    encoding="utf-8-sig",
) as handle:
    SD15_CONFIG = yaml.safe_load(handle)

with (
    PROJECT_ROOT
    / "config"
    / "experiments"
    / "sdxl.yaml"
).open(
    "r",
    encoding="utf-8-sig",
) as handle:
    SDXL_CONFIG = yaml.safe_load(handle)

with (
    PROJECT_ROOT
    / "config"
    / "evaluation"
    / "diffusion_uncertainty.yaml"
).open(
    "r",
    encoding="utf-8-sig",
) as handle:
    DIFFUSION_UNCERTAINTY_CONFIG = yaml.safe_load(
        handle
    )["diffusion_uncertainty"]

with (
    PROJECT_ROOT
    / "config"
    / "evaluation"
    / "damage_size_diffusion_uncertainty_extension.yaml"
).open(
    "r",
    encoding="utf-8-sig",
) as handle:
    DAMAGE_SIZE_UNCERTAINTY_CONFIG = yaml.safe_load(
        handle
    )["damage_size_diffusion_uncertainty_extension"]

SEED_POLICY_RECORDS = [
    {
        "scope": "OpenCV Telea benchmark",
        "model_id": "opencv_telea",
        "seeds": "not applicable",
        "policy": "deterministic classical algorithm",
    },
    {
        "scope": "LaMa benchmark",
        "model_id": "lama",
        "seeds": "not applicable",
        "policy": "deterministic learned baseline",
    },
    {
        "scope": "Stable Diffusion primary comparison",
        "model_id": "stable_diffusion_inpainting",
        "seeds": str(
            SD15_CONFIG["candidate_design"][
                "primary_seed"
            ]
        ),
        "policy": "fixed primary candidate seed",
    },
    {
        "scope": "Canonical Stable Diffusion uncertainty",
        "model_id": "stable_diffusion_inpainting",
        "seeds": ", ".join(
            str(value)
            for value in (
                DIFFUSION_UNCERTAINTY_CONFIG[
                    "population"
                ]["expected_seeds"]
            )
        ),
        "policy": "four exact seeds per eligible group",
    },
    {
        "scope": "Damage-size Stable Diffusion uncertainty",
        "model_id": "stable_diffusion_inpainting",
        "seeds": ", ".join(
            str(value)
            for value in (
                DAMAGE_SIZE_UNCERTAINTY_CONFIG[
                    "population"
                ]["expected_seeds"]
            )
        ),
        "policy": (
            "seed 2026 reused as the frozen anchor; "
            "2027–2029 generated by Notebook 22"
        ),
    },
    {
        "scope": "SDXL bounded feasibility",
        "model_id": "sdxl_inpainting",
        "seeds": str(SDXL_CONFIG["model"]["seed"]),
        "policy": "one fixed seed per selected feasibility case",
    },
]

SEED_POLICY_SNAPSHOT = pd.DataFrame(
    SEED_POLICY_RECORDS
)

CURRENT_ENVIRONMENT = {
    "python_version": platform.python_version(),
    "python_implementation": (
        platform.python_implementation()
    ),
    "accepted_python_minor_versions": list(
        RUNTIME_CONFIG[
            "accepted_python_minor_versions"
        ]
    ),
    "platform": platform.platform(),
    "machine": platform.machine(),
    "processor": platform.processor(),
}

REPRODUCIBILITY_SNAPSHOT = {
    "schema_version": "reproducibility_snapshot.v1",
    "notebook_id": NOTEBOOK_ID,
    "notebook_stem": NOTEBOOK_STEM,
    "generated_at_utc": SNAPSHOT_GENERATED_AT_UTC,
    "creates_new_scientific_evidence": False,
    "environment": CURRENT_ENVIRONMENT,
    "git": CURRENT_GIT_STATE,
    "dataset": DATASET_IDENTITY,
    "package_versions": json.loads(
        PACKAGE_VERSION_SNAPSHOT.to_json(
            orient="records"
        )
    ),
    "requirements": json.loads(
        REQUIREMENT_SNAPSHOT.to_json(
            orient="records"
        )
    ),
    "models": json.loads(
        MODEL_REPRODUCIBILITY.to_json(
            orient="records"
        )
    ),
    "seed_policies": json.loads(
        SEED_POLICY_SNAPSHOT.to_json(
            orient="records"
        )
    ),
    "bundled_evaluation_configurations": json.loads(
        CONFIGURATION_SNAPSHOT.to_json(
            orient="records"
        )
    ),
    "indexed_source_configurations": json.loads(
        SOURCE_CONFIGURATION_SNAPSHOT.to_json(
            orient="records"
        )
    ),
    "upstream_manifests": json.loads(
        UPSTREAM_MANIFEST_SNAPSHOT.to_json(
            orient="records"
        )
    ),
    "scientific_boundaries": BOUNDARY_FLAGS,
    "package_boundary": {
        "portable_review_bundle": True,
        "complete_repository_clone": False,
        "evaluation_configuration_count": int(
            len(CONFIGURATION_SNAPSHOT)
        ),
        "indexed_source_configuration_count": int(
            len(SOURCE_CONFIGURATION_SNAPSHOT)
        ),
        "upstream_manifest_count": int(
            len(UPSTREAM_MANIFEST_SNAPSHOT)
        ),
    },
}

display(MODEL_REPRODUCIBILITY)
display(SEED_POLICY_SNAPSHOT)
display(PACKAGE_VERSION_SNAPSHOT)
display(REQUIREMENT_SNAPSHOT)

print("Reproducibility evidence assembled in memory.")
print("Git commit:", CURRENT_GIT_STATE["git_commit"])
print("Git branch:", CURRENT_GIT_STATE["git_branch"])
print(
    "Git dirty before final commit:",
    CURRENT_GIT_STATE["git_dirty"],
)
print(
    "Bundled evaluation configurations:",
    len(CONFIGURATION_SNAPSHOT),
)
print(
    "Indexed source configurations:",
    len(SOURCE_CONFIGURATION_SNAPSHOT),
)
print(
    "Upstream manifest checksums:",
    len(UPSTREAM_MANIFEST_SNAPSHOT),
)
print("Canonical files persisted by this cell: 0")

,model_id,display_name,evaluation_status,implementation,implementation_version,model_identifier,model_revision,configuration_id,seed_policy,execution_device,execution_backend,precision,inference_width,inference_height,output_width,output_height,observed_hardware_json
0,lama,LaMa,fully_evaluated,IOPaint CLI model=lama,1.6.0,big-lama.pt,iopaint_lama_default,lama_iopaint_masked_composite_v1,not_applicable,cuda,iopaint,float32,768,768,768,768,"{""cpu_environment"":""Windows-11-10.0.26200-SP0 ..."
1,opencv_telea,OpenCV Telea,fully_evaluated,OpenCV cv2.INPAINT_TELEA,4.11.0,cv2.INPAINT_TELEA,opencv-4.11.0,opencv_telea_r3_threshold_policy_v1,not_applicable,cpu,opencv_cpu,uint8,768,768,768,768,"{""cpu_environment"":""Windows-11-10.0.26200-SP0 ..."
2,sdxl_inpainting,SDXL Inpainting,partial_evaluation,Diffusers StableDiffusionXLInpaintPipeline,3.0.0,diffusers/stable-diffusion-xl-1.0-inpainting-0.1,115134f363124c53c7d878647567d04daf26e41e,sdxl_quality_preserving_partial_evaluation_v1,2026,cuda,NaN,float16,768,768,768,768,"{""cuda_available"":true,""cuda_device_name"":""NVI..."
3,stable_diffusion_inpainting,Stable Diffusion Inpainting,fully_evaluated,Diffusers StableDiffusionInpaintPipeline,1.0.0,stable-diffusion-v1-5/stable-diffusion-inpainting,8a4288a76071f7280aedbdb3253bdb9e9d5d84bb,sd15_inpaint_fixed_policy_v1,2026,cuda,diffusers_stable_diffusion_inpaint,float16,512,512,768,768,"{""cuda_available"":true,""cuda_device_name"":""NVI..."


,scope,model_id,seeds,policy
0,OpenCV Telea benchmark,opencv_telea,not applicable,deterministic classical algorithm
1,LaMa benchmark,lama,not applicable,deterministic learned baseline
2,Stable Diffusion primary comparison,stable_diffusion_inpainting,2026,fixed primary candidate seed
3,Canonical Stable Diffusion uncertainty,stable_diffusion_inpainting,"2026, 2027, 2028, 2029",four exact seeds per eligible group
4,Damage-size Stable Diffusion uncertainty,stable_diffusion_inpainting,"2026, 2027, 2028, 2029",seed 2026 reused as the frozen anchor; 2027–20...
5,SDXL bounded feasibility,sdxl_inpainting,2026,one fixed seed per selected feasibility case


,package,distribution,installed_version,availability
0,Pillow,Pillow,9.5.0,installed
1,PyYAML,PyYAML,6.0.3,installed
2,accelerate,accelerate,1.14.0,installed
3,diffusers,diffusers,0.27.2,installed
4,iopaint,iopaint,1.6.0,installed
5,jmespath,jmespath,1.1.0,installed
6,lpips,lpips,0.1.4,installed
7,matplotlib,matplotlib,3.11.0,installed
8,numpy,numpy,1.26.4,installed
9,opencv-python,opencv-python,4.11.0.86,installed


,relative_path,size_bytes,sha256
0,requirements.txt,107,0d66b4c3bb24e5c03647df89121bd50e76167d6cde0970...
1,requirements_experiments.txt,848,fca385c534df4623f15d058f5b7d188034a6aeb01add4b...


Reproducibility evidence assembled in memory.
Git commit: ba720c143b74179ceafa2264cd1d9808bfc48944
Git branch: main
Git dirty before final commit: True
Bundled evaluation configurations: 25
Indexed source configurations: 13
Upstream manifest checksums: 35
Canonical files persisted by this cell: 0


In [17]:
def markdown_table(
    frame: pd.DataFrame,
    columns: list[str],
    labels: list[str],
) -> str:
    def clean(value: object) -> str:
        if pd.isna(value):
            return ""
        return (
            str(value)
            .replace("|", "\\|")
            .replace("\r", " ")
            .replace("\n", " ")
        )

    header = "| " + " | ".join(labels) + " |"
    divider = "| " + " | ".join(
        "---" for _ in labels
    ) + " |"

    rows = [
        "| "
        + " | ".join(
            clean(row[column])
            for column in columns
        )
        + " |"
        for _, row in frame.iterrows()
    ]

    return "\n".join([header, divider, *rows])


MODEL_REPRO_TABLE = markdown_table(
    MODEL_REPRODUCIBILITY,
    [
        "display_name",
        "evaluation_status",
        "implementation",
        "implementation_version",
        "model_identifier",
        "model_revision",
        "configuration_id",
        "seed_policy",
        "execution_device",
        "precision",
    ],
    [
        "Model",
        "Evaluation status",
        "Implementation",
        "Version",
        "Identifier",
        "Revision",
        "Configuration",
        "Primary seed",
        "Device",
        "Precision",
    ],
)

SEED_POLICY_TABLE = markdown_table(
    SEED_POLICY_SNAPSHOT,
    [
        "scope",
        "model_id",
        "seeds",
        "policy",
    ],
    [
        "Scope",
        "Model",
        "Seed or seeds",
        "Policy",
    ],
)

PACKAGE_VERSION_TABLE = markdown_table(
    PACKAGE_VERSION_SNAPSHOT,
    [
        "package",
        "installed_version",
        "availability",
    ],
    [
        "Package",
        "Recorded notebook environment",
        "Status",
    ],
)

REQUIREMENT_TABLE = markdown_table(
    REQUIREMENT_SNAPSHOT,
    [
        "relative_path",
        "size_bytes",
        "sha256",
    ],
    [
        "Environment file",
        "Bytes",
        "SHA-256",
    ],
)

COMPUTE_REPRO_TABLE = (
    COMPUTE_SCALABILITY.loc[
        (
            COMPUTE_SCALABILITY["record_type"]
            .eq("observed")
        )
        & (
            COMPUTE_SCALABILITY["summary_scope"]
            .eq("overall")
        )
        & (
            COMPUTE_SCALABILITY["experiment_id"]
            .eq("all")
        ),
        [
            "model_id",
            "candidate_count",
            "completed_count",
            "total_runtime_seconds",
            "median_runtime_seconds",
            "p95_runtime_seconds",
            "applicability_status",
        ],
    ]
    .copy()
    .sort_values("model_id", kind="stable")
    .reset_index(drop=True)
)

COMPUTE_REPRO_TABLE.insert(
    1,
    "display_name",
    COMPUTE_REPRO_TABLE["model_id"].map(
        MODEL_CARDS
        .set_index("model_id")["display_name"]
        .to_dict()
    ),
)

COMPUTE_NUMERIC_COLUMNS = [
    "candidate_count",
    "completed_count",
    "total_runtime_seconds",
    "median_runtime_seconds",
    "p95_runtime_seconds",
]

for column in COMPUTE_NUMERIC_COLUMNS:
    COMPUTE_REPRO_TABLE[column] = pd.to_numeric(
        COMPUTE_REPRO_TABLE[column],
        errors="coerce",
    ).round(3)

if len(COMPUTE_REPRO_TABLE) != 4:
    raise ValueError(
        "Expected four observed overall compute rows, "
        f"found {len(COMPUTE_REPRO_TABLE)}."
    )

if COMPUTE_REPRO_TABLE["display_name"].isna().any():
    missing_models = (
        COMPUTE_REPRO_TABLE.loc[
            COMPUTE_REPRO_TABLE[
                "display_name"
            ].isna(),
            "model_id",
        ]
        .astype(str)
        .tolist()
    )
    raise ValueError(
        "Missing display names for compute models: "
        f"{missing_models}"
    )

COMPUTE_TABLE_TEXT = markdown_table(
    COMPUTE_REPRO_TABLE,
    [
        "display_name",
        "candidate_count",
        "completed_count",
        "total_runtime_seconds",
        "median_runtime_seconds",
        "p95_runtime_seconds",
        "applicability_status",
    ],
    [
        "Model",
        "Candidates",
        "Completed",
        "Total seconds",
        "Median seconds",
        "P95 seconds",
        "Applicability",
    ],
)

REPRODUCIBILITY_APPENDIX_TEXT = f"""# Reproducibility Appendix

**Notebook:** `{NOTEBOOK_STEM}.ipynb`  
**Snapshot generated:** `{SNAPSHOT_GENERATED_AT_UTC}`  
**Dataset:** `{DATASET_IDENTITY["dataset_scope"]}` version `{DATASET_IDENTITY["dataset_version"]}`  
**Scientific role:** packaging and traceability only; no new scientific evidence is created.

## 1. Reproducibility boundary

This appendix records how the completed evidence was produced and how it can be audited.

The portable Notebook 36 package is a **review bundle**, not a complete executable clone of the repository. It contains validated reports, figures, compact tables, model cards, evaluation configurations, environment declarations, manifests, application entry points, and provenance.

A full experimental rerun additionally requires:

- the repository notebooks and helper modules;
- the controlled source paintings and metadata;
- dataset, preprocessing, mask, and experiment configurations;
- locally available model weights or caches;
- sufficient storage and compatible CPU or GPU hardware.

The package republishes validated evidence. It does not rerun restoration inference or recompute metrics.

## 2. Recommended reproduction sequence

1. Check out the repository at the recorded Git revision.
2. Create Python 3.11 or 3.12 environments as appropriate.
3. Install `requirements_experiments.txt` for notebook reproduction.
4. Confirm the controlled dataset configuration and raw image availability.
5. Run Notebooks 01–36 in numeric order.
6. Clear only the current notebook's owned output directory before a complete rerun.
7. Inspect each notebook's validation table and run manifest before continuing.
8. Refresh `outputs/inventory/` after the final validated notebook.
9. Use `requirements.txt` for the read-only Streamlit dashboard.
10. Do not interpret a successful run as conservation approval.

## 3. Dataset identity

- Dataset ID: **{DATASET_IDENTITY["dataset_id"]}**
- Dataset version: **{DATASET_IDENTITY["dataset_version"]}**
- Dataset scope: **{DATASET_IDENTITY["dataset_scope"]}**
- Expected paintings: **{DATASET_IDENTITY["expected_paintings"]}**
- Configuration schema: `{DATASET_IDENTITY["config_schema_version"]}`
- Configuration path: `{DATASET_IDENTITY["configuration_path"]}`
- Configuration SHA-256: `{DATASET_IDENTITY["configuration_sha256"]}`

This identity refers to the controlled 50-painting collection. It does not claim coverage of real conservation treatments or unseen museum collections.

## 4. Recorded Git and runtime state

- Git commit before the final Notebook 36 commit: `{CURRENT_GIT_STATE["git_commit"]}`
- Git branch: `{CURRENT_GIT_STATE["git_branch"]}`
- Git dirty state during notebook execution: `{CURRENT_GIT_STATE["git_dirty"]}`
- Git inspection error: `{CURRENT_GIT_STATE["git_error"] or "none"}`
- Python: `{CURRENT_ENVIRONMENT["python_implementation"]} {CURRENT_ENVIRONMENT["python_version"]}`
- Accepted project Python minors: `{", ".join(CURRENT_ENVIRONMENT["accepted_python_minor_versions"])}`
- Platform: `{CURRENT_ENVIRONMENT["platform"]}`
- Machine: `{CURRENT_ENVIRONMENT["machine"]}`

A dirty state is expected while the current notebook and its outputs are awaiting the user's final commit. The final repository commit should be recorded separately after validation.

## 5. Model identities and revisions

{MODEL_REPRO_TABLE}

### Interpretation

- OpenCV Telea and LaMa are deterministic under their fixed project contracts.
- Stable Diffusion uses a fixed primary seed and repeated-seed subsets for uncertainty analysis.
- SDXL uses one seed across ten bounded feasibility cases and has no empirical repeated-seed uncertainty estimate.
- Model revision identifiers improve traceability but do not guarantee byte-identical GPU results across hardware and software stacks.

## 6. Seed policies

{SEED_POLICY_TABLE}

Repeated seeds are nested observations within paintings and cases. They must not be counted as independent paintings.

## 7. Observed compute evidence

{COMPUTE_TABLE_TEXT}

These are recorded upstream observations from the local execution environment. They are not universal speed benchmarks. Any 300-painting scalability values elsewhere in the project are transparent projections rather than executed experiments.

## 8. Environment declarations

{REQUIREMENT_TABLE}

The dashboard environment and experiment environment are intentionally separated. The exact tested Notebook 35 dashboard environment contained four dependency-version differences from the deployment pins; despite those differences, all eight pages passed the local runtime smoke test.

### Packages observed in the Notebook 36 kernel

{PACKAGE_VERSION_TABLE}

An installed version of `not_installed` means the package was not required for this packaging notebook. It does not imply that the package was absent from the environment used by its producing notebook.

## 9. Configuration and manifest traceability

- Bundled evaluation configurations: **{len(CONFIGURATION_SNAPSHOT)}**
- Indexed source configurations: **{len(SOURCE_CONFIGURATION_SNAPSHOT)}**
- Upstream run manifests: **{len(UPSTREAM_MANIFEST_SNAPSHOT)}**
- Completed upstream gates: **{int(UPSTREAM_MANIFEST_SNAPSHOT["completion_gate_passed"].sum())} of {len(UPSTREAM_MANIFEST_SNAPSHOT)}**
- Upstream blocking failures: **{int(UPSTREAM_MANIFEST_SNAPSHOT["blocking_failures"].sum())}**
- Upstream warning failures: **{int(UPSTREAM_MANIFEST_SNAPSHOT["warning_failures"].sum())}**

The 25 bundled YAMLs are the approved `config/evaluation` snapshots. The 13 indexed source configurations record the dataset, preprocessing, canonical mask, and current experiment contracts needed by a full repository rerun. Their paths and SHA-256 checksums are retained in the reproducibility snapshot.

## 10. Hardware record

The model cards retain the observed execution hardware:

- **OpenCV Telea:** CPU execution.
- **LaMa:** NVIDIA GeForce RTX 3060 Laptop GPU using CUDA and IOPaint.
- **Stable Diffusion:** NVIDIA GeForce RTX 3060 Laptop GPU using CUDA and float16 Diffusers inference.
- **SDXL:** the same 6 GB laptop GPU with model CPU offload.

These observations document the completed runs. They do not establish universal minimum hardware requirements.

## 11. Dashboard reproduction

From the repository root:

```text
python -m pip install -r requirements.txt
python -m streamlit run streamlit_app.py
```

The dashboard reads the fixed Notebook 34 presentation package. The Notebook 36 portable review bundle includes the application entry point and helper for inspection, but it does not duplicate the complete Notebook 34 dashboard asset collection.
Notebook 35 validated all eight pages for local demonstration. No completed public deployment URL is recorded.

## 12. Verification route

A reviewer can verify the package by:
    1. reading the package manifest;
    2. comparing the declared file count and byte count with disk;
    3. recalculating individual SHA-256 values;
    4. checking the package-tree checksum;
    5. confirming that all five HTML reports are self-contained;
    6. checking all 35 upstream run manifests;
    7. reviewing the limitations and intentional omissions;
    8. tracing an indexed artifact back to its canonical repository-relative path.

## 13. Interpretation boundary

Reproducibility means that inputs, configurations, versions, seeds, outputs, and known deviations are recorded transparently.

It does not mean that:
- GPU diffusion results are guaranteed to be byte-identical on every machine;
- synthetic damage represents every real conservation condition;
- metric agreement establishes historical correctness;
- uncertainty is calibrated confidence;
- computational flags are expert ground truth;
- a model output is approved for conservation treatment.
  """

LIMITATIONS_AND_DEVIATIONS_TEXT = f"""# Limitations and Deviations

**Notebook:** `{NOTEBOOK_STEM}.ipynb`  
**Recorded:** `{SNAPSHOT_GENERATED_AT_UTC}`  
**Purpose:** Keep the limits of the final evidence visible.

## 1. Controlled synthetic scope

The study uses **50 paintings** and controlled synthetic damage. This supports repeatable comparison because a clean reference is available.

It does not establish performance on naturally aged, physically damaged, previously restored, or materially complex paintings. Results remain bounded to the evaluated collection and damage contracts.

## 2. Statistical independence

Paintings—not candidate rows—are the independent unit for grouped inference.

Repeated models, seeds, regions, masks, prompts, and damage levels remain nested within paintings or cases. These repeated observations are **not independent paintings**. Treating all **1,785 candidates** as independent paintings would overstate the evidence.

## 3. Metric limits

The framework intentionally keeps metric families and regions separate.

- Reference metrics measure pixel or structural similarity to the controlled clean image.
- LPIPS measures learned perceptual distance.
- CLIP and DINOv2 provide general feature-affinity evidence.
- Texture, colour, and seam metrics are diagnostic proxies.
- Semantic similarity is not historical authenticity.
- No universal combined score is reported.

A model can perform well on one metric while performing poorly on another. One metric must not be treated as a complete definition of restoration quality.

## 4. Model-specific limits

### OpenCV Telea

Telea is fast, deterministic, and effective for some thin or local regions. It has no semantic understanding and is not expected to reconstruct large missing structures faithfully.

### LaMa

LaMa is the strongest general benchmark baseline in this controlled study, leading **10 of 11 quality anchors**.

Its learned priors can still smooth texture or create plausible but unsupported structure. The exact licence of the downloaded converted weight artifact was not separately verified by the project.

### Stable Diffusion

Stable Diffusion produces prompt-conditioned candidates rather than historically verified reconstructions. It is more variable across metrics and seeds, and thin scratch geometry remains difficult even after the scratch-aware prompt ablation.

The **1,330 executed candidates** include supporting prompt and repeated-seed evidence. The comparative catalog retains **955 approved Stable Diffusion candidates**.

### SDXL

SDXL is a **ten-case feasibility study**, not a fourth complete benchmark. It is much slower on the recorded hardware, covers five paintings, and has only one seed per case.

## 5. Uncertainty limits

The **165 repeated-seed groups** measure empirical Stable Diffusion variability:

- 130 canonical groups;
- 35 damage-size groups;
- four seeds per group.

This is an inspection signal, **not calibrated confidence**. Low variability can occur when all candidates are consistently wrong. High variability does not by itself prove that every candidate is unusable.

Telea and LaMa are deterministic under their fixed contracts. Their input variation is therefore described as robustness or sensitivity rather than generative uncertainty.

## 6. Trustworthiness and explainability limits

Computational flags, counterfactuals, neighbours, saliency-style evidence, and spatial maps help identify where closer review is needed.

They are **not expert ground truth**.

The **1,703 of 1,785 candidates** receiving conservative review guidance should not be described as 1,703 objectively failed restorations. The rules are intentionally cautious and require expert interpretation.

Retrieval neighbours provide visual or semantic context. They do not prove that a restoration is correct.

## 7. Hardware and software deviations

- Notebook 36 ran under Python `{CURRENT_ENVIRONMENT["python_version"]}`.
- Python 3.11 remains the recommended fresh environment.
- Python 3.11 and 3.12 are both accepted by the project contract.
- Notebook 35 tested Streamlit `1.59.0`, while the deployment pin is `1.56.0`.
- Pillow, Plotly, PyArrow, and Streamlit differed from the declared Notebook 35 deployment pins.
- All eight dashboard pages nevertheless passed the recorded local smoke test.
- CUDA inference may not be byte-identical across GPUs, drivers, CUDA versions, or library builds.
- Observed runtimes describe one local workstation and are not universal benchmarks.

These are transparent reproducibility deviations, not hidden validation passes.

## 8. Deployment status

The Streamlit application is ready for local supervisor demonstration.

Public deployment is **not completed**. No external platform or public URL is recorded, so the project must not claim completed public availability.

## 9. Portable-package boundary

The portable package bundles compact material required for efficient review. Large or redundant collections are **indexed but not bundled**, including:

- all restoration candidates;
- raw and processed painting images;
- complete difference-map and uncertainty-map collections;
- 30 case reports and 50 painting reports;
- 30 selected-case grids;
- the complete Notebook 34 dashboard visual collection;
- model weights and local caches;
- the full executable notebook repository;
- dataset, preprocessing, mask, and experiment YAML files outside `config/evaluation`.

The reproducibility snapshot records canonical paths and checksums for critical source configurations. A full experimental rerun still requires the repository and its input data.

## 10. Compute and scalability boundary

Observed runtime and storage values come from the completed local runs.

Any 300-painting scalability scenario is a linear projection. It is not an executed experiment, performance guarantee, confidence interval, or cloud-cost estimate.

## 11. Conservation boundary

The project does not:

- authenticate artworks;
- establish original artistic intent;
- verify historical reconstruction;
- prescribe physical conservation treatment;
- replace conservator review;
- provide automatic approval for restored candidates.

Visual plausibility is not the same as restoration trustworthiness.

## 12. Final limitation conclusion

The evidence supports a transparent, region-aware, multi-metric comparison of selected pretrained methods under controlled synthetic damage.

It does not support universal model ranking, real-world conservation generality, calibrated confidence, automatic acceptance, or conservation approval.
"""

print("Reproducibility appendix constructed.")
print(
    "Appendix characters:",
    len(REPRODUCIBILITY_APPENDIX_TEXT),
)
print(
    "Appendix sections:",
    REPRODUCIBILITY_APPENDIX_TEXT.count("## "),
)
print("Limitations report constructed.")
print(
    "Limitations characters:",
    len(LIMITATIONS_AND_DEVIATIONS_TEXT),
)
print(
    "Limitations sections:",
    LIMITATIONS_AND_DEVIATIONS_TEXT.count("## "),
)

Reproducibility appendix constructed.
Appendix characters: 10279
Appendix sections: 15
Limitations report constructed.
Limitations characters: 6548
Limitations sections: 16


In [18]:
REPRODUCIBILITY_APPENDIX_PATH = OUTPUT_PATHS[
    "reproducibility_appendix"
]

LIMITATIONS_AND_DEVIATIONS_PATH = OUTPUT_PATHS[
    "limitations_and_deviations"
]

atomic_write_text(
    REPRODUCIBILITY_APPENDIX_PATH,
    REPRODUCIBILITY_APPENDIX_TEXT,
)

atomic_write_text(
    LIMITATIONS_AND_DEVIATIONS_PATH,
    LIMITATIONS_AND_DEVIATIONS_TEXT,
)

BATCH_5_WRITTEN_PATHS = [
    REPRODUCIBILITY_APPENDIX_PATH,
    LIMITATIONS_AND_DEVIATIONS_PATH,
]

BATCH_5_WRITE_VIEW = pd.DataFrame(
    [
        {
            "artifact": path.stem,
            "relative_path": (
                path
                .relative_to(PROJECT_ROOT)
                .as_posix()
            ),
            "format": (
                path.suffix.lower().lstrip(".")
            ),
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }
        for path in BATCH_5_WRITTEN_PATHS
    ]
)

display(BATCH_5_WRITE_VIEW)

print("Batch 5 reports persisted.")
print("Files written:", len(BATCH_5_WRITTEN_PATHS))
print(
    "Total bytes:",
    int(BATCH_5_WRITE_VIEW["size_bytes"].sum()),
)
print(
    "Canonical files persisted through Batch 5:",
    len(BATCH_4_WRITTEN_PATHS)
    + len(BATCH_5_WRITTEN_PATHS),
)

,artifact,relative_path,format,size_bytes,sha256
0,reproducibility_appendix,outputs/36_supervisor_publication_reproducibil...,md,10283,0f565c74b0570c4b83a7ddb50468ea18020612676bfa33...
1,limitations_and_deviations,outputs/36_supervisor_publication_reproducibil...,md,6552,88a368e2331da7b5e84ecbe93de2ed6e0f36df3d3ec4c2...


Batch 5 reports persisted.
Files written: 2
Total bytes: 16835
Canonical files persisted through Batch 5: 6


In [19]:
RELOADED_REPRODUCIBILITY_APPENDIX = (
    REPRODUCIBILITY_APPENDIX_PATH.read_text(
        encoding="utf-8",
    )
)

RELOADED_LIMITATIONS_AND_DEVIATIONS = (
    LIMITATIONS_AND_DEVIATIONS_PATH.read_text(
        encoding="utf-8",
    )
)

EXPECTED_FILES_THROUGH_BATCH_5 = sorted(
    path.relative_to(OUTPUT_ROOT).as_posix()
    for path in (
        BATCH_4_WRITTEN_PATHS
        + BATCH_5_WRITTEN_PATHS
    )
)

OBSERVED_FILES_THROUGH_BATCH_5 = sorted(
    path.relative_to(OUTPUT_ROOT).as_posix()
    for path in OUTPUT_ROOT.rglob("*")
    if path.is_file()
)

REQUIRED_REPRODUCIBILITY_HEADINGS = [
    "## 1. Reproducibility boundary",
    "## 2. Recommended reproduction sequence",
    "## 3. Dataset identity",
    "## 4. Recorded Git and runtime state",
    "## 5. Model identities and revisions",
    "## 6. Seed policies",
    "## 7. Observed compute evidence",
    "## 8. Environment declarations",
    "## 9. Configuration and manifest traceability",
    "## 10. Hardware record",
    "## 11. Dashboard reproduction",
    "## 12. Verification route",
    "## 13. Interpretation boundary",
]

MISSING_REPRODUCIBILITY_HEADINGS = [
    heading
    for heading in REQUIRED_REPRODUCIBILITY_HEADINGS
    if heading
    not in RELOADED_REPRODUCIBILITY_APPENDIX
]

REQUIRED_LIMITATION_HEADINGS = [
    "## 1. Controlled synthetic scope",
    "## 2. Statistical independence",
    "## 3. Metric limits",
    "## 4. Model-specific limits",
    "## 5. Uncertainty limits",
    "## 6. Trustworthiness and explainability limits",
    "## 7. Hardware and software deviations",
    "## 8. Deployment status",
    "## 9. Portable-package boundary",
    "## 10. Compute and scalability boundary",
    "## 11. Conservation boundary",
    "## 12. Final limitation conclusion",
]

MISSING_LIMITATION_HEADINGS = [
    heading
    for heading in REQUIRED_LIMITATION_HEADINGS
    if heading
    not in RELOADED_LIMITATIONS_AND_DEVIATIONS
]

REQUIRED_MODEL_REVISIONS = (
    MODEL_REPRODUCIBILITY["model_revision"]
    .astype(str)
    .tolist()
)

MISSING_MODEL_REVISIONS = [
    revision
    for revision in REQUIRED_MODEL_REVISIONS
    if revision
    not in RELOADED_REPRODUCIBILITY_APPENDIX
]

REQUIRED_LIMITATION_PHRASES = [
    "controlled synthetic damage",
    "not calibrated confidence",
    "not expert ground truth",
    "ten-case feasibility study",
    "not completed",
    "indexed but not bundled",
    "not independent paintings",
    "No universal combined score",
]

LIMITATIONS_CASEFOLD = (
    RELOADED_LIMITATIONS_AND_DEVIATIONS.casefold()
)

MISSING_LIMITATION_PHRASES = [
    phrase
    for phrase in REQUIRED_LIMITATION_PHRASES
    if phrase.casefold() not in LIMITATIONS_CASEFOLD
]

CONFIGURATION_CHECKSUMS_VALID = bool(
    CONFIGURATION_SNAPSHOT[
        "source_sha256"
    ]
    .astype(str)
    .str.fullmatch(r"[0-9a-f]{64}")
    .all()
)

SOURCE_CONFIGURATION_CHECKSUMS_VALID = bool(
    SOURCE_CONFIGURATION_SNAPSHOT[
        "sha256"
    ]
    .astype(str)
    .str.fullmatch(r"[0-9a-f]{64}")
    .all()
)

MANIFEST_CHECKSUMS_VALID = bool(
    UPSTREAM_MANIFEST_SNAPSHOT[
        "manifest_sha256"
    ]
    .astype(str)
    .str.fullmatch(r"[0-9a-f]{64}")
    .all()
)

REQUIREMENT_CHECKSUMS_VALID = bool(
    REQUIREMENT_SNAPSHOT[
        "sha256"
    ]
    .astype(str)
    .str.fullmatch(r"[0-9a-f]{64}")
    .all()
)

try:
    json.dumps(
        REPRODUCIBILITY_SNAPSHOT,
        ensure_ascii=False,
    )
    SNAPSHOT_IS_JSON_SERIALIZABLE = True
    SNAPSHOT_SERIALIZATION_ERROR = ""
except (TypeError, ValueError) as exc:
    SNAPSHOT_IS_JSON_SERIALIZABLE = False
    SNAPSHOT_SERIALIZATION_ERROR = (
        f"{type(exc).__name__}: {exc}"
    )

BATCH_5_CHECKS = (
    (
        "canonical_files",
        "Exactly the six approved files exist through Batch 5",
        EXPECTED_FILES_THROUGH_BATCH_5,
        OBSERVED_FILES_THROUGH_BATCH_5,
        (
            OBSERVED_FILES_THROUGH_BATCH_5
            == EXPECTED_FILES_THROUGH_BATCH_5
        ),
    ),
    (
        "reproducibility_appendix_reload",
        "The reproducibility appendix reloads without content loss",
        REPRODUCIBILITY_APPENDIX_TEXT,
        RELOADED_REPRODUCIBILITY_APPENDIX,
        (
            RELOADED_REPRODUCIBILITY_APPENDIX
            == REPRODUCIBILITY_APPENDIX_TEXT
        ),
    ),
    (
        "limitations_reload",
        "The limitations report reloads without content loss",
        LIMITATIONS_AND_DEVIATIONS_TEXT,
        RELOADED_LIMITATIONS_AND_DEVIATIONS,
        (
            RELOADED_LIMITATIONS_AND_DEVIATIONS
            == LIMITATIONS_AND_DEVIATIONS_TEXT
        ),
    ),
    (
        "reproducibility_structure",
        "Every reproducibility section is present",
        [],
        MISSING_REPRODUCIBILITY_HEADINGS,
        not MISSING_REPRODUCIBILITY_HEADINGS,
    ),
    (
        "limitations_structure",
        "Every limitations section is present",
        [],
        MISSING_LIMITATION_HEADINGS,
        not MISSING_LIMITATION_HEADINGS,
    ),
    (
        "model_revisions",
        "All four exact model revisions are recorded",
        [],
        MISSING_MODEL_REVISIONS,
        not MISSING_MODEL_REVISIONS,
    ),
    (
        "seed_coverage",
        "The four-seed uncertainty policy is recorded",
        "2026, 2027, 2028, 2029",
        (
            "2026, 2027, 2028, 2029"
            in RELOADED_REPRODUCIBILITY_APPENDIX
        ),
        (
            "2026, 2027, 2028, 2029"
            in RELOADED_REPRODUCIBILITY_APPENDIX
        ),
    ),
    (
        "evaluation_configuration_snapshot",
        "All 25 bundled evaluation configurations have valid checksums",
        {
            "count": 25,
            "checksums_valid": True,
        },
        {
            "count": len(
                CONFIGURATION_SNAPSHOT
            ),
            "checksums_valid": (
                CONFIGURATION_CHECKSUMS_VALID
            ),
        },
        (
            len(CONFIGURATION_SNAPSHOT) == 25
            and CONFIGURATION_CHECKSUMS_VALID
        ),
    ),
    (
        "source_configuration_snapshot",
        "All 13 indexed source configurations have valid checksums",
        {
            "count": 13,
            "checksums_valid": True,
        },
        {
            "count": len(
                SOURCE_CONFIGURATION_SNAPSHOT
            ),
            "checksums_valid": (
                SOURCE_CONFIGURATION_CHECKSUMS_VALID
            ),
        },
        (
            len(SOURCE_CONFIGURATION_SNAPSHOT) == 13
            and SOURCE_CONFIGURATION_CHECKSUMS_VALID
        ),
    ),
    (
        "manifest_snapshot",
        "All 35 upstream manifests have valid checksums",
        {
            "count": 35,
            "checksums_valid": True,
        },
        {
            "count": len(
                UPSTREAM_MANIFEST_SNAPSHOT
            ),
            "checksums_valid": (
                MANIFEST_CHECKSUMS_VALID
            ),
        },
        (
            len(UPSTREAM_MANIFEST_SNAPSHOT) == 35
            and MANIFEST_CHECKSUMS_VALID
        ),
    ),
    (
        "requirements_snapshot",
        "Both environment declarations have valid checksums",
        {
            "count": 2,
            "checksums_valid": True,
        },
        {
            "count": len(REQUIREMENT_SNAPSHOT),
            "checksums_valid": (
                REQUIREMENT_CHECKSUMS_VALID
            ),
        },
        (
            len(REQUIREMENT_SNAPSHOT) == 2
            and REQUIREMENT_CHECKSUMS_VALID
        ),
    ),
    (
        "accepted_python",
        "The executing Python minor is approved",
        RUNTIME_CONFIG[
            "accepted_python_minor_versions"
        ],
        CURRENT_PYTHON_MINOR,
        (
            CURRENT_PYTHON_MINOR
            in RUNTIME_CONFIG[
                "accepted_python_minor_versions"
            ]
        ),
    ),
    (
        "git_state",
        "Git state was captured successfully",
        "",
        CURRENT_GIT_STATE["git_error"],
        not CURRENT_GIT_STATE["git_error"],
    ),
    (
        "limitation_coverage",
        "All mandatory interpretation limits are explicit",
        [],
        MISSING_LIMITATION_PHRASES,
        not MISSING_LIMITATION_PHRASES,
    ),
    (
        "snapshot_serializable",
        "The in-memory provenance snapshot is JSON serializable",
        "serializable",
        (
            SNAPSHOT_SERIALIZATION_ERROR
            or "serializable"
        ),
        SNAPSHOT_IS_JSON_SERIALIZABLE,
    ),
    (
        "scientific_role",
        "The snapshot explicitly creates no new scientific evidence",
        False,
        REPRODUCIBILITY_SNAPSHOT[
            "creates_new_scientific_evidence"
        ],
        (
            REPRODUCIBILITY_SNAPSHOT[
                "creates_new_scientific_evidence"
            ]
            is False
        ),
    ),
)

BATCH_5_VALIDATION_STAGE = (
    "batch_5_reproducibility_limitations"
)

RETAINED_VALIDATION_CHECKS = [
    check
    for check in VALIDATION.checks
    if (
        check.validation_stage
        != BATCH_5_VALIDATION_STAGE
    )
]

REMOVED_BATCH_5_CHECK_COUNT = (
    len(VALIDATION.checks)
    - len(RETAINED_VALIDATION_CHECKS)
)

if REMOVED_BATCH_5_CHECK_COUNT:
    VALIDATION = ValidationCollector()
    VALIDATION.extend(
        RETAINED_VALIDATION_CHECKS
    )

for (
    check_id,
    description,
    expected,
    observed,
    passed,
) in BATCH_5_CHECKS:
    VALIDATION.add(
        validation_stage=(
            BATCH_5_VALIDATION_STAGE
        ),
        check_id=check_id,
        check_description=description,
        severity="blocking",
        expected=expected,
        observed=observed,
        passed=bool(passed),
        details=(
            ""
            if passed
            else (
                "Reproducibility or limitation "
                "documentation is incomplete."
            )
        ),
    )

print(
    "Stale Batch 5 validation checks replaced:",
    REMOVED_BATCH_5_CHECK_COUNT,
)

VALIDATION.raise_for_blocking()

BATCH_5_VALIDATION = VALIDATION.to_dataframe()

BATCH_5_STAGE_SUMMARY = (
    BATCH_5_VALIDATION
    .groupby(
        ["validation_stage", "severity"],
        dropna=False,
    )
    .agg(
        checks=("check_id", "size"),
        passed=("passed", "sum"),
    )
    .reset_index()
)

BATCH_5_STAGE_SUMMARY["failed"] = (
    BATCH_5_STAGE_SUMMARY["checks"]
    - BATCH_5_STAGE_SUMMARY["passed"]
)

display(BATCH_5_STAGE_SUMMARY)
display(BATCH_5_WRITE_VIEW)

print("Batch 5 reproducibility documentation passed.")
print(
    "Cumulative validation checks:",
    len(BATCH_5_VALIDATION),
)
print(
    "Blocking failures:",
    len(VALIDATION.blocking_failures),
)
print(
    "Reproducibility sections:",
    len(REQUIRED_REPRODUCIBILITY_HEADINGS),
)
print(
    "Limitations sections:",
    len(REQUIRED_LIMITATION_HEADINGS),
)
print(
    "Package versions recorded:",
    len(PACKAGE_VERSION_SNAPSHOT),
)
print(
    "Canonical files persisted through Batch 5:",
    len(OBSERVED_FILES_THROUGH_BATCH_5),
)

Stale Batch 5 validation checks replaced: 0


,validation_stage,severity,checks,passed,failed
0,batch_1_contract,blocking,11,11,0
1,batch_1_dependencies_sources,blocking,9,9,0
2,batch_1_final_preflight,blocking,1,1,0
3,batch_1_inventory_governance,blocking,15,15,0
4,batch_2_artifact_registry,blocking,10,10,0
5,batch_2_final_audit,blocking,1,1,0
6,batch_2_upstream_manifests,blocking,14,14,0
7,batch_3_evidence_inputs,blocking,9,9,0
8,batch_3_final_synthesis,blocking,1,1,0
9,batch_3_model_synthesis,blocking,8,8,0


,artifact,relative_path,format,size_bytes,sha256
0,reproducibility_appendix,outputs/36_supervisor_publication_reproducibil...,md,10283,0f565c74b0570c4b83a7ddb50468ea18020612676bfa33...
1,limitations_and_deviations,outputs/36_supervisor_publication_reproducibil...,md,6552,88a368e2331da7b5e84ecbe93de2ed6e0f36df3d3ec4c2...


Batch 5 reproducibility documentation passed.
Cumulative validation checks: 115
Blocking failures: 0
Reproducibility sections: 13
Limitations sections: 12
Package versions recorded: 20
Canonical files persisted through Batch 5: 6


## Batch 6 — Curated package assembly

This batch:

- materializes the fixed 106-file copy plan;
- verifies every copied source by SHA-256;
- adds the six Notebook 36 supervisor documents;
- writes the package README and reproducibility snapshot;
- prints copy progress every ten files;
- rejects unexpected package files and stale temporary files.

The package remains a compact review bundle rather than a full executable repository clone.

In [20]:
import os
import shutil

from restoration_eval.supervisor_package import (
    materialize_copy_plan,
    package_file_records,
    package_tree_checksum,
)


PACKAGE_ROOT = OUTPUT_PATHS["package_root"]

PACKAGE_GENERATED_COPY_MAP = {
    SUPERVISOR_SUMMARY_PATH: (
        PACKAGE_ROOT
        / "reports"
        / "supervisor_summary.md"
    ),
    REPRODUCIBILITY_APPENDIX_PATH: (
        PACKAGE_ROOT
        / "reports"
        / "reproducibility_appendix.md"
    ),
    LIMITATIONS_AND_DEVIATIONS_PATH: (
        PACKAGE_ROOT
        / "reports"
        / "limitations_and_deviations.md"
    ),
    KEY_FINDINGS_PATH: (
        PACKAGE_ROOT
        / "data"
        / "key_findings.json"
    ),
    OPEN_QUESTIONS_PATH: (
        PACKAGE_ROOT
        / "data"
        / "open_questions.md"
    ),
    FEEDBACK_AGENDA_PATH: (
        PACKAGE_ROOT
        / "data"
        / "feedback_agenda.md"
    ),
}

PACKAGE_README_PATH = OUTPUT_PATHS[
    "package_readme"
]

REPRODUCIBILITY_SNAPSHOT_PATH = OUTPUT_PATHS[
    "reproducibility_snapshot"
]

FIXED_COPY_DESTINATIONS = {
    item.destination.resolve()
    for item in COPY_PLAN
}

GENERATED_COPY_DESTINATIONS = {
    path.resolve()
    for path in PACKAGE_GENERATED_COPY_MAP.values()
}

GENERATED_NOTEBOOK_DESTINATIONS = {
    PACKAGE_README_PATH.resolve(),
    REPRODUCIBILITY_SNAPSHOT_PATH.resolve(),
}

APPROVED_BATCH_6_DESTINATIONS = (
    FIXED_COPY_DESTINATIONS
    | GENERATED_COPY_DESTINATIONS
    | GENERATED_NOTEBOOK_DESTINATIONS
)

EXISTING_PACKAGE_FILES = {
    path.resolve()
    for path in PACKAGE_ROOT.rglob("*")
    if path.is_file()
}

UNEXPECTED_PREASSEMBLY_FILES = sorted(
    path.relative_to(PACKAGE_ROOT).as_posix()
    for path in (
        EXISTING_PACKAGE_FILES
        - APPROVED_BATCH_6_DESTINATIONS
    )
)

if UNEXPECTED_PREASSEMBLY_FILES:
    raise RuntimeError(
        "Unexpected files already exist in the package: "
        f"{UNEXPECTED_PREASSEMBLY_FILES}"
    )

if len(APPROVED_BATCH_6_DESTINATIONS) != 114:
    raise ValueError(
        "Expected 114 unique Batch 6 package destinations, "
        f"found {len(APPROVED_BATCH_6_DESTINATIONS)}."
    )

TEXT_FENCE = chr(96) * 3

PACKAGE_README_TEXT = f"""# Painting Restoration Evaluation Review Package

This package contains the validated supervisor, publication, and reproducibility material produced by Notebook 36.

## Start here

1. Read [Supervisor Summary](reports/supervisor_summary.md).
2. Open [Final Evaluation Report](reports/final_evaluation.html).
3. Review the four model reports under `reports/models/`.
4. Use `tables/` for compact evidence and report indexes.
5. Use `figures/thesis/` and `figures/publication/` for final figures.
6. Read [Limitations and Deviations](reports/limitations_and_deviations.md).
7. Inspect the [Reproducibility Snapshot](provenance/reproducibility_snapshot.json).

## Package scope

The package contains:

- one final self-contained HTML report;
- four self-contained model reports;
- the Notebook 35 deployment-readiness report;
- three Notebook 36 supervisor-facing reports;
- 18 thesis figures;
- six publication figures;
- eight compact tables and report indexes;
- four model cards;
- 35 upstream run manifests;
- 25 evaluation-configuration snapshots;
- two environment declaration files;
- the Streamlit entry point and application helper;
- key findings, open questions, and feedback agenda;
- the Notebook 36 reproducibility snapshot.

## Headline evidence

- **50 paintings**
- **525 registered cases**
- **410 restoration cases**
- **1,785 approved candidates**
- **11 quality anchors**
- **165 repeated-seed uncertainty groups**
- **23,964 indexed visual records**
- **104 indexed reports**
- **10 bounded SDXL feasibility cases**

## Model conclusion

- **LaMa** is the strongest general benchmark baseline and leads 10 of 11 quality anchors.
- **OpenCV Telea** is the fastest deterministic baseline.
- **Stable Diffusion** provides prompt-conditioned and repeated-seed evidence but requires closer case-level review.
- **SDXL** remains a ten-case feasibility study rather than a fourth full benchmark.

## Important boundaries

- Visual plausibility is not historical correctness or restoration trustworthiness.
- Controlled synthetic damage does not establish real-world conservation generality.
- Repeated-seed variability is not calibrated confidence.
- Computational flags are not expert ground truth.
- No universal combined quality or trustworthiness score is reported.
- No result constitutes conservation approval.

## Reports

- [Supervisor summary](reports/supervisor_summary.md)
- [Final evaluation](reports/final_evaluation.html)
- [LaMa model report](reports/models/lama.html)
- [OpenCV Telea model report](reports/models/opencv_telea.html)
- [Stable Diffusion model report](reports/models/stable_diffusion_inpainting.html)
- [SDXL model report](reports/models/sdxl_inpainting.html)
- [Deployment readiness](reports/deployment_readiness.md)
- [Reproducibility appendix](reports/reproducibility_appendix.md)
- [Limitations and deviations](reports/limitations_and_deviations.md)

## Meeting material

- [Key findings](data/key_findings.json)
- [Open questions](data/open_questions.md)
- [Feedback agenda](data/feedback_agenda.md)

## Environment

- `environment/requirements.txt` is the dashboard environment.
- `environment/requirements_experiments.txt` is the notebook and experiment environment.
- Python 3.11 is recommended for a fresh environment.
- Python 3.11 and 3.12 are accepted by the project contract.

## Local dashboard

The included application files do not contain the complete Notebook 34 dashboard asset collection. Run the dashboard from the complete repository checkout:

{TEXT_FENCE}text
python -m pip install -r requirements.txt
python -m streamlit run streamlit_app.py
{TEXT_FENCE}

Notebook 35 validated all eight pages for local demonstration. No completed public deployment URL is recorded.

## Material indexed but not bundled

To keep the package compact, it does not duplicate:

- the complete restoration-image collection;
- raw or processed paintings;
- full difference-map and uncertainty-map collections;
- 30 case reports and 50 painting reports;
- 30 selected-case grids;
- the complete Notebook 34 dashboard visual collection;
- model weights or caches;
- the full notebook repository.

These remain discoverable through the bundled indexes, manifests, and provenance records.

## Integrity

The external Notebook 36 package manifest records:

- each package-relative path;
- byte size;
- SHA-256 checksum;
- total file and byte counts;
- the package-tree checksum.

The package was assembled without restoration inference or scientific metric recomputation.
"""

REPRODUCIBILITY_SNAPSHOT["assembly"] = {
    "assembled_at_utc": (
        datetime.now(timezone.utc)
        .isoformat(timespec="seconds")
        .replace("+00:00", "Z")
    ),
    "fixed_copy_file_count": len(COPY_PLAN),
    "notebook_document_copy_count": len(
        PACKAGE_GENERATED_COPY_MAP
    ),
    "generated_package_file_count": 2,
    "expected_package_file_count": len(
        APPROVED_BATCH_6_DESTINATIONS
    ),
    "planned_copy_size_bytes": (
        COPY_PLAN_SIZE_BYTES
    ),
    "package_size_limit_bytes": (
        MAXIMUM_PACKAGE_SIZE_BYTES
    ),
    "package_root": (
        PACKAGE_ROOT
        .relative_to(PROJECT_ROOT)
        .as_posix()
    ),
}

PACKAGE_GENERATED_PLAN = pd.DataFrame(
    [
        {
            "role": source.stem,
            "source_path": (
                source
                .relative_to(PROJECT_ROOT)
                .as_posix()
            ),
            "destination_path": (
                destination
                .relative_to(PROJECT_ROOT)
                .as_posix()
            ),
            "source_size_bytes": (
                source.stat().st_size
            ),
            "source_sha256": sha256_file(
                source
            ),
        }
        for source, destination in (
            PACKAGE_GENERATED_COPY_MAP.items()
        )
    ]
)

display(PACKAGE_GENERATED_PLAN)

print("Batch 6 assembly plan prepared.")
print(
    "Fixed copy destinations:",
    len(FIXED_COPY_DESTINATIONS),
)
print(
    "Notebook document copies:",
    len(GENERATED_COPY_DESTINATIONS),
)
print("Notebook-generated package files: 2")
print(
    "Total approved package files:",
    len(APPROVED_BATCH_6_DESTINATIONS),
)
print(
    "Unexpected preassembly files:",
    len(UNEXPECTED_PREASSEMBLY_FILES),
)

,role,source_path,destination_path,source_size_bytes,source_sha256
0,supervisor_summary,outputs/36_supervisor_publication_reproducibil...,outputs/36_supervisor_publication_reproducibil...,11922,30776207bc993ac65fb5d6d7e012f4db94d3f267c7e8b7...
1,reproducibility_appendix,outputs/36_supervisor_publication_reproducibil...,outputs/36_supervisor_publication_reproducibil...,10283,0f565c74b0570c4b83a7ddb50468ea18020612676bfa33...
2,limitations_and_deviations,outputs/36_supervisor_publication_reproducibil...,outputs/36_supervisor_publication_reproducibil...,6552,88a368e2331da7b5e84ecbe93de2ed6e0f36df3d3ec4c2...
3,key_findings,outputs/36_supervisor_publication_reproducibil...,outputs/36_supervisor_publication_reproducibil...,8729,5bb22f1284e6d12ee27796d1dfb36878438347a09ca88e...
4,open_questions,outputs/36_supervisor_publication_reproducibil...,outputs/36_supervisor_publication_reproducibil...,1877,972ffb0e7f3f169ee11094ffc3f7f3683b3dec0e756d7f...
5,feedback_agenda,outputs/36_supervisor_publication_reproducibil...,outputs/36_supervisor_publication_reproducibil...,2130,caafc091239e807b8548d388eeff3200d3d3a53a3ab85b...


Batch 6 assembly plan prepared.
Fixed copy destinations: 106
Notebook document copies: 6
Notebook-generated package files: 2
Total approved package files: 114
Unexpected preassembly files: 0


In [21]:
def package_copy_progress(
    completed: int,
    total: int,
    item: object,
) -> None:
    relative_destination = (
        item.destination
        .relative_to(PACKAGE_ROOT)
        .as_posix()
    )

    print(
        f"Copied {completed}/{total}: "
        f"{relative_destination}"
    )


COPIED_FIXED_PATHS = materialize_copy_plan(
    COPY_PLAN,
    progress_callback=package_copy_progress,
)

print(
    "Fixed copy plan completed:",
    len(COPIED_FIXED_PATHS),
)

COPIED_NOTEBOOK_DOCUMENTS = []

for number, (
    source,
    destination,
) in enumerate(
    PACKAGE_GENERATED_COPY_MAP.items(),
    start=1,
):
    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = destination.with_suffix(
        destination.suffix + ".tmp"
    )

    try:
        shutil.copy2(source, temporary)

        if (
            sha256_file(temporary)
            != sha256_file(source)
        ):
            raise IOError(
                "Notebook document checksum mismatch: "
                f"{source}"
            )

        os.replace(temporary, destination)
    finally:
        temporary.unlink(missing_ok=True)

    COPIED_NOTEBOOK_DOCUMENTS.append(
        destination
    )

    print(
        "Added Notebook 36 document "
        f"{number}/{len(PACKAGE_GENERATED_COPY_MAP)}: "
        f"{destination.relative_to(PACKAGE_ROOT)}"
    )

atomic_write_text(
    PACKAGE_README_PATH,
    PACKAGE_README_TEXT,
)

atomic_write_json(
    REPRODUCIBILITY_SNAPSHOT_PATH,
    REPRODUCIBILITY_SNAPSHOT,
)

PACKAGE_FILES_BATCH_6 = package_file_records(
    PACKAGE_ROOT,
    PROJECT_ROOT,
)

PACKAGE_TREE_CHECKSUM_BATCH_6 = (
    package_tree_checksum(
        PACKAGE_FILES_BATCH_6
    )
)

PACKAGE_SIZE_BYTES_BATCH_6 = int(
    PACKAGE_FILES_BATCH_6["size_bytes"].sum()
)

PACKAGE_SIZE_MIB_BATCH_6 = (
    PACKAGE_SIZE_BYTES_BATCH_6
    / (1024 ** 2)
)

print("Curated package assembled.")
print(
    "Fixed copied files:",
    len(COPIED_FIXED_PATHS),
)
print(
    "Notebook document copies:",
    len(COPIED_NOTEBOOK_DOCUMENTS),
)
print(
    "Generated package files:",
    2,
)
print(
    "Package files:",
    len(PACKAGE_FILES_BATCH_6),
)
print(
    "Package size MiB:",
    round(PACKAGE_SIZE_MIB_BATCH_6, 3),
)
print(
    "Package tree checksum:",
    PACKAGE_TREE_CHECKSUM_BATCH_6,
)

Copied 10/106: tables/model_cards.csv
Copied 20/106: environment/requirements_experiments.txt
Copied 30/106: figures/thesis/08_synthetic_degradation.png
Copied 40/106: figures/thesis/18_scalability_projection.png
Copied 50/106: configuration/evaluation/dashboard_assets.yaml
Copied 60/106: configuration/evaluation/lpips.yaml
Copied 70/106: configuration/evaluation/supervisor_package.yaml
Copied 80/106: manifests/notebook_runs/09_opencv_telea_restoration.json
Copied 90/106: manifests/notebook_runs/19_uncertainty_and_spatial_explanation_maps.json
Copied 100/106: manifests/notebook_runs/29_explainable_ai_and_case_retrieval.json
Copied 106/106: manifests/notebook_runs/35_dashboard_and_deployment_validation.json
Fixed copy plan completed: 106
Added Notebook 36 document 1/6: reports\supervisor_summary.md
Added Notebook 36 document 2/6: reports\reproducibility_appendix.md
Added Notebook 36 document 3/6: reports\limitations_and_deviations.md
Added Notebook 36 document 4/6: data\key_findings.jso

In [22]:
FIXED_COPY_AUDIT_RECORDS = []

for row in COPY_PLAN_TABLE.itertuples(
    index=False
):
    destination = (
        PROJECT_ROOT
        / str(row.destination_path)
    )

    observed_sha256 = (
        sha256_file(destination)
        if destination.is_file()
        else ""
    )

    FIXED_COPY_AUDIT_RECORDS.append(
        {
            "source_path": row.source_path,
            "destination_path": (
                row.destination_path
            ),
            "source_sha256": (
                row.source_sha256
            ),
            "destination_sha256": (
                observed_sha256
            ),
            "exists": destination.is_file(),
            "matches": (
                destination.is_file()
                and observed_sha256
                == row.source_sha256
            ),
        }
    )

FIXED_COPY_AUDIT = pd.DataFrame(
    FIXED_COPY_AUDIT_RECORDS
)

GENERATED_COPY_AUDIT_RECORDS = []

for source, destination in (
    PACKAGE_GENERATED_COPY_MAP.items()
):
    source_sha256 = sha256_file(source)

    destination_sha256 = (
        sha256_file(destination)
        if destination.is_file()
        else ""
    )

    GENERATED_COPY_AUDIT_RECORDS.append(
        {
            "source_path": (
                source
                .relative_to(PROJECT_ROOT)
                .as_posix()
            ),
            "destination_path": (
                destination
                .relative_to(PROJECT_ROOT)
                .as_posix()
            ),
            "source_sha256": source_sha256,
            "destination_sha256": (
                destination_sha256
            ),
            "exists": destination.is_file(),
            "matches": (
                destination.is_file()
                and source_sha256
                == destination_sha256
            ),
        }
    )

GENERATED_COPY_AUDIT = pd.DataFrame(
    GENERATED_COPY_AUDIT_RECORDS
)

PACKAGE_FILES_BATCH_6 = package_file_records(
    PACKAGE_ROOT,
    PROJECT_ROOT,
)

PACKAGE_FILES_BATCH_6["top_level_group"] = (
    PACKAGE_FILES_BATCH_6[
        "package_relative_path"
    ]
    .astype(str)
    .map(
        lambda value: (
            value.split("/", 1)[0]
            if "/" in value
            else "root"
        )
    )
)

PACKAGE_GROUP_COUNTS_BATCH_6 = {
    str(group): int(count)
    for group, count in (
        PACKAGE_FILES_BATCH_6
        .groupby(
            "top_level_group",
            dropna=False,
        )
        .size()
        .sort_index()
        .items()
    )
}

EXPECTED_PACKAGE_GROUP_COUNTS_BATCH_6 = {
    "application": 2,
    "configuration": 25,
    "data": 3,
    "environment": 2,
    "figures": 24,
    "manifests": 35,
    "model_cards": 4,
    "provenance": 1,
    "reports": 9,
    "root": 1,
    "tables": 8,
}

RELOADED_PACKAGE_README = (
    PACKAGE_README_PATH.read_text(
        encoding="utf-8",
    )
)

with REPRODUCIBILITY_SNAPSHOT_PATH.open(
    "r",
    encoding="utf-8-sig",
) as handle:
    RELOADED_REPRODUCIBILITY_SNAPSHOT = (
        json.load(handle)
    )

REQUIRED_PACKAGE_README_LINKS = [
    "reports/supervisor_summary.md",
    "reports/final_evaluation.html",
    "reports/models/lama.html",
    "reports/models/opencv_telea.html",
    (
        "reports/models/"
        "stable_diffusion_inpainting.html"
    ),
    "reports/models/sdxl_inpainting.html",
    "reports/deployment_readiness.md",
    "reports/reproducibility_appendix.md",
    "reports/limitations_and_deviations.md",
    (
        "provenance/"
        "reproducibility_snapshot.json"
    ),
]

MISSING_PACKAGE_README_LINKS = [
    link
    for link in REQUIRED_PACKAGE_README_LINKS
    if link not in RELOADED_PACKAGE_README
]

STALE_PACKAGE_TEMP_FILES = sorted(
    path.relative_to(PACKAGE_ROOT).as_posix()
    for path in PACKAGE_ROOT.rglob("*")
    if (
        path.is_file()
        and path.name.endswith(".tmp")
    )
)

PACKAGE_PATHS_INSIDE_ROOT = bool(
    PACKAGE_FILES_BATCH_6[
        "relative_path"
    ]
    .astype(str)
    .map(
        lambda value: (
            (PROJECT_ROOT / value)
            .resolve()
            .is_relative_to(
                PACKAGE_ROOT.resolve()
            )
        )
    )
    .all()
)

OUTPUT_FILES_THROUGH_BATCH_6 = sorted(
    path.relative_to(OUTPUT_ROOT).as_posix()
    for path in OUTPUT_ROOT.rglob("*")
    if path.is_file()
)

BATCH_6_CHECKS = (
    (
        "package_file_count",
        "The assembled package contains exactly 114 files",
        114,
        len(PACKAGE_FILES_BATCH_6),
        len(PACKAGE_FILES_BATCH_6) == 114,
    ),
    (
        "output_file_count",
        "Exactly 120 files exist under the Notebook 36 output root",
        120,
        len(OUTPUT_FILES_THROUGH_BATCH_6),
        len(OUTPUT_FILES_THROUGH_BATCH_6) == 120,
    ),
    (
        "package_group_counts",
        "Package groups match the approved assembly layout",
        EXPECTED_PACKAGE_GROUP_COUNTS_BATCH_6,
        PACKAGE_GROUP_COUNTS_BATCH_6,
        (
            PACKAGE_GROUP_COUNTS_BATCH_6
            == EXPECTED_PACKAGE_GROUP_COUNTS_BATCH_6
        ),
    ),
    (
        "fixed_copy_checksums",
        "All 106 fixed copies match their sources",
        {
            "rows": 106,
            "matches": 106,
        },
        {
            "rows": len(FIXED_COPY_AUDIT),
            "matches": int(
                FIXED_COPY_AUDIT[
                    "matches"
                ].sum()
            ),
        },
        (
            len(FIXED_COPY_AUDIT) == 106
            and FIXED_COPY_AUDIT[
                "matches"
            ].all()
        ),
    ),
    (
        "notebook_document_checksums",
        "All six Notebook 36 document copies match their sources",
        {
            "rows": 6,
            "matches": 6,
        },
        {
            "rows": len(
                GENERATED_COPY_AUDIT
            ),
            "matches": int(
                GENERATED_COPY_AUDIT[
                    "matches"
                ].sum()
            ),
        },
        (
            len(GENERATED_COPY_AUDIT) == 6
            and GENERATED_COPY_AUDIT[
                "matches"
            ].all()
        ),
    ),
    (
        "package_readme",
        "The package README exists and contains every required entry point",
        [],
        MISSING_PACKAGE_README_LINKS,
        (
            PACKAGE_README_PATH.is_file()
            and not MISSING_PACKAGE_README_LINKS
        ),
    ),
    (
        "reproducibility_snapshot",
        "The reproducibility snapshot reloads under the approved schema",
        "reproducibility_snapshot.v1",
        RELOADED_REPRODUCIBILITY_SNAPSHOT.get(
            "schema_version"
        ),
        (
            RELOADED_REPRODUCIBILITY_SNAPSHOT.get(
                "schema_version"
            )
            == "reproducibility_snapshot.v1"
        ),
    ),
    (
        "package_size",
        "The assembled package remains below 50 MiB",
        "<= 50 MiB",
        round(PACKAGE_SIZE_MIB_BATCH_6, 3),
        (
            PACKAGE_SIZE_BYTES_BATCH_6
            <= MAXIMUM_PACKAGE_SIZE_BYTES
        ),
    ),
    (
        "package_paths",
        "Every package file remains inside the package root",
        True,
        PACKAGE_PATHS_INSIDE_ROOT,
        PACKAGE_PATHS_INSIDE_ROOT,
    ),
    (
        "temporary_files",
        "No stale temporary files remain",
        [],
        STALE_PACKAGE_TEMP_FILES,
        not STALE_PACKAGE_TEMP_FILES,
    ),
    (
        "destination_uniqueness",
        "Every package-relative path is unique",
        True,
        bool(
            PACKAGE_FILES_BATCH_6[
                "package_relative_path"
            ].is_unique
        ),
        bool(
            PACKAGE_FILES_BATCH_6[
                "package_relative_path"
            ].is_unique
        ),
    ),
    (
        "assembly_boundary",
        "The package is explicitly a review bundle rather than a repository clone",
        {
            "portable_review_bundle": True,
            "complete_repository_clone": False,
        },
        {
            "portable_review_bundle": (
                RELOADED_REPRODUCIBILITY_SNAPSHOT[
                    "package_boundary"
                ][
                    "portable_review_bundle"
                ]
            ),
            "complete_repository_clone": (
                RELOADED_REPRODUCIBILITY_SNAPSHOT[
                    "package_boundary"
                ][
                    "complete_repository_clone"
                ]
            ),
        },
        (
            RELOADED_REPRODUCIBILITY_SNAPSHOT[
                "package_boundary"
            ][
                "portable_review_bundle"
            ]
            is True
            and RELOADED_REPRODUCIBILITY_SNAPSHOT[
                "package_boundary"
            ][
                "complete_repository_clone"
            ]
            is False
        ),
    ),
)

BATCH_6_VALIDATION_STAGE = (
    "batch_6_package_assembly"
)

RETAINED_VALIDATION_CHECKS = [
    check
    for check in VALIDATION.checks
    if (
        check.validation_stage
        != BATCH_6_VALIDATION_STAGE
    )
]

if (
    len(RETAINED_VALIDATION_CHECKS)
    != len(VALIDATION.checks)
):
    VALIDATION = ValidationCollector()
    VALIDATION.extend(
        RETAINED_VALIDATION_CHECKS
    )

for (
    check_id,
    description,
    expected,
    observed,
    passed,
) in BATCH_6_CHECKS:
    VALIDATION.add(
        validation_stage=(
            BATCH_6_VALIDATION_STAGE
        ),
        check_id=check_id,
        check_description=description,
        severity="blocking",
        expected=expected,
        observed=observed,
        passed=bool(passed),
        details=(
            ""
            if passed
            else (
                "The curated package does not "
                "match its approved assembly contract."
            )
        ),
    )

VALIDATION.raise_for_blocking()

BATCH_6_VALIDATION = VALIDATION.to_dataframe()

BATCH_6_STAGE_SUMMARY = (
    BATCH_6_VALIDATION
    .groupby(
        ["validation_stage", "severity"],
        dropna=False,
    )
    .agg(
        checks=("check_id", "size"),
        passed=("passed", "sum"),
    )
    .reset_index()
)

BATCH_6_STAGE_SUMMARY["failed"] = (
    BATCH_6_STAGE_SUMMARY["checks"]
    - BATCH_6_STAGE_SUMMARY["passed"]
)

PACKAGE_GROUP_VIEW_BATCH_6 = (
    PACKAGE_FILES_BATCH_6
    .groupby(
        "top_level_group",
        dropna=False,
    )
    .agg(
        files=("package_relative_path", "size"),
        size_bytes=("size_bytes", "sum"),
    )
    .reset_index()
)

PACKAGE_GROUP_VIEW_BATCH_6["size_mib"] = (
    PACKAGE_GROUP_VIEW_BATCH_6[
        "size_bytes"
    ]
    / (1024 ** 2)
)

display(BATCH_6_STAGE_SUMMARY)
display(PACKAGE_GROUP_VIEW_BATCH_6)

print("Batch 6 package assembly passed.")
print(
    "Cumulative validation checks:",
    len(BATCH_6_VALIDATION),
)
print(
    "Blocking failures:",
    len(VALIDATION.blocking_failures),
)
print(
    "Package files:",
    len(PACKAGE_FILES_BATCH_6),
)
print(
    "Package size MiB:",
    round(PACKAGE_SIZE_MIB_BATCH_6, 3),
)
print(
    "Package tree checksum:",
    PACKAGE_TREE_CHECKSUM_BATCH_6,
)

,validation_stage,severity,checks,passed,failed
0,batch_1_contract,blocking,11,11,0
1,batch_1_dependencies_sources,blocking,9,9,0
2,batch_1_final_preflight,blocking,1,1,0
3,batch_1_inventory_governance,blocking,15,15,0
4,batch_2_artifact_registry,blocking,10,10,0
5,batch_2_final_audit,blocking,1,1,0
6,batch_2_upstream_manifests,blocking,14,14,0
7,batch_3_evidence_inputs,blocking,9,9,0
8,batch_3_final_synthesis,blocking,1,1,0
9,batch_3_model_synthesis,blocking,8,8,0


,top_level_group,files,size_bytes,size_mib
0,application,2,111186,0.106035
1,configuration,25,456708,0.435551
2,data,3,12736,0.012146
3,environment,2,955,0.000911
4,figures,24,2427579,2.315120
5,manifests,35,734530,0.700502
6,model_cards,4,40142,0.038282
7,provenance,1,39038,0.037230
8,reports,9,24408866,23.278109
9,root,1,4495,0.004287


Batch 6 package assembly passed.
Cumulative validation checks: 127
Blocking failures: 0
Package files: 114
Package size MiB: 27.677
Package tree checksum: 69ff56193ea8a56dcc867d52a1799666c74aad960c59f393ba409fbfa75cec8b


## Batch 7 — Artifact index and package manifest

This batch:

- catalogs every physical file in the 114-file package;
- distinguishes copied files from Notebook 36-generated files;
- indexes the approved large collections that are intentionally not bundled;
- writes the complete artifact index;
- writes an independently readable package manifest;
- records package counts, bytes, checksums, groups, and provenance.

The manifest excludes itself from its checksum tree to avoid a recursive checksum definition.

In [23]:
from restoration_eval.supervisor_package import (
    PACKAGE_MANIFEST_SCHEMA_VERSION,
)


ARTIFACT_INDEX_SCHEMA_VERSION = (
    "supervisor_artifact_index.v1"
)

PACKAGE_MANIFEST_CREATED_AT_UTC = (
    datetime.now(timezone.utc)
    .isoformat(timespec="seconds")
    .replace("+00:00", "Z")
)

PACKAGE_FILES_FOR_INDEX = package_file_records(
    PACKAGE_ROOT,
    PROJECT_ROOT,
)

COPY_PLAN_BY_DESTINATION = {
    str(row.destination_path): row
    for row in COPY_PLAN_TABLE.itertuples(
        index=False
    )
}

GENERATED_SOURCE_BY_DESTINATION = {
    destination
    .relative_to(PROJECT_ROOT)
    .as_posix(): (
        source
        .relative_to(PROJECT_ROOT)
        .as_posix()
    )
    for source, destination in (
        PACKAGE_GENERATED_COPY_MAP.items()
    )
}

GENERATED_ROLE_BY_DESTINATION = {
    (
        PACKAGE_ROOT
        / "reports"
        / "supervisor_summary.md"
    )
    .relative_to(PROJECT_ROOT)
    .as_posix(): "supervisor_summary",
    (
        PACKAGE_ROOT
        / "reports"
        / "reproducibility_appendix.md"
    )
    .relative_to(PROJECT_ROOT)
    .as_posix(): "reproducibility_appendix",
    (
        PACKAGE_ROOT
        / "reports"
        / "limitations_and_deviations.md"
    )
    .relative_to(PROJECT_ROOT)
    .as_posix(): "limitations_and_deviations",
    (
        PACKAGE_ROOT
        / "data"
        / "key_findings.json"
    )
    .relative_to(PROJECT_ROOT)
    .as_posix(): "key_findings",
    (
        PACKAGE_ROOT
        / "data"
        / "open_questions.md"
    )
    .relative_to(PROJECT_ROOT)
    .as_posix(): "open_questions",
    (
        PACKAGE_ROOT
        / "data"
        / "feedback_agenda.md"
    )
    .relative_to(PROJECT_ROOT)
    .as_posix(): "feedback_agenda",
    PACKAGE_README_PATH
    .relative_to(PROJECT_ROOT)
    .as_posix(): "package_readme",
    REPRODUCIBILITY_SNAPSHOT_PATH
    .relative_to(PROJECT_ROOT)
    .as_posix(): "reproducibility_snapshot",
}

UPSTREAM_VALIDATION_BY_ID = (
    MANIFEST_DETAILS
    .set_index("notebook_id")[
        "validation_status"
    ]
    .astype(str)
    .to_dict()
)

PACKAGE_ARTIFACT_RECORDS = []

for row in PACKAGE_FILES_FOR_INDEX.itertuples(
    index=False
):
    repository_path = str(row.relative_path)
    package_path = str(
        row.package_relative_path
    )

    if repository_path in COPY_PLAN_BY_DESTINATION:
        copy_row = COPY_PLAN_BY_DESTINATION[
            repository_path
        ]

        source_path = str(
            copy_row.source_path
        )

        if source_path.startswith("outputs/"):
            producer_notebook_id = (
                source_path
                .split("/", 2)[1]
                .split("_", 1)[0]
            )
        else:
            producer_notebook_id = (
                "repository"
            )

        validation_status = (
            UPSTREAM_VALIDATION_BY_ID.get(
                producer_notebook_id,
                "passed",
            )
        )

        delivery_status = "copied"
        artifact_group = str(copy_row.group)
        artifact_role = str(copy_row.role)
        canonical_source_path = source_path
        source_sha256 = str(
            copy_row.source_sha256
        )

    else:
        producer_notebook_id = NOTEBOOK_ID
        validation_status = "passed"
        delivery_status = (
            "generated_by_notebook_36"
        )
        artifact_group = (
            package_path.split("/", 1)[0]
            if "/" in package_path
            else "root"
        )
        artifact_role = (
            GENERATED_ROLE_BY_DESTINATION[
                repository_path
            ]
        )
        canonical_source_path = (
            GENERATED_SOURCE_BY_DESTINATION.get(
                repository_path,
                repository_path,
            )
        )

        canonical_source = (
            PROJECT_ROOT
            / canonical_source_path
        )

        source_sha256 = (
            sha256_file(canonical_source)
            if canonical_source.is_file()
            else str(row.sha256)
        )

    PACKAGE_ARTIFACT_RECORDS.append(
        {
            "delivery_status": delivery_status,
            "artifact_group": artifact_group,
            "artifact_role": artifact_role,
            "producer_notebook_id": (
                producer_notebook_id
            ),
            "canonical_source_path": (
                canonical_source_path
            ),
            "package_relative_path": (
                package_path
            ),
            "source_index_path": "",
            "collection_path": "",
            "format": str(row.format),
            "size_bytes": int(row.size_bytes),
            "sha256": str(row.sha256),
            "source_sha256": source_sha256,
            "record_count": 1,
            "source_exists": True,
            "validation_status": (
                validation_status
            ),
            "omission_reason": "",
            "schema_version": (
                ARTIFACT_INDEX_SCHEMA_VERSION
            ),
        }
    )

INDEXED_COLLECTION_SPECS = [
    {
        "artifact_group": "reports",
        "artifact_role": (
            "thirty_self_contained_case_reports"
        ),
        "producer_notebook_id": "32",
        "source_index_path": (
            "outputs/"
            "32_case_and_painting_report_generation/"
            "data/case_report_index.csv"
        ),
        "collection_path": (
            "outputs/"
            "32_case_and_painting_report_generation/"
            "reports/cases"
        ),
        "expected_records": 30,
        "omission_reason": (
            "The complete reports remain available "
            "through the bundled case-report index."
        ),
    },
    {
        "artifact_group": "reports",
        "artifact_role": (
            "fifty_self_contained_painting_reports"
        ),
        "producer_notebook_id": "32",
        "source_index_path": (
            "outputs/"
            "32_case_and_painting_report_generation/"
            "data/painting_report_index.csv"
        ),
        "collection_path": (
            "outputs/"
            "32_case_and_painting_report_generation/"
            "reports/paintings"
        ),
        "expected_records": 50,
        "omission_reason": (
            "The complete reports remain available "
            "through the bundled painting-report index."
        ),
    },
    {
        "artifact_group": "figures",
        "artifact_role": (
            "thirty_selected_case_grids"
        ),
        "producer_notebook_id": "32",
        "source_index_path": (
            "outputs/"
            "32_case_and_painting_report_generation/"
            "data/selected_cases.csv"
        ),
        "collection_path": (
            "outputs/"
            "32_case_and_painting_report_generation/"
            "figures/selected_case_grids"
        ),
        "expected_records": 30,
        "omission_reason": (
            "The grids are discoverable from the "
            "bundled selected-case index."
        ),
    },
    {
        "artifact_group": "dashboard",
        "artifact_role": (
            "complete_dashboard_visual_index"
        ),
        "producer_notebook_id": "34",
        "source_index_path": (
            "outputs/"
            "34_final_streamlit_dashboard_assets/"
            "data/dashboard_indexes/"
            "visual_asset_index.csv"
        ),
        "collection_path": (
            "canonical_paths_recorded_per_index_row"
        ),
        "expected_records": 23964,
        "omission_reason": (
            "The complete visual collection is large "
            "and remains at its canonical paths."
        ),
    },
    {
        "artifact_group": "visual_evidence",
        "artifact_role": (
            "restoration_map_and_raw_image_collections"
        ),
        "producer_notebook_id": "01-29",
        "source_index_path": (
            "outputs/"
            "34_final_streamlit_dashboard_assets/"
            "data/dashboard_indexes/"
            "visual_asset_index.csv"
        ),
        "collection_path": (
            "canonical_paths_recorded_per_index_row"
        ),
        "expected_records": 23964,
        "omission_reason": (
            "Restorations, maps, masks, and source "
            "images are indexed without duplication."
        ),
    },
    {
        "artifact_group": "external_dependencies",
        "artifact_role": (
            "model_weights_and_local_caches"
        ),
        "producer_notebook_id": "30",
        "source_index_path": (
            "outputs/"
            "30_model_cards_compute_and_scalability/"
            "data/model_cards.csv"
        ),
        "collection_path": (
            "external_or_local_cache_not_packaged"
        ),
        "expected_records": 4,
        "omission_reason": (
            "Model weights and caches are excluded "
            "because of size, licensing, and local "
            "cache ownership."
        ),
    },
]

INDEXED_COLLECTION_RECORDS = []

for specification in INDEXED_COLLECTION_SPECS:
    source_index_path = safe_repo_path(
        specification["source_index_path"],
        PROJECT_ROOT,
        must_exist=True,
    )

    source_index = pd.read_csv(
        source_index_path,
        low_memory=False,
    )

    observed_records = len(source_index)

    if (
        observed_records
        != specification["expected_records"]
    ):
        raise ValueError(
            "Indexed collection count mismatch for "
            f"{specification['artifact_role']}: "
            f"expected "
            f"{specification['expected_records']}, "
            f"observed {observed_records}."
        )

    INDEXED_COLLECTION_RECORDS.append(
        {
            "delivery_status": (
                "indexed_not_bundled"
            ),
            "artifact_group": (
                specification["artifact_group"]
            ),
            "artifact_role": (
                specification["artifact_role"]
            ),
            "producer_notebook_id": (
                specification[
                    "producer_notebook_id"
                ]
            ),
            "canonical_source_path": (
                specification[
                    "source_index_path"
                ]
            ),
            "package_relative_path": "",
            "source_index_path": (
                specification[
                    "source_index_path"
                ]
            ),
            "collection_path": (
                specification[
                    "collection_path"
                ]
            ),
            "format": "collection_index",
            "size_bytes": int(
                source_index_path.stat().st_size
            ),
            "sha256": sha256_file(
                source_index_path
            ),
            "source_sha256": sha256_file(
                source_index_path
            ),
            "record_count": int(
                observed_records
            ),
            "source_exists": True,
            "validation_status": "passed",
            "omission_reason": (
                specification[
                    "omission_reason"
                ]
            ),
            "schema_version": (
                ARTIFACT_INDEX_SCHEMA_VERSION
            ),
        }
    )

ARTIFACT_INDEX = pd.DataFrame(
    PACKAGE_ARTIFACT_RECORDS
    + INDEXED_COLLECTION_RECORDS
)

ARTIFACT_INDEX.insert(
    0,
    "artifact_id",
    [
        f"artifact_{number:04d}"
        for number in range(
            1,
            len(ARTIFACT_INDEX) + 1,
        )
    ],
)

ARTIFACT_INDEX = (
    ARTIFACT_INDEX
    .sort_values(
        [
            "delivery_status",
            "artifact_group",
            "artifact_role",
            "package_relative_path",
            "canonical_source_path",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

ARTIFACT_INDEX["artifact_id"] = [
    f"artifact_{number:04d}"
    for number in range(
        1,
        len(ARTIFACT_INDEX) + 1,
    )
]

ARTIFACT_STATUS_COUNTS = {
    str(status): int(count)
    for status, count in (
        ARTIFACT_INDEX[
            "delivery_status"
        ]
        .value_counts()
        .sort_index()
        .items()
    )
}

display(
    ARTIFACT_INDEX.groupby(
        [
            "delivery_status",
            "artifact_group",
        ],
        dropna=False,
    )
    .size()
    .rename("records")
    .reset_index()
)

print("Complete artifact index constructed.")
print("Artifact records:", len(ARTIFACT_INDEX))
print(
    "Delivery-status counts:",
    ARTIFACT_STATUS_COUNTS,
)
print(
    "Physical package files:",
    len(PACKAGE_ARTIFACT_RECORDS),
)
print(
    "Indexed collections:",
    len(INDEXED_COLLECTION_RECORDS),
)

,delivery_status,artifact_group,records
0,copied,application,2
1,copied,configuration,25
2,copied,environment,2
3,copied,manifests,35
4,copied,model_cards,4
5,copied,publication_figures,6
6,copied,reports,6
7,copied,tables,8
8,copied,thesis_figures,18
9,generated_by_notebook_36,data,3


Complete artifact index constructed.
Artifact records: 120
Delivery-status counts: {'copied': 106, 'generated_by_notebook_36': 8, 'indexed_not_bundled': 6}
Physical package files: 114
Indexed collections: 6


In [24]:
ARTIFACT_INDEX_PATH = OUTPUT_PATHS[
    "artifact_index"
]

PACKAGE_MANIFEST_PATH = OUTPUT_PATHS[
    "package_manifest"
]

ARTIFACT_INDEX_TEMPORARY_PATH = (
    ARTIFACT_INDEX_PATH.with_suffix(
        ARTIFACT_INDEX_PATH.suffix + ".tmp"
    )
)

try:
    ARTIFACT_INDEX.to_csv(
        ARTIFACT_INDEX_TEMPORARY_PATH,
        index=False,
        encoding="utf-8",
        lineterminator="\n",
    )
    os.replace(
        ARTIFACT_INDEX_TEMPORARY_PATH,
        ARTIFACT_INDEX_PATH,
    )
finally:
    ARTIFACT_INDEX_TEMPORARY_PATH.unlink(
        missing_ok=True
    )

PACKAGE_FILES_FOR_MANIFEST = (
    package_file_records(
        PACKAGE_ROOT,
        PROJECT_ROOT,
    )
    .sort_values(
        "package_relative_path",
        kind="stable",
    )
    .reset_index(drop=True)
)

PACKAGE_TREE_CHECKSUM = package_tree_checksum(
    PACKAGE_FILES_FOR_MANIFEST
)

PACKAGE_MANIFEST_FILE_RECORDS = (
    json.loads(
        PACKAGE_FILES_FOR_MANIFEST[
            [
                "package_relative_path",
                "format",
                "size_bytes",
                "sha256",
            ]
        ].to_json(
            orient="records"
        )
    )
)

PACKAGE_MANIFEST_PAYLOAD = {
    "schema_version": (
        PACKAGE_MANIFEST_SCHEMA_VERSION
    ),
    "notebook_id": NOTEBOOK_ID,
    "notebook_stem": NOTEBOOK_STEM,
    "generated_at_utc": (
        PACKAGE_MANIFEST_CREATED_AT_UTC
    ),
    "manifest_scope": (
        "physical_files_inside_package_root"
    ),
    "self_included": False,
    "self_exclusion_reason": (
        "The external manifest is excluded to "
        "avoid a recursive checksum definition."
    ),
    "creates_new_scientific_evidence": False,
    "package_root": (
        PACKAGE_ROOT
        .relative_to(PROJECT_ROOT)
        .as_posix()
    ),
    "package_file_count": int(
        len(PACKAGE_FILES_FOR_MANIFEST)
    ),
    "package_size_bytes": int(
        PACKAGE_FILES_FOR_MANIFEST[
            "size_bytes"
        ].sum()
    ),
    "package_size_mib": round(
        (
            PACKAGE_FILES_FOR_MANIFEST[
                "size_bytes"
            ].sum()
            / (1024 ** 2)
        ),
        6,
    ),
    "maximum_package_size_mib": int(
        PACKAGE_POLICY[
            "maximum_package_size_mib"
        ]
    ),
    "package_tree_sha256": (
        PACKAGE_TREE_CHECKSUM
    ),
    "package_group_counts": (
        PACKAGE_GROUP_COUNTS_BATCH_6
    ),
    "artifact_index": {
        "relative_path": (
            ARTIFACT_INDEX_PATH
            .relative_to(PROJECT_ROOT)
            .as_posix()
        ),
        "record_count": int(
            len(ARTIFACT_INDEX)
        ),
        "sha256": sha256_file(
            ARTIFACT_INDEX_PATH
        ),
        "delivery_status_counts": (
            ARTIFACT_STATUS_COUNTS
        ),
    },
    "upstream": {
        "manifest_count": int(
            len(UPSTREAM_MANIFEST_SNAPSHOT)
        ),
        "completion_gates_passed": int(
            UPSTREAM_MANIFEST_SNAPSHOT[
                "completion_gate_passed"
            ].sum()
        ),
        "blocking_failures": int(
            UPSTREAM_MANIFEST_SNAPSHOT[
                "blocking_failures"
            ].sum()
        ),
        "warning_failures": int(
            UPSTREAM_MANIFEST_SNAPSHOT[
                "warning_failures"
            ].sum()
        ),
    },
    "git": CURRENT_GIT_STATE,
    "validation_status": (
        "assembly_complete_pending_"
        "portability_validation"
    ),
    "files": PACKAGE_MANIFEST_FILE_RECORDS,
}

atomic_write_json(
    PACKAGE_MANIFEST_PATH,
    PACKAGE_MANIFEST_PAYLOAD,
)

print("Artifact index and package manifest persisted.")
print(
    "Artifact-index rows:",
    len(ARTIFACT_INDEX),
)
print(
    "Artifact-index SHA-256:",
    sha256_file(ARTIFACT_INDEX_PATH),
)
print(
    "Package-manifest files:",
    PACKAGE_MANIFEST_PAYLOAD[
        "package_file_count"
    ],
)
print(
    "Package tree SHA-256:",
    PACKAGE_MANIFEST_PAYLOAD[
        "package_tree_sha256"
    ],
)

Artifact index and package manifest persisted.
Artifact-index rows: 120
Artifact-index SHA-256: 91b143add9042f9c2b5a0b45f40bdcd6b48841b3076a685767be2dc5c78582de
Package-manifest files: 114
Package tree SHA-256: 69ff56193ea8a56dcc867d52a1799666c74aad960c59f393ba409fbfa75cec8b


In [25]:
RELOADED_ARTIFACT_INDEX = pd.read_csv(
    ARTIFACT_INDEX_PATH,
    low_memory=False,
)

with PACKAGE_MANIFEST_PATH.open(
    "r",
    encoding="utf-8-sig",
) as handle:
    RELOADED_PACKAGE_MANIFEST = json.load(
        handle
    )

OBSERVED_ARTIFACT_STATUS_COUNTS = {
    str(status): int(count)
    for status, count in (
        RELOADED_ARTIFACT_INDEX[
            "delivery_status"
        ]
        .value_counts()
        .sort_index()
        .items()
    )
}

EXPECTED_ARTIFACT_STATUS_COUNTS = {
    "copied": 106,
    "generated_by_notebook_36": 8,
    "indexed_not_bundled": 6,
}

BUNDLED_ARTIFACTS = (
    RELOADED_ARTIFACT_INDEX.loc[
        RELOADED_ARTIFACT_INDEX[
            "delivery_status"
        ].isin(
            [
                "copied",
                "generated_by_notebook_36",
            ]
        )
    ]
    .copy()
)

INDEXED_ARTIFACTS = (
    RELOADED_ARTIFACT_INDEX.loc[
        RELOADED_ARTIFACT_INDEX[
            "delivery_status"
        ].eq("indexed_not_bundled")
    ]
    .copy()
)

OBSERVED_PACKAGE_PATHS = set(
    PACKAGE_FILES_FOR_MANIFEST[
        "package_relative_path"
    ].astype(str)
)

INDEXED_PACKAGE_PATHS = set(
    BUNDLED_ARTIFACTS[
        "package_relative_path"
    ].astype(str)
)

PACKAGE_PATH_COVERAGE_MATCHES = (
    OBSERVED_PACKAGE_PATHS
    == INDEXED_PACKAGE_PATHS
)

PACKAGE_CHECKSUM_BY_PATH = (
    PACKAGE_FILES_FOR_MANIFEST
    .set_index("package_relative_path")[
        "sha256"
    ]
    .astype(str)
    .to_dict()
)

BUNDLED_ARTIFACTS[
    "observed_package_sha256"
] = (
    BUNDLED_ARTIFACTS[
        "package_relative_path"
    ]
    .astype(str)
    .map(PACKAGE_CHECKSUM_BY_PATH)
)

BUNDLED_CHECKSUMS_MATCH = bool(
    BUNDLED_ARTIFACTS["sha256"]
    .astype(str)
    .eq(
        BUNDLED_ARTIFACTS[
            "observed_package_sha256"
        ].astype(str)
    )
    .all()
)

INDEXED_SOURCES_EXIST = bool(
    INDEXED_ARTIFACTS[
        "source_index_path"
    ]
    .astype(str)
    .map(
        lambda value: (
            PROJECT_ROOT / value
        ).is_file()
    )
    .all()
)

MANIFEST_FILE_PATHS = {
    str(record["package_relative_path"])
    for record in (
        RELOADED_PACKAGE_MANIFEST[
            "files"
        ]
    )
}

MANIFEST_FILE_CHECKSUMS = {
    str(record["package_relative_path"]): str(
        record["sha256"]
    )
    for record in (
        RELOADED_PACKAGE_MANIFEST[
            "files"
        ]
    )
}

PACKAGE_MANIFEST_PATHS_MATCH = (
    MANIFEST_FILE_PATHS
    == OBSERVED_PACKAGE_PATHS
)

PACKAGE_MANIFEST_CHECKSUMS_MATCH = (
    MANIFEST_FILE_CHECKSUMS
    == PACKAGE_CHECKSUM_BY_PATH
)

TEMPORARY_FILES_AFTER_BATCH_7 = sorted(
    path.relative_to(OUTPUT_ROOT).as_posix()
    for path in OUTPUT_ROOT.rglob("*")
    if (
        path.is_file()
        and path.name.endswith(".tmp")
    )
)

OUTPUT_FILES_THROUGH_BATCH_7 = sorted(
    path.relative_to(OUTPUT_ROOT).as_posix()
    for path in OUTPUT_ROOT.rglob("*")
    if path.is_file()
)

BATCH_7_CHECKS = (
    (
        "artifact_index_row_count",
        "The complete artifact index contains 120 records",
        120,
        len(RELOADED_ARTIFACT_INDEX),
        len(RELOADED_ARTIFACT_INDEX) == 120,
    ),
    (
        "artifact_status_counts",
        "Artifact delivery statuses match the approved contract",
        EXPECTED_ARTIFACT_STATUS_COUNTS,
        OBSERVED_ARTIFACT_STATUS_COUNTS,
        (
            OBSERVED_ARTIFACT_STATUS_COUNTS
            == EXPECTED_ARTIFACT_STATUS_COUNTS
        ),
    ),
    (
        "artifact_ids_unique",
        "Every artifact-index identifier is unique",
        True,
        bool(
            RELOADED_ARTIFACT_INDEX[
                "artifact_id"
            ].is_unique
        ),
        bool(
            RELOADED_ARTIFACT_INDEX[
                "artifact_id"
            ].is_unique
        ),
    ),
    (
        "artifact_schema",
        "Every artifact row uses the approved schema",
        [
            ARTIFACT_INDEX_SCHEMA_VERSION
        ],
        sorted(
            RELOADED_ARTIFACT_INDEX[
                "schema_version"
            ]
            .astype(str)
            .unique()
            .tolist()
        ),
        (
            set(
                RELOADED_ARTIFACT_INDEX[
                    "schema_version"
                ].astype(str)
            )
            == {
                ARTIFACT_INDEX_SCHEMA_VERSION
            }
        ),
    ),
    (
        "package_path_coverage",
        "Every physical package file appears exactly once in the index",
        True,
        PACKAGE_PATH_COVERAGE_MATCHES,
        PACKAGE_PATH_COVERAGE_MATCHES,
    ),
    (
        "bundled_checksums",
        "Every bundled artifact checksum matches the physical package",
        True,
        BUNDLED_CHECKSUMS_MATCH,
        BUNDLED_CHECKSUMS_MATCH,
    ),
    (
        "indexed_collections",
        "All six intentionally omitted collections have valid source indexes",
        {
            "records": 6,
            "sources_exist": True,
        },
        {
            "records": len(
                INDEXED_ARTIFACTS
            ),
            "sources_exist": (
                INDEXED_SOURCES_EXIST
            ),
        },
        (
            len(INDEXED_ARTIFACTS) == 6
            and INDEXED_SOURCES_EXIST
        ),
    ),
    (
        "package_manifest_schema",
        "The package manifest uses the approved schema",
        PACKAGE_MANIFEST_SCHEMA_VERSION,
        RELOADED_PACKAGE_MANIFEST.get(
            "schema_version"
        ),
        (
            RELOADED_PACKAGE_MANIFEST.get(
                "schema_version"
            )
            == PACKAGE_MANIFEST_SCHEMA_VERSION
        ),
    ),
    (
        "package_manifest_counts",
        "Package manifest counts reconcile with disk",
        {
            "files": 114,
            "bytes": int(
                PACKAGE_FILES_FOR_MANIFEST[
                    "size_bytes"
                ].sum()
            ),
        },
        {
            "files": (
                RELOADED_PACKAGE_MANIFEST[
                    "package_file_count"
                ]
            ),
            "bytes": (
                RELOADED_PACKAGE_MANIFEST[
                    "package_size_bytes"
                ]
            ),
        },
        (
            RELOADED_PACKAGE_MANIFEST[
                "package_file_count"
            ]
            == 114
            and RELOADED_PACKAGE_MANIFEST[
                "package_size_bytes"
            ]
            == int(
                PACKAGE_FILES_FOR_MANIFEST[
                    "size_bytes"
                ].sum()
            )
        ),
    ),
    (
        "package_manifest_paths",
        "Manifest paths match every package file",
        True,
        PACKAGE_MANIFEST_PATHS_MATCH,
        PACKAGE_MANIFEST_PATHS_MATCH,
    ),
    (
        "package_manifest_checksums",
        "Manifest checksums match every package file",
        True,
        PACKAGE_MANIFEST_CHECKSUMS_MATCH,
        PACKAGE_MANIFEST_CHECKSUMS_MATCH,
    ),
    (
        "package_tree_checksum",
        "The package-tree checksum reconciles",
        PACKAGE_TREE_CHECKSUM,
        RELOADED_PACKAGE_MANIFEST[
            "package_tree_sha256"
        ],
        (
            RELOADED_PACKAGE_MANIFEST[
                "package_tree_sha256"
            ]
            == PACKAGE_TREE_CHECKSUM
        ),
    ),
    (
        "artifact_index_checksum",
        "The manifest records the current artifact-index checksum",
        sha256_file(ARTIFACT_INDEX_PATH),
        RELOADED_PACKAGE_MANIFEST[
            "artifact_index"
        ]["sha256"],
        (
            RELOADED_PACKAGE_MANIFEST[
                "artifact_index"
            ]["sha256"]
            == sha256_file(
                ARTIFACT_INDEX_PATH
            )
        ),
    ),
    (
        "temporary_files",
        "No stale temporary files remain after persistence",
        [],
        TEMPORARY_FILES_AFTER_BATCH_7,
        not TEMPORARY_FILES_AFTER_BATCH_7,
    ),
    (
        "output_file_count",
        "Exactly 122 files exist through Batch 7",
        122,
        len(OUTPUT_FILES_THROUGH_BATCH_7),
        len(OUTPUT_FILES_THROUGH_BATCH_7) == 122,
    ),
)

BATCH_7_VALIDATION_STAGE = (
    "batch_7_artifact_index_manifest"
)

RETAINED_VALIDATION_CHECKS = [
    check
    for check in VALIDATION.checks
    if (
        check.validation_stage
        != BATCH_7_VALIDATION_STAGE
    )
]

if (
    len(RETAINED_VALIDATION_CHECKS)
    != len(VALIDATION.checks)
):
    VALIDATION = ValidationCollector()
    VALIDATION.extend(
        RETAINED_VALIDATION_CHECKS
    )

for (
    check_id,
    description,
    expected,
    observed,
    passed,
) in BATCH_7_CHECKS:
    VALIDATION.add(
        validation_stage=(
            BATCH_7_VALIDATION_STAGE
        ),
        check_id=check_id,
        check_description=description,
        severity="blocking",
        expected=expected,
        observed=observed,
        passed=bool(passed),
        details=(
            ""
            if passed
            else (
                "The artifact index or package "
                "manifest does not reconcile."
            )
        ),
    )

VALIDATION.raise_for_blocking()

BATCH_7_VALIDATION = VALIDATION.to_dataframe()

BATCH_7_STAGE_SUMMARY = (
    BATCH_7_VALIDATION
    .groupby(
        ["validation_stage", "severity"],
        dropna=False,
    )
    .agg(
        checks=("check_id", "size"),
        passed=("passed", "sum"),
    )
    .reset_index()
)

BATCH_7_STAGE_SUMMARY["failed"] = (
    BATCH_7_STAGE_SUMMARY["checks"]
    - BATCH_7_STAGE_SUMMARY["passed"]
)

display(BATCH_7_STAGE_SUMMARY)
display(
    RELOADED_ARTIFACT_INDEX.groupby(
        "delivery_status",
        dropna=False,
    )
    .size()
    .rename("records")
    .reset_index()
)

print("Batch 7 artifact indexing passed.")
print(
    "Cumulative validation checks:",
    len(BATCH_7_VALIDATION),
)
print(
    "Blocking failures:",
    len(VALIDATION.blocking_failures),
)
print(
    "Artifact-index records:",
    len(RELOADED_ARTIFACT_INDEX),
)
print(
    "Package-manifest file records:",
    len(
        RELOADED_PACKAGE_MANIFEST[
            "files"
        ]
    ),
)
print(
    "Canonical files through Batch 7:",
    len(OUTPUT_FILES_THROUGH_BATCH_7),
)

,validation_stage,severity,checks,passed,failed
0,batch_1_contract,blocking,11,11,0
1,batch_1_dependencies_sources,blocking,9,9,0
2,batch_1_final_preflight,blocking,1,1,0
3,batch_1_inventory_governance,blocking,15,15,0
4,batch_2_artifact_registry,blocking,10,10,0
5,batch_2_final_audit,blocking,1,1,0
6,batch_2_upstream_manifests,blocking,14,14,0
7,batch_3_evidence_inputs,blocking,9,9,0
8,batch_3_final_synthesis,blocking,1,1,0
9,batch_3_model_synthesis,blocking,8,8,0


,delivery_status,records
0,copied,106
1,generated_by_notebook_36,8
2,indexed_not_bundled,6


Batch 7 artifact indexing passed.
Cumulative validation checks: 142
Blocking failures: 0
Artifact-index records: 120
Package-manifest file records: 114
Canonical files through Batch 7: 122


## Batch 8 — Portability and integrity validation

This batch:

- corrects package-local image links in the generated supervisor summary;
- preserves the canonical supervisor report and all upstream files;
- refreshes the artifact index and package manifest after that adjustment;
- verifies every package checksum and file format;
- checks Markdown links, embedded HTML resources, PNG files, path lengths,
  package size, and temporary-file cleanup;
- preserves the upstream deployment warning.

No restoration inference, metric computation, or notebook execution is performed.

In [26]:
PACKAGE_SUMMARY_PATH = (
    PACKAGE_ROOT / "reports" / "supervisor_summary.md"
)

CANONICAL_SUMMARY_TEXT = SUPERVISOR_SUMMARY_PATH.read_text(
    encoding="utf-8"
)

OLD_FIGURE_PREFIX = "../package/figures/"
NEW_FIGURE_PREFIX = "../figures/"

SUMMARY_LINK_ADJUSTMENT_COUNT = (
    CANONICAL_SUMMARY_TEXT.count(OLD_FIGURE_PREFIX)
)

if SUMMARY_LINK_ADJUSTMENT_COUNT != 6:
    raise ValueError(
        "Expected six canonical supervisor-summary figure links; "
        f"found {SUMMARY_LINK_ADJUSTMENT_COUNT}."
    )

PORTABLE_SUMMARY_TEXT = CANONICAL_SUMMARY_TEXT.replace(
    OLD_FIGURE_PREFIX,
    NEW_FIGURE_PREFIX,
)

CURRENT_PACKAGE_SUMMARY_TEXT = PACKAGE_SUMMARY_PATH.read_text(
    encoding="utf-8"
)

if CURRENT_PACKAGE_SUMMARY_TEXT not in {
    CANONICAL_SUMMARY_TEXT,
    PORTABLE_SUMMARY_TEXT,
}:
    raise ValueError(
        "The packaged supervisor summary contains unexpected changes. "
        "It has not been overwritten."
    )

ARTIFACT_INDEX = pd.read_csv(
    ARTIFACT_INDEX_PATH,
    keep_default_na=False,
    low_memory=False,
)

SUMMARY_INDEX_MASK = ARTIFACT_INDEX[
    "package_relative_path"
].eq("reports/supervisor_summary.md")

if int(SUMMARY_INDEX_MASK.sum()) != 1:
    raise ValueError(
        "Expected exactly one packaged supervisor-summary index row."
    )

CANONICAL_SUMMARY_SHA256 = sha256_file(
    SUPERVISOR_SUMMARY_PATH
)

RECORDED_SUMMARY_SOURCE_SHA256 = str(
    ARTIFACT_INDEX.loc[
        SUMMARY_INDEX_MASK,
        "source_sha256",
    ].iloc[0]
)

if (
    CANONICAL_SUMMARY_SHA256
    != RECORDED_SUMMARY_SOURCE_SHA256
):
    raise ValueError(
        "The canonical supervisor summary changed after indexing. "
        "Review that source change before continuing."
    )

atomic_write_text(
    PACKAGE_SUMMARY_PATH,
    PORTABLE_SUMMARY_TEXT,
)

ARTIFACT_INDEX.loc[
    SUMMARY_INDEX_MASK,
    "sha256",
] = sha256_file(PACKAGE_SUMMARY_PATH)

ARTIFACT_INDEX.loc[
    SUMMARY_INDEX_MASK,
    "size_bytes",
] = PACKAGE_SUMMARY_PATH.stat().st_size

# This row remains generated_by_notebook_36.
# source_sha256 continues to identify the unchanged canonical report.

ARTIFACT_INDEX_TEMPORARY_PATH = (
    ARTIFACT_INDEX_PATH.with_suffix(".csv.tmp")
)

try:
    ARTIFACT_INDEX.to_csv(
        ARTIFACT_INDEX_TEMPORARY_PATH,
        index=False,
        encoding="utf-8",
        lineterminator="\n",
    )
    os.replace(
        ARTIFACT_INDEX_TEMPORARY_PATH,
        ARTIFACT_INDEX_PATH,
    )
finally:
    ARTIFACT_INDEX_TEMPORARY_PATH.unlink(
        missing_ok=True
    )

with PACKAGE_MANIFEST_PATH.open(
    "r",
    encoding="utf-8-sig",
) as handle:
    PACKAGE_MANIFEST_PAYLOAD = json.load(handle)

PACKAGE_FILES_FOR_MANIFEST = package_file_records(
    PACKAGE_ROOT,
    PROJECT_ROOT,
)

PACKAGE_TREE_CHECKSUM = package_tree_checksum(
    PACKAGE_FILES_FOR_MANIFEST
)

PACKAGE_MANIFEST_PAYLOAD.update(
    {
        "package_file_count": int(
            len(PACKAGE_FILES_FOR_MANIFEST)
        ),
        "package_size_bytes": int(
            PACKAGE_FILES_FOR_MANIFEST["size_bytes"].sum()
        ),
        "package_size_mib": round(
            PACKAGE_FILES_FOR_MANIFEST["size_bytes"].sum()
            / (1024 ** 2),
            6,
        ),
        "package_tree_sha256": PACKAGE_TREE_CHECKSUM,
        "validation_status": "pending_portability_validation",
        "files": json.loads(
            PACKAGE_FILES_FOR_MANIFEST[
                [
                    "package_relative_path",
                    "format",
                    "size_bytes",
                    "sha256",
                ]
            ].to_json(orient="records")
        ),
        "presentation_adjustments": [
            {
                "package_relative_path": (
                    "reports/supervisor_summary.md"
                ),
                "adjustment": (
                    "Rebase generated-summary figure links "
                    "for the package/reports location."
                ),
                "old_prefix": OLD_FIGURE_PREFIX,
                "new_prefix": NEW_FIGURE_PREFIX,
                "adjusted_link_count": (
                    SUMMARY_LINK_ADJUSTMENT_COUNT
                ),
                "canonical_source_sha256": (
                    CANONICAL_SUMMARY_SHA256
                ),
                "package_sha256": sha256_file(
                    PACKAGE_SUMMARY_PATH
                ),
                "canonical_source_modified": False,
            }
        ],
    }
)

PACKAGE_MANIFEST_PAYLOAD["artifact_index"]["sha256"] = (
    sha256_file(ARTIFACT_INDEX_PATH)
)

atomic_write_json(
    PACKAGE_MANIFEST_PATH,
    PACKAGE_MANIFEST_PAYLOAD,
)

print("Package-local supervisor-summary links corrected.")
print("Adjusted figure links:", SUMMARY_LINK_ADJUSTMENT_COUNT)
print("Canonical supervisor report modified: False")
print("Upstream files modified: False")
print("Artifact index and package checksums refreshed.")
print("Package files:", len(PACKAGE_FILES_FOR_MANIFEST))

Package-local supervisor-summary links corrected.
Adjusted figure links: 6
Canonical supervisor report modified: False
Upstream files modified: False
Artifact index and package checksums refreshed.
Package files: 114


In [27]:
import re
from urllib.parse import unquote, urlsplit

from PIL import Image

from restoration_eval.supervisor_package import (
    audit_self_contained_html,
)


PACKAGE_FILES_BATCH_8 = package_file_records(
    PACKAGE_ROOT,
    PROJECT_ROOT,
)

with PACKAGE_MANIFEST_PATH.open(
    "r",
    encoding="utf-8-sig",
) as handle:
    RELOADED_PACKAGE_MANIFEST = json.load(handle)

MANIFEST_RECORDS_BY_PATH = {
    record["package_relative_path"]: record
    for record in RELOADED_PACKAGE_MANIFEST["files"]
}

MARKDOWN_LINK_PATTERN = re.compile(
    r"!?\[[^\]]*\]\(([^)]+)\)"
)

EXTERNAL_HTML_RESOURCE_PATTERN = re.compile(
    r"""<(?:img|script|link|iframe|source)\b"""
    r"""[^>]*(?:src|href)\s*=\s*["'](?:https?:)?//""",
    flags=re.IGNORECASE,
)

FILE_PORTABILITY_RECORDS = []
MARKDOWN_LINK_RECORDS = []
HTML_PORTABILITY_RECORDS = []

for number, row in enumerate(
    PACKAGE_FILES_BATCH_8.itertuples(index=False),
    start=1,
):
    path = PROJECT_ROOT / row.relative_path
    package_relative_path = str(
        row.package_relative_path
    )
    suffix = path.suffix.lower()

    read_error = ""

    try:
        if suffix == ".png":
            with Image.open(path) as picture:
                picture.verify()

        elif suffix == ".csv":
            pd.read_csv(path, low_memory=False)

        elif suffix == ".json":
            with path.open(
                "r",
                encoding="utf-8-sig",
            ) as handle:
                json.load(handle)

        elif suffix in {".yaml", ".yml"}:
            with path.open(
                "r",
                encoding="utf-8-sig",
            ) as handle:
                yaml.safe_load(handle)

        elif suffix == ".py":
            ast.parse(
                path.read_text(encoding="utf-8-sig"),
                filename=package_relative_path,
            )

        elif suffix == ".html":
            html_text = path.read_text(
                encoding="utf-8"
            )
            html_audit = audit_self_contained_html(
                path
            )

            external_resource_count = len(
                EXTERNAL_HTML_RESOURCE_PATTERN.findall(
                    html_text
                )
            )

            HTML_PORTABILITY_RECORDS.append(
                {
                    "package_relative_path": (
                        package_relative_path
                    ),
                    "embedded_images": int(
                        html_audit[
                            "embedded_data_uri_count"
                        ]
                    ),
                    "local_references": int(
                        html_audit[
                            "local_reference_count"
                        ]
                    ),
                    "external_resources": (
                        external_resource_count
                    ),
                    "passed": (
                        html_audit["self_contained"]
                        and external_resource_count == 0
                        and html_audit[
                            "embedded_data_uri_count"
                        ] > 0
                    ),
                }
            )

        elif suffix == ".md":
            markdown_text = path.read_text(
                encoding="utf-8"
            )

            for raw_target in (
                MARKDOWN_LINK_PATTERN.findall(
                    markdown_text
                )
            ):
                target = raw_target.strip().strip("<>")
                parsed = urlsplit(target)

                if (
                    parsed.scheme
                    in {"http", "https", "mailto"}
                    or not parsed.path
                ):
                    continue

                resolved_target = (
                    path.parent
                    / unquote(parsed.path)
                ).resolve()

                inside_package = (
                    resolved_target.is_relative_to(
                        PACKAGE_ROOT.resolve()
                    )
                )

                MARKDOWN_LINK_RECORDS.append(
                    {
                        "document": package_relative_path,
                        "link": target,
                        "inside_package": inside_package,
                        "target_exists": (
                            resolved_target.is_file()
                        ),
                        "passed": (
                            inside_package
                            and resolved_target.is_file()
                        ),
                    }
                )

        elif suffix == ".txt":
            path.read_text(encoding="utf-8-sig")

        else:
            raise ValueError(
                f"Undeclared package format: {suffix}"
            )

    except Exception as exc:
        read_error = (
            f"{type(exc).__name__}: {exc}"
        )

    expected_record = MANIFEST_RECORDS_BY_PATH.get(
        package_relative_path,
        {},
    )

    FILE_PORTABILITY_RECORDS.append(
        {
            "package_relative_path": package_relative_path,
            "format": suffix.lstrip("."),
            "size_bytes": int(row.size_bytes),
            "sha256": str(row.sha256),
            "manifest_matches": (
                expected_record.get("sha256")
                == str(row.sha256)
                and expected_record.get("size_bytes")
                == int(row.size_bytes)
            ),
            "readable": not read_error,
            "error": read_error,
        }
    )

    if (
        number % 10 == 0
        or number == len(PACKAGE_FILES_BATCH_8)
    ):
        print(
            f"Inspected {number}/"
            f"{len(PACKAGE_FILES_BATCH_8)} files"
        )

FILE_PORTABILITY_AUDIT = pd.DataFrame(
    FILE_PORTABILITY_RECORDS
)

MARKDOWN_LINK_AUDIT = pd.DataFrame(
    MARKDOWN_LINK_RECORDS,
    columns=[
        "document",
        "link",
        "inside_package",
        "target_exists",
        "passed",
    ],
)

HTML_PORTABILITY_AUDIT = pd.DataFrame(
    HTML_PORTABILITY_RECORDS,
    columns=[
        "package_relative_path",
        "embedded_images",
        "local_references",
        "external_resources",
        "passed",
    ],
)

FIXED_COPY_MISMATCHES_BATCH_8 = []

for row in COPY_PLAN_TABLE.itertuples(
    index=False
):
    source = PROJECT_ROOT / row.source_path
    destination = (
        PROJECT_ROOT / row.destination_path
    )

    if (
        sha256_file(source) != row.source_sha256
        or sha256_file(destination)
        != row.source_sha256
    ):
        FIXED_COPY_MISMATCHES_BATCH_8.append(
            str(row.destination_path)
        )

GENERATED_DOCUMENT_MISMATCHES = []

for source, destination in (
    PACKAGE_GENERATED_COPY_MAP.items()
):
    if destination == PACKAGE_SUMMARY_PATH:
        matches = (
            destination.read_text(encoding="utf-8")
            == PORTABLE_SUMMARY_TEXT
        )
    else:
        matches = (
            sha256_file(source)
            == sha256_file(destination)
        )

    if not matches:
        GENERATED_DOCUMENT_MISMATCHES.append(
            destination
            .relative_to(PACKAGE_ROOT)
            .as_posix()
        )

PACKAGE_FORMAT_COUNTS = {
    str(kind): int(count)
    for kind, count in (
        FILE_PORTABILITY_AUDIT["format"]
        .value_counts()
        .sort_index()
        .items()
    )
}

EXPECTED_PACKAGE_FORMAT_COUNTS = {
    "csv": 8,
    "html": 5,
    "json": 37,
    "md": 11,
    "png": 24,
    "py": 2,
    "txt": 2,
    "yaml": 25,
}

PACKAGE_SIZE_BYTES_BATCH_8 = int(
    PACKAGE_FILES_BATCH_8["size_bytes"].sum()
)

MAXIMUM_PACKAGE_PATH_LENGTH = int(
    PACKAGE_FILES_BATCH_8["relative_path"]
    .astype(str)
    .str.len()
    .max()
)

OUTPUT_FILES_THROUGH_BATCH_8 = [
    path
    for path in OUTPUT_ROOT.rglob("*")
    if path.is_file()
]

TEMPORARY_FILES_BATCH_8 = [
    path.relative_to(OUTPUT_ROOT).as_posix()
    for path in OUTPUT_FILES_THROUGH_BATCH_8
    if path.name.endswith(".tmp")
]

display(HTML_PORTABILITY_AUDIT)
display(MARKDOWN_LINK_AUDIT)

display(
    FILE_PORTABILITY_AUDIT.loc[
        ~FILE_PORTABILITY_AUDIT["readable"]
        | ~FILE_PORTABILITY_AUDIT["manifest_matches"]
    ]
)

print("Package inspection completed.")
print("Format counts:", PACKAGE_FORMAT_COUNTS)
print("Markdown links inspected:", len(MARKDOWN_LINK_AUDIT))
print("HTML reports inspected:", len(HTML_PORTABILITY_AUDIT))
print("Fixed-copy mismatches:", len(FIXED_COPY_MISMATCHES_BATCH_8))
print(
    "Generated-document mismatches:",
    len(GENERATED_DOCUMENT_MISMATCHES),
)

Inspected 10/114 files
Inspected 20/114 files
Inspected 30/114 files
Inspected 40/114 files
Inspected 50/114 files
Inspected 60/114 files
Inspected 70/114 files
Inspected 80/114 files
Inspected 90/114 files
Inspected 100/114 files
Inspected 110/114 files
Inspected 114/114 files


,package_relative_path,embedded_images,local_references,external_resources,passed
0,reports/final_evaluation.html,68,0,0,True
1,reports/models/lama.html,16,0,0,True
2,reports/models/opencv_telea.html,16,0,0,True
3,reports/models/sdxl_inpainting.html,12,0,0,True
4,reports/models/stable_diffusion_inpainting.html,19,0,0,True


,document,link,inside_package,target_exists,passed
0,README.md,reports/supervisor_summary.md,True,True,True
1,README.md,reports/final_evaluation.html,True,True,True
2,README.md,reports/limitations_and_deviations.md,True,True,True
3,README.md,provenance/reproducibility_snapshot.json,True,True,True
4,README.md,reports/supervisor_summary.md,True,True,True
5,README.md,reports/final_evaluation.html,True,True,True
6,README.md,reports/models/lama.html,True,True,True
7,README.md,reports/models/opencv_telea.html,True,True,True
8,README.md,reports/models/stable_diffusion_inpainting.html,True,True,True
9,README.md,reports/models/sdxl_inpainting.html,True,True,True


,package_relative_path,format,size_bytes,sha256,manifest_matches,readable,error


Package inspection completed.
Format counts: {'csv': 8, 'html': 5, 'json': 37, 'md': 11, 'png': 24, 'py': 2, 'txt': 2, 'yaml': 25}
Markdown links inspected: 22
HTML reports inspected: 5
Fixed-copy mismatches: 0
Generated-document mismatches: 0


In [28]:
BATCH_8_CHECKS = [
    (
        "file_count",
        "All 114 physical package files were inspected",
        114,
        len(FILE_PORTABILITY_AUDIT),
        len(FILE_PORTABILITY_AUDIT) == 114,
    ),
    (
        "format_coverage",
        "Package formats match the approved assembly",
        EXPECTED_PACKAGE_FORMAT_COUNTS,
        PACKAGE_FORMAT_COUNTS,
        PACKAGE_FORMAT_COUNTS
        == EXPECTED_PACKAGE_FORMAT_COUNTS,
    ),
    (
        "file_readability",
        "Every package file opens or parses successfully",
        [],
        FILE_PORTABILITY_AUDIT.loc[
            ~FILE_PORTABILITY_AUDIT["readable"],
            ["package_relative_path", "error"],
        ].to_dict(orient="records"),
        bool(FILE_PORTABILITY_AUDIT["readable"].all()),
    ),
    (
        "manifest_integrity",
        "Every physical file matches its manifest checksum and size",
        True,
        bool(
            FILE_PORTABILITY_AUDIT[
                "manifest_matches"
            ].all()
        ),
        bool(
            FILE_PORTABILITY_AUDIT[
                "manifest_matches"
            ].all()
        ),
    ),
    (
        "fixed_copy_integrity",
        "All 106 fixed-copy sources and destinations remain unchanged",
        [],
        FIXED_COPY_MISMATCHES_BATCH_8,
        not FIXED_COPY_MISMATCHES_BATCH_8,
    ),
    (
        "generated_document_integrity",
        "Generated documents retain their declared content and link adjustment",
        [],
        GENERATED_DOCUMENT_MISMATCHES,
        not GENERATED_DOCUMENT_MISMATCHES,
    ),
    (
        "markdown_links",
        "All local Markdown links resolve inside the package",
        [],
        MARKDOWN_LINK_AUDIT.loc[
            ~MARKDOWN_LINK_AUDIT["passed"],
            ["document", "link"],
        ].to_dict(orient="records"),
        (
            len(MARKDOWN_LINK_AUDIT) > 0
            and bool(
                MARKDOWN_LINK_AUDIT["passed"].all()
            )
        ),
    ),
    (
        "html_portability",
        "All five HTML reports embed images and require no external resources",
        5,
        int(
            HTML_PORTABILITY_AUDIT["passed"].sum()
        ),
        (
            len(HTML_PORTABILITY_AUDIT) == 5
            and bool(
                HTML_PORTABILITY_AUDIT["passed"].all()
            )
        ),
    ),
    (
        "package_size",
        "The package remains below the configured size ceiling",
        MAXIMUM_PACKAGE_SIZE_BYTES,
        PACKAGE_SIZE_BYTES_BATCH_8,
        (
            PACKAGE_SIZE_BYTES_BATCH_8
            <= MAXIMUM_PACKAGE_SIZE_BYTES
        ),
    ),
    (
        "path_length",
        "Package paths satisfy the configured length limit",
        PACKAGE_POLICY[
            "maximum_relative_path_characters"
        ],
        MAXIMUM_PACKAGE_PATH_LENGTH,
        (
            MAXIMUM_PACKAGE_PATH_LENGTH
            <= int(
                PACKAGE_POLICY[
                    "maximum_relative_path_characters"
                ]
            )
        ),
    ),
    (
        "case_insensitive_paths",
        "Package paths remain unique on Windows",
        True,
        bool(
            PACKAGE_FILES_BATCH_8[
                "package_relative_path"
            ]
            .str.casefold()
            .is_unique
        ),
        bool(
            PACKAGE_FILES_BATCH_8[
                "package_relative_path"
            ]
            .str.casefold()
            .is_unique
        ),
    ),
    (
        "temporary_files",
        "No temporary files remain",
        [],
        TEMPORARY_FILES_BATCH_8,
        not TEMPORARY_FILES_BATCH_8,
    ),
    (
        "output_file_count",
        "Portability validation adds no extra output files",
        122,
        len(OUTPUT_FILES_THROUGH_BATCH_8),
        len(OUTPUT_FILES_THROUGH_BATCH_8) == 122,
    ),
    (
        "canonical_summary_preserved",
        "The canonical supervisor report remains unchanged",
        CANONICAL_SUMMARY_SHA256,
        sha256_file(SUPERVISOR_SUMMARY_PATH),
        (
            sha256_file(SUPERVISOR_SUMMARY_PATH)
            == CANONICAL_SUMMARY_SHA256
        ),
    ),
]

BATCH_8_VALIDATION_STAGE = (
    "batch_8_portability_integrity"
)

RETAINED_VALIDATION_CHECKS = [
    check
    for check in VALIDATION.checks
    if check.validation_stage
    != BATCH_8_VALIDATION_STAGE
]

UPDATED_VALIDATION = ValidationCollector()
UPDATED_VALIDATION.extend(RETAINED_VALIDATION_CHECKS)

for (
    check_id,
    description,
    expected,
    observed,
    passed,
) in BATCH_8_CHECKS:
    UPDATED_VALIDATION.add(
        validation_stage=BATCH_8_VALIDATION_STAGE,
        check_id=check_id,
        check_description=description,
        severity="blocking",
        expected=expected,
        observed=observed,
        passed=bool(passed),
        details=(
            ""
            if passed
            else "Inspect the Batch 8 audit tables."
        ),
    )

VALIDATION = UPDATED_VALIDATION

BATCH_8_VALIDATION = VALIDATION.to_dataframe()

display(
    BATCH_8_VALIDATION.loc[
        BATCH_8_VALIDATION["validation_stage"].eq(
            BATCH_8_VALIDATION_STAGE
        ),
        ["check_id", "passed", "observed"],
    ]
)

VALIDATION.raise_for_blocking()

UPSTREAM_WARNING_COUNT = int(
    PACKAGE_MANIFEST_PAYLOAD["upstream"][
        "warning_failures"
    ]
)

PACKAGE_MANIFEST_PAYLOAD["validation_status"] = (
    "passed_with_upstream_warning"
    if UPSTREAM_WARNING_COUNT
    else "passed"
)

PACKAGE_MANIFEST_PAYLOAD["portability_validation"] = {
    "validated_at_utc": (
        datetime.now(timezone.utc)
        .isoformat(timespec="seconds")
        .replace("+00:00", "Z")
    ),
    "check_count": len(BATCH_8_CHECKS),
    "blocking_failures": 0,
    "inspected_package_files": len(
        FILE_PORTABILITY_AUDIT
    ),
    "self_contained_html_reports": len(
        HTML_PORTABILITY_AUDIT
    ),
    "validated_local_markdown_links": len(
        MARKDOWN_LINK_AUDIT
    ),
    "validated_png_files": int(
        PACKAGE_FORMAT_COUNTS["png"]
    ),
    "canonical_sources_modified": False,
    "upstream_warning_count": UPSTREAM_WARNING_COUNT,
    "dashboard_scope": "local_demonstration_only",
    "public_deployment_completed": False,
}

atomic_write_json(
    PACKAGE_MANIFEST_PATH,
    PACKAGE_MANIFEST_PAYLOAD,
)

with PACKAGE_MANIFEST_PATH.open(
    "r",
    encoding="utf-8-sig",
) as handle:
    RELOADED_PACKAGE_MANIFEST = json.load(handle)

print("Batch 8 portability and integrity passed.")
print("Cumulative checks:", len(BATCH_8_VALIDATION))
print("Blocking failures:", len(VALIDATION.blocking_failures))
print(
    "Package status:",
    RELOADED_PACKAGE_MANIFEST["validation_status"],
)
print("Upstream warnings preserved:", UPSTREAM_WARNING_COUNT)
print("Package files:", len(PACKAGE_FILES_BATCH_8))
print(
    "Package size MiB:",
    round(PACKAGE_SIZE_BYTES_BATCH_8 / (1024 ** 2), 3),
)
print("Next: Batch 9 — final completion and manifests.")

,check_id,passed,observed
142,file_count,True,114
143,format_coverage,True,"{""csv"": 8, ""html"": 5, ""json"": 37, ""md"": 11, ""p..."
144,file_readability,True,[]
145,manifest_integrity,True,True
146,fixed_copy_integrity,True,[]
147,generated_document_integrity,True,[]
148,markdown_links,True,[]
149,html_portability,True,5
150,package_size,True,29021450
151,path_length,True,141


Batch 8 portability and integrity passed.
Cumulative checks: 156
Blocking failures: 0
Package status: passed_with_upstream_warning
Upstream warnings preserved: 1
Package files: 114
Package size MiB: 27.677
Next: Batch 9 — final completion and manifests.


## Batch 9 — Final manifests and completion gate

This final batch:

- maps the roadmap responsibilities to validated evidence;
- persists consolidated validation and 12 canonical artifact records;
- records inputs, configurations, environment, Git state, and output checksums;
- verifies the exact final output set;
- completes the run manifest only after the completion checks pass;
- independently reloads and verifies the final files.

The inherited Notebook 35 deployment warning remains explicit.
No upstream notebook, scientific output, or global inventory is modified here.

In [29]:
from restoration_eval.manifests import (
    MANIFESTS_MODULE_VERSION,
    artifact_records_dataframe,
    build_artifact_record,
    build_run_manifest,
    sha256_path,
    write_artifact_manifest,
    write_run_manifest,
)


def replace_n36_validation_stage(
    stage,
    checks,
    severity="blocking",
):
    replacement = ValidationCollector()

    replacement.extend(
        check
        for check in VALIDATION.checks
        if check.validation_stage != stage
    )

    for key, description, expected, observed, passed in checks:
        replacement.add(
            validation_stage=stage,
            check_id=key,
            check_description=description,
            severity=severity,
            expected=expected,
            observed=observed,
            passed=bool(passed),
            details="" if passed else str(observed),
        )

    return replacement


# Allow the entire final batch to be rerun safely.
N36_RETAINED_CHECKS = [
    check
    for check in VALIDATION.checks
    if not check.validation_stage.startswith("batch_9_")
]

VALIDATION = ValidationCollector()
VALIDATION.extend(N36_RETAINED_CHECKS)
VALIDATION.raise_for_blocking()


def n36_stage_passed(stage):
    checks = [
        check
        for check in VALIDATION.checks
        if check.validation_stage == stage
    ]
    return bool(checks) and all(
        check.passed
        for check in checks
        if check.severity in {"blocking", "error"}
    )


N36_ARTIFACT_SCHEMAS = {
    "supervisor_summary": "supervisor_summary.v1",
    "reproducibility_appendix": "reproducibility_appendix.v1",
    "limitations_and_deviations": "limitations_and_deviations.v1",
    "artifact_index": "supervisor_artifact_index.v1",
    "key_findings": "supervisor_key_findings.v1",
    "open_questions": "supervisor_open_questions.v1",
    "feedback_agenda": "supervisor_feedback_agenda.v1",
    "package_root": "supervisor_review_package.v1",
    "package_readme": "supervisor_package_readme.v1",
    "reproducibility_snapshot": "reproducibility_snapshot.v1",
    "package_manifest": "supervisor_package_manifest.v1",
    "validation": "validation_checks.v1",
}

with PACKAGE_MANIFEST_PATH.open(
    "r",
    encoding="utf-8-sig",
) as handle:
    FINAL_PACKAGE_MANIFEST = json.load(handle)

FINAL_ARTIFACT_INDEX = pd.read_csv(
    ARTIFACT_INDEX_PATH,
    keep_default_na=False,
)

EXPECTED_PHYSICAL_OUTPUTS = {
    (
        PACKAGE_ROOT / record["package_relative_path"]
    ).resolve()
    for record in FINAL_PACKAGE_MANIFEST["files"]
}

EXPECTED_PHYSICAL_OUTPUTS.update(
    path.resolve()
    for key, path in OUTPUT_PATHS.items()
    if key != "package_root"
)

if len(EXPECTED_PHYSICAL_OUTPUTS) != 125:
    raise ValueError(
        "The final output contract should contain 125 physical files; "
        f"found {len(EXPECTED_PHYSICAL_OUTPUTS)}."
    )

TRACEABILITY_SPECS = [
    (
        "Supervisor summary and exact proposal questions",
        "Three exact RQs and ten summary sections",
        n36_stage_passed("batch_4_supervisor_documents"),
    ),
    (
        "Final HTML and four model reports",
        "Five self-contained HTML reports",
        len(HTML_PORTABILITY_AUDIT) == 5
        and HTML_PORTABILITY_AUDIT["passed"].all(),
    ),
    (
        "Compact evidence and LaTeX-ready tables",
        "Eight tables, including latex_tables.csv",
        len(list((PACKAGE_ROOT / "tables").glob("*.csv"))) == 8
        and (PACKAGE_ROOT / "tables/latex_tables.csv").is_file(),
    ),
    (
        "Thesis-ready and publication-ready figures",
        "18 thesis figures and six publication figures",
        len(list((PACKAGE_ROOT / "figures/thesis").glob("*.png"))) == 18
        and len(list(
            (PACKAGE_ROOT / "figures/publication").glob("*.png")
        )) == 6,
    ),
    (
        "Model cards and model identities",
        "Four cards, implementations, revisions, and configurations",
        len(MODEL_REPRODUCIBILITY) == 4
        and len(list(
            (PACKAGE_ROOT / "model_cards").glob("*.md")
        )) == 4,
    ),
    (
        "Case and painting report discovery",
        "30 case-report and 50 painting-report records",
        len(CASE_REPORT_INDEX) == 30
        and len(PAINTING_REPORT_INDEX) == 50,
    ),
    (
        "All upstream completion manifests",
        "35 gates passed; zero upstream blocking failures",
        len(UPSTREAM_MANIFEST_SNAPSHOT) == 35
        and UPSTREAM_MANIFEST_SNAPSHOT[
            "completion_gate_passed"
        ].all()
        and int(UPSTREAM_MANIFEST_SNAPSHOT[
            "blocking_failures"
        ].sum()) == 0,
    ),
    (
        "Configuration and dataset traceability",
        "25 bundled evaluation and 13 indexed source configurations",
        len(CONFIGURATION_SNAPSHOT) == 25
        and len(SOURCE_CONFIGURATION_SNAPSHOT) == 13
        and DATASET_IDENTITY["expected_paintings"] == 50,
    ),
    (
        "Environment, Git, hardware, and seed provenance",
        "20 package versions, two requirements files, six seed policies",
        len(PACKAGE_VERSION_SNAPSHOT) == 20
        and len(REQUIREMENT_SNAPSHOT) == 2
        and len(SEED_POLICY_SNAPSHOT) == 6
        and not CURRENT_GIT_STATE["git_error"],
    ),
    (
        "Observed compute and projection boundaries",
        "Four observed model summaries; projections explicitly labelled",
        len(COMPUTE_REPRO_TABLE) == 4
        and n36_stage_passed(
            "batch_5_reproducibility_limitations"
        ),
    ),
    (
        "Reproducibility appendix and limitations",
        "13 appendix sections and 12 limitations sections",
        n36_stage_passed(
            "batch_5_reproducibility_limitations"
        ),
    ),
    (
        "Meeting documents and validated findings",
        "Key findings, open questions, and feedback agenda",
        n36_stage_passed("batch_4_supervisor_documents"),
    ),
    (
        "Complete delivery index and intentional omissions",
        "114 package files and six indexed collections",
        len(FINAL_ARTIFACT_INDEX) == 120
        and FINAL_ARTIFACT_INDEX["delivery_status"]
        .eq("indexed_not_bundled").sum() == 6,
    ),
    (
        "Fixed-copy integrity and documented presentation adjustment",
        "106 preserved copies; six rebased summary links",
        not FIXED_COPY_MISMATCHES_BATCH_8
        and not GENERATED_DOCUMENT_MISMATCHES,
    ),
    (
        "Portability, formats, paths, and package size",
        "114 readable files, 22 valid links, and 24 validated PNGs",
        n36_stage_passed("batch_8_portability_integrity"),
    ),
    (
        "Honest dashboard and scientific scope",
        "Local demonstration only; no new scientific evidence",
        FINAL_PACKAGE_MANIFEST[
            "portability_validation"
        ]["public_deployment_completed"] is False
        and FINAL_PACKAGE_MANIFEST[
            "creates_new_scientific_evidence"
        ] is False,
    ),
]

ROADMAP_TRACEABILITY = pd.DataFrame(
    [
        {
            "responsibility_id": f"n36_r{number:02d}",
            "responsibility": responsibility,
            "evidence": evidence,
            "passed": bool(passed),
        }
        for number, (
            responsibility,
            evidence,
            passed,
        ) in enumerate(TRACEABILITY_SPECS, start=1)
    ]
)

VALIDATION = replace_n36_validation_stage(
    "batch_9_roadmap_traceability",
    [
        (
            row.responsibility_id,
            row.responsibility,
            True,
            row.evidence,
            row.passed,
        )
        for row in ROADMAP_TRACEABILITY.itertuples(index=False)
    ],
)

INHERITED_WARNING_COUNT = int(
    FINAL_PACKAGE_MANIFEST["upstream"]["warning_failures"]
)

VALIDATION = replace_n36_validation_stage(
    "batch_9_inherited_warning",
    [
        (
            "upstream_deployment_warning",
            "Retain the Notebook 35 dependency-alignment warning",
            0,
            {
                "inherited_warning_count": INHERITED_WARNING_COUNT,
                "scope": (
                    "N35 dependency-pin differences; "
                    "local demonstration passed."
                ),
            },
            INHERITED_WARNING_COUNT == 0,
        )
    ],
    severity="warning",
)

VALIDATION.raise_for_blocking()

display(ROADMAP_TRACEABILITY)

print("Final contract ready.")
print("Roadmap responsibilities:", len(ROADMAP_TRACEABILITY))
print("Canonical artifact records to write:", len(N36_ARTIFACT_SCHEMAS))
print("Expected final physical files:", len(EXPECTED_PHYSICAL_OUTPUTS))
print("Inherited upstream warnings:", INHERITED_WARNING_COUNT)

,responsibility_id,responsibility,evidence,passed
0,n36_r01,Supervisor summary and exact proposal questions,Three exact RQs and ten summary sections,True
1,n36_r02,Final HTML and four model reports,Five self-contained HTML reports,True
2,n36_r03,Compact evidence and LaTeX-ready tables,"Eight tables, including latex_tables.csv",True
3,n36_r04,Thesis-ready and publication-ready figures,18 thesis figures and six publication figures,True
4,n36_r05,Model cards and model identities,"Four cards, implementations, revisions, and co...",True
5,n36_r06,Case and painting report discovery,30 case-report and 50 painting-report records,True
6,n36_r07,All upstream completion manifests,35 gates passed; zero upstream blocking failures,True
7,n36_r08,Configuration and dataset traceability,25 bundled evaluation and 13 indexed source co...,True
8,n36_r09,"Environment, Git, hardware, and seed provenance","20 package versions, two requirements files, s...",True
9,n36_r10,Observed compute and projection boundaries,Four observed model summaries; projections exp...,True


Final contract ready.
Roadmap responsibilities: 16
Canonical artifact records to write: 12
Expected final physical files: 125
Inherited upstream warnings: 1


In [30]:
N36_VALIDATION_STATUS = (
    "warning" if INHERITED_WARNING_COUNT else "passed"
)


def build_n36_artifacts():
    records = []

    for output_key, schema_version in (
        N36_ARTIFACT_SCHEMAS.items()
    ):
        path = OUTPUT_PATHS[output_key]

        artifact_type = (
            "review_package"
            if path.is_dir()
            else {
                ".csv": "table",
                ".json": "metadata",
                ".md": "report",
            }.get(path.suffix.lower(), "file")
        )

        row_count = None

        if output_key == "artifact_index":
            row_count = len(FINAL_ARTIFACT_INDEX)
        elif output_key == "validation":
            row_count = len(VALIDATION.checks)

        records.append(
            build_artifact_record(
                artifact_key=f"supervisor_package.{output_key}",
                producer_notebook=NOTEBOOK_STEM,
                path=path,
                artifact_type=artifact_type,
                artifact_role=output_key,
                schema_version=schema_version,
                dataset_scope="controlled_50",
                experiment_id="supervisor_publication_package",
                validation_status=N36_VALIDATION_STATUS,
                row_count=row_count,
                project_root=PROJECT_ROOT,
            )
        )

    return records


VALIDATION.write_csv(OUTPUT_PATHS["validation"])

N36_ARTIFACT_RECORDS = build_n36_artifacts()

write_artifact_manifest(
    OUTPUT_PATHS["artifacts"],
    N36_ARTIFACT_RECORDS,
)

N36_INPUT_PATHS = {
    Path(path).resolve()
    for path in REQUIRED_INPUT_PATHS.values()
}

N36_INPUT_PATHS.update(
    item.source.resolve()
    for item in COPY_PLAN
)

N36_INPUT_PATHS.update(
    (PROJECT_ROOT / path).resolve()
    for path in SOURCE_CONFIGURATION_RELATIVE_PATHS
)

N36_INPUT_RECORDS = [
    {
        "input_key": f"source_{number:03d}",
        "relative_path": path.relative_to(
            PROJECT_ROOT
        ).as_posix(),
        "checksum": sha256_path(path),
    }
    for number, path in enumerate(
        sorted(N36_INPUT_PATHS, key=lambda item: item.as_posix()),
        start=1,
    )
]

N36_CONFIGURATION_CHECKSUMS = {
    record["relative_path"]: record["checksum"]
    for record in N36_INPUT_RECORDS
    if record["relative_path"].startswith("config/")
}

N36_EXPECTED_COUNTS = {
    "physical_output_files": 125,
    "canonical_output_roles": 14,
    "artifact_records": 12,
    "artifact_index_records": 120,
    "package_files": 114,
    "upstream_notebooks": 35,
    "roadmap_responsibilities": 16,
}


def build_n36_manifest(completed=False):
    output_records = []

    for key, path in sorted(OUTPUT_PATHS.items()):
        record = {
            "artifact_key": f"supervisor_package.{key}",
            "relative_path": path.relative_to(
                PROJECT_ROOT
            ).as_posix(),
            "artifact_type": (
                "directory"
                if key == "package_root"
                else path.suffix.lstrip(".")
            ),
        }

        if key != "run_manifest" and path.exists():
            record["checksum"] = sha256_path(path)

        output_records.append(record)

    now = (
        datetime.now(timezone.utc)
        .isoformat(timespec="seconds")
        .replace("+00:00", "Z")
    )

    manifest = build_run_manifest(
        notebook_id=NOTEBOOK_ID,
        notebook_name=NOTEBOOK_STEM,
        origin=NOTEBOOK_CONFIG["origin"],
        run_status="completed" if completed else "running",
        started_at_utc=RUN_STARTED_AT_UTC,
        completed_at_utc=now if completed else "",
        inventory_run_id=INVENTORY_RUN["inventory_run_id"],
        dataset_versions=DATASET_IDENTITY,
        configuration_paths=sorted(
            N36_CONFIGURATION_CHECKSUMS
        ),
        configuration_checksums_by_path=(
            N36_CONFIGURATION_CHECKSUMS
        ),
        helper_versions={
            "supervisor_package": SUPERVISOR_PACKAGE_VERSION,
            "manifests": MANIFESTS_MODULE_VERSION,
            "validation": VALIDATION_MODULE_VERSION,
            "paths": PATHS_MODULE_VERSION,
        },
        inputs=N36_INPUT_RECORDS,
        outputs=output_records,
        expected_counts=N36_EXPECTED_COUNTS,
        observed_counts={
            "physical_output_files": sum(
                path.is_file()
                for path in OUTPUT_ROOT.rglob("*")
            ),
            "canonical_output_roles": len(OUTPUT_PATHS),
            "artifact_records": len(N36_ARTIFACT_RECORDS),
            "artifact_index_records": len(FINAL_ARTIFACT_INDEX),
            "package_files": len(
                FINAL_PACKAGE_MANIFEST["files"]
            ),
            "upstream_notebooks": len(
                UPSTREAM_MANIFEST_SNAPSHOT
            ),
            "roadmap_responsibilities": len(
                ROADMAP_TRACEABILITY
            ),
            "validation_rows": len(VALIDATION.checks),
        },
        validation_summary=VALIDATION.summary(),
        known_limitations=[
            *SCIENTIFIC_BOUNDARIES["statements"],
            (
                "The package is a review bundle, "
                "not a complete executable repository clone."
            ),
            (
                "N35 dependency-version differences remain "
                "an inherited warning; public deployment "
                "is not completed."
            ),
            (
                "Generated package-summary links were rebased; "
                "the canonical report and upstream files "
                "were not modified."
            ),
        ],
        project_root=PROJECT_ROOT,
        package_names=list(
            REPRO_PACKAGE_DISTRIBUTIONS.values()
        ),
        hardware={
            "execution_mode": "packaging_and_integrity_validation",
            "restoration_inference_performed": False,
            "scientific_metrics_recomputed": False,
            "upstream_hardware_reference": (
                MODEL_CARDS_PATH.relative_to(
                    PROJECT_ROOT
                ).as_posix()
            ),
        },
        run_id=globals().get("N36_RUN_ID"),
    )

    manifest.update(
        {
            "validation_status": N36_VALIDATION_STATUS,
            "refactor_status": (
                "refactored" if completed else "in_progress"
            ),
            "completion_gate_passed": bool(completed),
            "creates_new_scientific_evidence": False,
            "inherited_upstream_warning_count": (
                INHERITED_WARNING_COUNT
            ),
            "package_validation_status": (
                FINAL_PACKAGE_MANIFEST["validation_status"]
            ),
            "package_tree_sha256": (
                FINAL_PACKAGE_MANIFEST["package_tree_sha256"]
            ),
            "artifact_manifest_checksum": sha256_file(
                OUTPUT_PATHS["artifacts"]
            ),
            "roadmap_traceability": json.loads(
                ROADMAP_TRACEABILITY.to_json(
                    orient="records"
                )
            ),
        }
    )

    return manifest


N36_PROVISIONAL_MANIFEST = build_n36_manifest(
    completed=False
)

N36_RUN_ID = N36_PROVISIONAL_MANIFEST["run_id"]

write_run_manifest(
    OUTPUT_PATHS["run_manifest"],
    N36_PROVISIONAL_MANIFEST,
)

print("Provisional manifests persisted.")
print("Run ID:", N36_RUN_ID)
print("Manifest inputs:", len(N36_INPUT_RECORDS))
print("Artifact records:", len(N36_ARTIFACT_RECORDS))
print("Completion gate remains False until final verification.")

Provisional manifests persisted.
Run ID: run_4e221aa9405d4d5cbeb7ae120dfad71a
Manifest inputs: 127
Artifact records: 12
Completion gate remains False until final verification.


In [31]:
N36_RELOADED_ARTIFACTS = pd.read_csv(
    OUTPUT_PATHS["artifacts"],
    keep_default_na=False,
)

N36_RELOADED_CHECKS = pd.read_csv(
    OUTPUT_PATHS["validation"],
    keep_default_na=False,
)

N36_PHYSICAL_OUTPUTS = {
    path.resolve()
    for path in OUTPUT_ROOT.rglob("*")
    if path.is_file()
}

N36_ARTIFACT_HASH_FAILURES = [
    str(row.artifact_key)
    for row in N36_RELOADED_ARTIFACTS.itertuples(index=False)
    if sha256_path(
        PROJECT_ROOT / row.relative_path
    ) != str(row.checksum)
]

N36_CURRENT_PACKAGE_FILES = package_file_records(
    PACKAGE_ROOT,
    PROJECT_ROOT,
)

N36_PACKAGE_TREE_MATCHES = (
    package_tree_checksum(N36_CURRENT_PACKAGE_FILES)
    == FINAL_PACKAGE_MANIFEST["package_tree_sha256"]
)

N36_MISSING_OUTPUTS = sorted(
    path.relative_to(OUTPUT_ROOT).as_posix()
    for path in EXPECTED_PHYSICAL_OUTPUTS
    - N36_PHYSICAL_OUTPUTS
)

N36_UNEXPECTED_OUTPUTS = sorted(
    path.relative_to(OUTPUT_ROOT).as_posix()
    for path in N36_PHYSICAL_OUTPUTS
    - EXPECTED_PHYSICAL_OUTPUTS
)

N36_FINAL_CHECKS = [
    (
        "physical_output_set",
        "The exact final physical output set exists",
        {"missing": [], "unexpected": []},
        {
            "missing": N36_MISSING_OUTPUTS,
            "unexpected": N36_UNEXPECTED_OUTPUTS,
        },
        not N36_MISSING_OUTPUTS
        and not N36_UNEXPECTED_OUTPUTS,
    ),
    (
        "physical_output_count",
        "Exactly 125 physical output files exist",
        125,
        len(N36_PHYSICAL_OUTPUTS),
        len(N36_PHYSICAL_OUTPUTS) == 125,
    ),
    (
        "artifact_record_count",
        "Twelve canonical artifacts are registered",
        12,
        len(N36_RELOADED_ARTIFACTS),
        len(N36_RELOADED_ARTIFACTS) == 12,
    ),
    (
        "artifact_checksums",
        "All artifact checksums match persisted files",
        [],
        N36_ARTIFACT_HASH_FAILURES,
        not N36_ARTIFACT_HASH_FAILURES,
    ),
    (
        "validation_reload",
        "Consolidated validation reloads with every current check",
        len(VALIDATION.checks),
        len(N36_RELOADED_CHECKS),
        len(N36_RELOADED_CHECKS) == len(VALIDATION.checks),
    ),
    (
        "package_tree",
        "The final package tree remains unchanged",
        True,
        N36_PACKAGE_TREE_MATCHES,
        N36_PACKAGE_TREE_MATCHES,
    ),
    (
        "index_checksum",
        "The package manifest identifies the current artifact index",
        FINAL_PACKAGE_MANIFEST["artifact_index"]["sha256"],
        sha256_file(ARTIFACT_INDEX_PATH),
        (
            sha256_file(ARTIFACT_INDEX_PATH)
            == FINAL_PACKAGE_MANIFEST[
                "artifact_index"
            ]["sha256"]
        ),
    ),
    (
        "roadmap_completion",
        "All sixteen responsibility groups passed",
        16,
        int(ROADMAP_TRACEABILITY["passed"].sum()),
        len(ROADMAP_TRACEABILITY) == 16
        and ROADMAP_TRACEABILITY["passed"].all(),
    ),
    (
        "no_blocking_failures",
        "No blocking validation failure remains",
        0,
        len(VALIDATION.blocking_failures),
        not VALIDATION.blocking_failures,
    ),
]

VALIDATION = replace_n36_validation_stage(
    "batch_9_completion_gate",
    N36_FINAL_CHECKS,
)

display(
    VALIDATION.to_dataframe().loc[
        lambda frame: frame["validation_stage"].eq(
            "batch_9_completion_gate"
        ),
        ["check_id", "passed", "observed"],
    ]
)

VALIDATION.raise_for_blocking()

# Final write order avoids stale validation checksums.
VALIDATION.write_csv(OUTPUT_PATHS["validation"])

N36_ARTIFACT_RECORDS = build_n36_artifacts()

write_artifact_manifest(
    OUTPUT_PATHS["artifacts"],
    N36_ARTIFACT_RECORDS,
)

N36_FINAL_MANIFEST = build_n36_manifest(
    completed=True
)

write_run_manifest(
    OUTPUT_PATHS["run_manifest"],
    N36_FINAL_MANIFEST,
)

print("Final completion gate passed.")
print("Final validation rows:", len(VALIDATION.checks))
print("Canonical artifacts:", len(N36_ARTIFACT_RECORDS))
print("Run status:", N36_FINAL_MANIFEST["run_status"])
print("Validation status:", N36_FINAL_MANIFEST["validation_status"])
print("Proceed to the independent final reload below.")

,check_id,passed,observed
173,physical_output_set,True,"{""missing"": [], ""unexpected"": []}"
174,physical_output_count,True,125
175,artifact_record_count,True,12
176,artifact_checksums,True,[]
177,validation_reload,True,173
178,package_tree,True,True
179,index_checksum,True,817c7c0901894158c381e58d04227a906482b6e4112639...
180,roadmap_completion,True,16
181,no_blocking_failures,True,0


Final completion gate passed.
Final validation rows: 182
Canonical artifacts: 12
Run status: completed
Validation status: warning
Proceed to the independent final reload below.


In [32]:
FINAL_RELOADED_CHECKS = pd.read_csv(
    OUTPUT_PATHS["validation"],
    keep_default_na=False,
)

FINAL_RELOADED_ARTIFACTS = pd.read_csv(
    OUTPUT_PATHS["artifacts"],
    keep_default_na=False,
)

with OUTPUT_PATHS["run_manifest"].open(
    "r",
    encoding="utf-8-sig",
) as handle:
    FINAL_RELOADED_MANIFEST = json.load(handle)

with PACKAGE_MANIFEST_PATH.open(
    "r",
    encoding="utf-8-sig",
) as handle:
    FINAL_RELOADED_PACKAGE_MANIFEST = json.load(handle)

FINAL_PHYSICAL_OUTPUTS = {
    path.resolve()
    for path in OUTPUT_ROOT.rglob("*")
    if path.is_file()
}

FINAL_PASSED_VALUES = (
    FINAL_RELOADED_CHECKS["passed"]
    .astype(str)
    .str.strip()
    .str.casefold()
    .eq("true")
)

FINAL_BLOCKING_FAILURES = int(
    (
        ~FINAL_PASSED_VALUES
        & FINAL_RELOADED_CHECKS["severity"].isin(
            ["blocking", "error"]
        )
    ).sum()
)

FINAL_WARNING_FAILURES = int(
    (
        ~FINAL_PASSED_VALUES
        & FINAL_RELOADED_CHECKS["severity"].eq("warning")
    ).sum()
)

FINAL_ARTIFACT_HASH_FAILURES = [
    str(row.artifact_key)
    for row in FINAL_RELOADED_ARTIFACTS.itertuples(index=False)
    if sha256_path(
        PROJECT_ROOT / row.relative_path
    ) != str(row.checksum)
]

FINAL_RUN_OUTPUT_HASH_FAILURES = [
    record["artifact_key"]
    for record in FINAL_RELOADED_MANIFEST["outputs"]
    if "checksum" in record
    and sha256_path(
        PROJECT_ROOT / record["relative_path"]
    ) != record["checksum"]
]

FINAL_PACKAGE_FILES = package_file_records(
    PACKAGE_ROOT,
    PROJECT_ROOT,
)

FINAL_PACKAGE_FILE_TUPLES = {
    (
        str(row.package_relative_path),
        int(row.size_bytes),
        str(row.sha256),
    )
    for row in FINAL_PACKAGE_FILES.itertuples(index=False)
}

FINAL_DECLARED_PACKAGE_TUPLES = {
    (
        str(record["package_relative_path"]),
        int(record["size_bytes"]),
        str(record["sha256"]),
    )
    for record in FINAL_RELOADED_PACKAGE_MANIFEST["files"]
}

FINAL_RUN_OUTPUT_PATHS = {
    (
        PROJECT_ROOT / record["relative_path"]
    ).resolve()
    for record in FINAL_RELOADED_MANIFEST["outputs"]
}

FINAL_EXPECTED_OUTPUT_ROLES = {
    path.resolve()
    for path in OUTPUT_PATHS.values()
}

FINAL_COMPLETION_FAILURES = []

FINAL_ASSERTIONS = {
    "physical output set": (
        FINAL_PHYSICAL_OUTPUTS == EXPECTED_PHYSICAL_OUTPUTS
    ),
    "physical output count": len(FINAL_PHYSICAL_OUTPUTS) == 125,
    "artifact record count": len(FINAL_RELOADED_ARTIFACTS) == 12,
    "artifact checksums": not FINAL_ARTIFACT_HASH_FAILURES,
    "run output checksums": not FINAL_RUN_OUTPUT_HASH_FAILURES,
    "run output coverage": (
        FINAL_RUN_OUTPUT_PATHS == FINAL_EXPECTED_OUTPUT_ROLES
    ),
    "blocking failures": FINAL_BLOCKING_FAILURES == 0,
    "inherited warning retained": (
        FINAL_WARNING_FAILURES == int(INHERITED_WARNING_COUNT > 0)
    ),
    "validation count": (
        FINAL_RELOADED_MANIFEST["validation_summary"]["check_count"]
        == len(FINAL_RELOADED_CHECKS)
    ),
    "artifact manifest checksum": (
        FINAL_RELOADED_MANIFEST["artifact_manifest_checksum"]
        == sha256_file(OUTPUT_PATHS["artifacts"])
    ),
    "artifact-index checksum": (
        FINAL_RELOADED_PACKAGE_MANIFEST[
            "artifact_index"
        ]["sha256"]
        == sha256_file(ARTIFACT_INDEX_PATH)
    ),
    "package file records": (
        FINAL_PACKAGE_FILE_TUPLES
        == FINAL_DECLARED_PACKAGE_TUPLES
    ),
    "package tree checksum": (
        package_tree_checksum(FINAL_PACKAGE_FILES)
        == FINAL_RELOADED_PACKAGE_MANIFEST["package_tree_sha256"]
    ),
    "completed run": (
        FINAL_RELOADED_MANIFEST["run_status"] == "completed"
    ),
    "completion gate": (
        FINAL_RELOADED_MANIFEST["completion_gate_passed"] is True
    ),
    "no new science": (
        FINAL_RELOADED_MANIFEST[
            "creates_new_scientific_evidence"
        ] is False
    ),
}

FINAL_COMPLETION_FAILURES = [
    name
    for name, passed in FINAL_ASSERTIONS.items()
    if not passed
]

if FINAL_COMPLETION_FAILURES:
    FINAL_RELOADED_MANIFEST.update(
        {
            "run_status": "partial",
            "validation_status": "failed",
            "completion_gate_passed": False,
            "final_reload_failures": FINAL_COMPLETION_FAILURES,
        }
    )

    write_run_manifest(
        OUTPUT_PATHS["run_manifest"],
        FINAL_RELOADED_MANIFEST,
    )

    raise RuntimeError(
        "Final independent verification failed: "
        + ", ".join(FINAL_COMPLETION_FAILURES)
    )

N36_COMPLETION_GATE_PASSED = True

display(ROADMAP_TRACEABILITY)

display(
    FINAL_RELOADED_ARTIFACTS[
        [
            "artifact_key",
            "format",
            "file_count",
            "validation_status",
        ]
    ]
)

print("NOTEBOOK 36 COMPLETION GATE: PASSED")
print("Run ID:", FINAL_RELOADED_MANIFEST["run_id"])
print("Physical output files:", len(FINAL_PHYSICAL_OUTPUTS))
print("Package files:", len(FINAL_PACKAGE_FILES))
print("Canonical artifact records:", len(FINAL_RELOADED_ARTIFACTS))
print("Validation rows:", len(FINAL_RELOADED_CHECKS))
print("Blocking failures:", FINAL_BLOCKING_FAILURES)
print("Inherited warning records:", FINAL_WARNING_FAILURES)
print(
    "Package size MiB:",
    round(FINAL_PACKAGE_FILES["size_bytes"].sum() / (1024 ** 2), 3),
)
print("No upstream files were modified.")
print("Save the notebook for the final audit and inventory refresh.")

,responsibility_id,responsibility,evidence,passed
0,n36_r01,Supervisor summary and exact proposal questions,Three exact RQs and ten summary sections,True
1,n36_r02,Final HTML and four model reports,Five self-contained HTML reports,True
2,n36_r03,Compact evidence and LaTeX-ready tables,"Eight tables, including latex_tables.csv",True
3,n36_r04,Thesis-ready and publication-ready figures,18 thesis figures and six publication figures,True
4,n36_r05,Model cards and model identities,"Four cards, implementations, revisions, and co...",True
5,n36_r06,Case and painting report discovery,30 case-report and 50 painting-report records,True
6,n36_r07,All upstream completion manifests,35 gates passed; zero upstream blocking failures,True
7,n36_r08,Configuration and dataset traceability,25 bundled evaluation and 13 indexed source co...,True
8,n36_r09,"Environment, Git, hardware, and seed provenance","20 package versions, two requirements files, s...",True
9,n36_r10,Observed compute and projection boundaries,Four observed model summaries; projections exp...,True


,artifact_key,format,file_count,validation_status
0,supervisor_package.supervisor_summary,md,1,warning
1,supervisor_package.reproducibility_appendix,md,1,warning
2,supervisor_package.limitations_and_deviations,md,1,warning
3,supervisor_package.artifact_index,csv,1,warning
4,supervisor_package.key_findings,json,1,warning
5,supervisor_package.open_questions,md,1,warning
6,supervisor_package.feedback_agenda,md,1,warning
7,supervisor_package.package_root,directory,114,warning
8,supervisor_package.package_readme,md,1,warning
9,supervisor_package.reproducibility_snapshot,json,1,warning


NOTEBOOK 36 COMPLETION GATE: PASSED
Run ID: run_4e221aa9405d4d5cbeb7ae120dfad71a
Physical output files: 125
Package files: 114
Canonical artifact records: 12
Validation rows: 182
Blocking failures: 0
Inherited warning records: 1
Package size MiB: 27.677
No upstream files were modified.
Save the notebook for the final audit and inventory refresh.
